<a id="nb07-top"></a>

# Notebook 07 — SCS figures and tables

I build every figure and table of the *Sustainable Cities and Society* article and its
supplementary material here, in manuscript order, from the published dataset v1.0.1
(DOI [10.5281/zenodo.23777933](https://doi.org/10.5281/zenodo.23777933); all versions
10.5281/zenodo.23050559). The association results come from NB06, because the dataset does not
hold them.

**How to run.** Run All, or `python run_all.py --scope manuscript`, after NB01–NB06 have run
once. The notebook makes no GEE, EUMETSAT or OSM call. Each item needs only the two Setup cells;
Fig. 6 also needs Fig. 5, and Supp. Table S6 needs Table 3. For a supplementary set, the first
cell shows one example per level and the second writes the whole set.

**Checks.** I assert every number shown against the dataset, log every file read in
`_data_sources.csv`, and check every figure file's format.

**Outputs** in `Results/manuscript_outputs/`: `main_figures/`, `main_tables/`,
`supplementary_figures/`, `supplementary_tables/` and `captions.md`.

<a id="nb07-contents"></a>

## Contents

- **[0 · Setup](#nb07-setup)**
- **[1 · Main article](#nb07-main)**
  - [Front matter](#nb07-front)
    - [Graphical abstract](#nb07-graphical-abstract)
  - [1.1 · Methods (manuscript section 2)](#nb07-1-1)
    - [Fig. 1 · Study area](#nb07-fig-1)
    - [Fig. 2 · Methodological workflow](#nb07-fig-2)
    - [Fig. 3 · Three-zone delineation of five cities](#nb07-fig-3)
    - [Table 1 · Typology composition of the 56 cities](#nb07-table-1)
    - [Fig. 4 · Stakeholder participants by cluster and city](#nb07-fig-4)
  - [1.2 · Results 3.1 — long-term and event-scale surface temperature (Objective 1)](#nb07-1-2)
    - [Fig. 5 · Annual trends, selected clusters](#nb07-fig-5)
    - [Fig. 6 · Monthly trends, selected clusters](#nb07-fig-6)
    - [Fig. 7 · Persistent heat events by cluster](#nb07-fig-7)
    - [Fig. 8 · Event-scale SUHII by city](#nb07-fig-8)
  - [1.3 · Results 3.2 — surface hot and cold spots (Objective 2)](#nb07-1-3)
    - [Table 2 · Significant spots by cluster](#nb07-table-2)
    - [Fig. 9 · Surface hot and cold spots, all cities, day](#nb07-fig-9)
    - [Fig. 10 · Surface hot and cold spots, all cities, night](#nb07-fig-10)
    - [Fig. 11 · Spot area by cluster (nested L50 / L75 / L90)](#nb07-fig-11)
  - [1.4 · Results 3.3 — population within spots (Objective 3)](#nb07-1-4)
    - [Fig. 12 · Population within hot and cold spots](#nb07-fig-12)
  - [1.5 · Results 3.4 — stakeholder accounts (Objective 4)](#nb07-1-5)
    - [Table 3 · Stakeholder accounts against the satellite record](#nb07-table-3)
    - [Fig. 13 · Stakeholder perception by cluster](#nb07-fig-13)
    - [Fig. 14 · Hit rate against buffer distance](#nb07-fig-14)
  - [1.6 · Results 3.5 — typology and the thermal anomalies (Objective 5)](#nb07-1-6)
    - [Table 4 · Typology–anomaly association](#nb07-table-4)
    - [Table 5 · Best single-element attribution (excerpt)](#nb07-table-5)
    - [Fig. 15 · Effect sizes (forest plot)](#nb07-fig-15)
- **[2 · Supplementary figures](#nb07-supp-figures)**
    - [S1 · Population trajectories: S1.0 national, S1.1 cities, S1.2 clusters](#nb07-s1)
    - [S2 · City delineation maps](#nb07-s2)
    - [S3 · Stakeholder sample maps: S3.1 by cluster, S3.2 by city](#nb07-s3)
    - [S4 · Long-term trends: S4.1 clusters, S4.2 cities, S4.3 national](#nb07-s4)
    - [S5 · Daily series and persistent heat events: S5.0 panel, S5.1 cities, S5.2 clusters, S5.3 national](#nb07-s5)
    - [S6 · Event frequency by city and cluster, day and night](#nb07-s6)
    - [S7 · Agreement between thermal products: S7.1 MODIS vs ERA5-Land, S7.2 SEVIRI vs ERA5-Land](#nb07-s7)
    - [S8 · Gi* z-score distributions, S8.1–S8.56](#nb07-s8)
    - [S9 · City spot area, ranked: S9.1 hot spots, S9.2 cold spots](#nb07-s9)
    - [S10 · Population within spots: national maps and half-donuts](#nb07-s10)
    - [S11 · City-level stakeholder perception](#nb07-s11)
- **[3 · Supplementary tables](#nb07-supp-tables)**
    - [Supp. Table S1 · The 24 physiographic landscape typologies](#nb07-supp-table-s1)
    - [Supp. Table S2 · Normality tests (Shapiro–Wilk)](#nb07-supp-table-s2)
    - [Supp. Table S3 · Heat events per city](#nb07-supp-table-s3)
    - [Supp. Table S4 · Selected events, event tier (day / night) and sensor agreement](#nb07-supp-table-s4)
    - [Supp. Table S5 · Spots and contours per city](#nb07-supp-table-s5)
    - [Supp. Table S6 · Stakeholder accounts per city](#nb07-supp-table-s6)
    - [Supp. Table S7 · Hit rate by city and buffer](#nb07-supp-table-s7)
    - [Supp. Table S8 · Best-element attribution, complete](#nb07-supp-table-s8)
- **[4 · Assembly](#nb07-assembly)**

<a id="nb07-setup"></a>

## 0 · Setup

I set the paths, read the dataset with `pkg()`, assert every number shown with `check()` and
save every figure with `savefig()`, which writes each format explicitly and checks the file
header. `read_source()` logs every file read in `_data_sources.csv` and refuses a forbidden
source. A figure set already written by the current code from the current inputs is skipped;
`REBUILD = True` redraws everything. The second cell loads the shared inputs: zone geometries,
the UN WUP table and the typology map.

[↑ Contents](#nb07-contents)

In [ ]:
import sys, warnings, json, re, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import xlsxwriter
from shapely.geometry import Point
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

# repo root = parent of this /Notebook folder
ROOT = Path.cwd()
if ROOT.name.lower() == "notebook":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
import config
import visualization as viz
import temporal_panels as tp
import pub_hotspot_tables as _pht
import pub_figures as pf
import scs_style as S          # shared: approved cluster labels, colours, checked savefig
# The plotting modules in src/ are reloaded before every cell, so a cell run on its own in a
# long-running kernel always draws with the current code.
_ip = get_ipython()
if "autoreload" not in _ip.extension_manager.loaded:
    _ip.run_line_magic("load_ext", "autoreload")
_ip.run_line_magic("autoreload", "1")
_ip.run_line_magic("aimport", "visualization, scs_style, temporal_panels, pub_hotspot_tables, pub_figures")

warnings.filterwarnings("ignore", message=".*FancyArrowPatch.*")
plt.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight",
                     "pdf.fonttype": 42, "ps.fonttype": 42})

# ── input caches ──
RESULTS_AOI = ROOT / "Results" / "01_aoi_delineation"
RESULTS_03  = ROOT / "Results" / "03_temporal_analysis"
STATS03     = RESULTS_03 / "Cluster25_Composites" / "stats"
RESULTS_04  = ROOT / "Results" / "04_temporal_analysis_daily"
RESULTS_05  = ROOT / "Results" / "05_hotspot_analysis"
RESULTS_06  = ROOT / "Results" / "06_hotspot_modelling"
STUDY_YEAR  = config.PROJECT_STUDY_YEAR

# ── the published v1.0.1 package: the source of record for every value ──
PKG  = RESULTS_05 / "surface_thermal_anomaly_stakeholder_composite"
GPKG = PKG / "surface_thermal_anomaly_stakeholder_composite.gpkg"
assert GPKG.exists(), f"build the package first: {GPKG}"

_LAYERS = {}


def pkg(layer, geom=False):
    """Read a layer of the published package, once (every call is logged)."""
    read_source(GPKG, layer=layer)
    key = (layer, geom)
    if key not in _LAYERS:
        _LAYERS[key] = (gpd.read_file(GPKG, layer=layer) if geom
                        else pyogrio.read_dataframe(GPKG, layer=layer, read_geometry=False))
    return _LAYERS[key].copy()


ASSERTIONS = []


def check(label, ok, detail=""):
    """Record and print an assertion. A figure must not disagree with the package."""
    ASSERTIONS.append((label, bool(ok), detail, CURRENT_ITEM))
    print(("  [OK]   " if ok else "  [FAIL] ") + label + (f" - {detail}" if detail else ""))
    if not ok:
        raise AssertionError(f"{label}: {detail}")


# ── cluster labels, approved: nothing below spells a cluster name out by hand ──
CL = S.labels()                                   # indexed by integer cluster number
CLUSTER_LABEL      = {int(i): r.label_short for i, r in CL.iterrows()}
CLUSTER_LABEL_LONG = {int(i): r.label_long for i, r in CL.iterrows()}


def cl_header(cid, n=None):
    """Panel header: cluster in words, its code, and how many cities it holds."""
    return S.panel_header(int(cid), n)


# ── output tree ──
OUT       = ROOT / "Results" / "manuscript_outputs"
MAIN_FIG  = OUT / "main_figures"
SUP_FIG   = OUT / "supplementary_figures"
MAIN_TAB  = OUT / "main_tables"
SUP_TAB   = OUT / "supplementary_tables"
for d in (MAIN_FIG, SUP_FIG, MAIN_TAB, SUP_TAB):
    d.mkdir(parents=True, exist_ok=True)


# ── data-source policy and provenance ──
# Every file this notebook reads goes through read_source(): a forbidden source stops the
# run, and every read is logged (item, path, category, permitted, size, mtime, MD5) to
# _data_sources.csv.
FORBIDDEN_PARTS = ("_superseded_", "_to_delete", "_archive_v1.0.0", "/notebooks/",  # forbidden-list
                   "stakeholder_hitrate_1000m")                                     # forbidden-list
PERMITTED = {          # cache or input -> the items allowed to read it
    "input": {"Fig. 1", "Fig. 12", "S1", "S3", "S10", "Supp. Table S1"},
    "NB01": {"Fig. 3", "S1", "S2", "S3"},
    "NB02": {"S3"},
    "NB03": {"Fig. 5", "Fig. 6", "S4"},
    "NB04": {"S5", "S7"},
    "NB05": {"Fig. 9", "Fig. 10", "S8"},
    "NB06": {"Table 4", "Table 5", "Fig. 15", "Supp. Table S2", "Supp. Table S8"},
}
SOURCES_CSV = OUT / "_data_sources.csv"
SOURCE_ROWS, _MD5 = [], {}
CURRENT_ITEM = "Setup"


def begin(item):
    """The item that the following reads and checks belong to."""
    global CURRENT_ITEM
    CURRENT_ITEM = item


def _category(p):
    p = Path(p).resolve()
    if p == GPKG.resolve():
        return "PKG"
    s = p.as_posix()
    for k in ("01", "02", "03", "04", "05", "06"):
        if f"/Results/{k}_" in s:
            return f"NB{k}"
    return "input" if "/data/inputs/" in s else "other"


def read_source(path, layer=""):
    """Refuse a forbidden source, log the read, and return the path unchanged."""
    p = Path(path)
    s = p.resolve().as_posix()
    if any(f in s for f in FORBIDDEN_PARTS):
        raise PermissionError(f"{CURRENT_ITEM}: forbidden data source {p}")
    cat = _category(p)
    permitted = ("yes" if cat == "PKG" or CURRENT_ITEM in PERMITTED.get(cat, ())
                 else "shared" if CURRENT_ITEM == "Setup" else "OOS")
    if p.is_file():
        if s not in _MD5:
            _MD5[s] = hashlib.md5(p.read_bytes()).hexdigest()
        size, h = p.stat().st_size, _MD5[s]
        mtime = pd.Timestamp(p.stat().st_mtime, unit="s").strftime("%Y-%m-%d %H:%M")
    else:
        size, mtime, h = (sum(1 for _ in p.iterdir()) if p.is_dir() else -1), "", \
            ("directory" if p.is_dir() else "missing")
    row = {"item": CURRENT_ITEM, "path": p.resolve().relative_to(ROOT.resolve()).as_posix()
           if p.resolve().is_relative_to(ROOT.resolve()) else s, "layer": layer,
           "category": cat, "permitted": permitted, "size": size, "mtime": mtime, "md5": h}
    SOURCE_ROWS.append(row)
    pd.DataFrame([row]).to_csv(SOURCES_CSV, mode="a", header=False, index=False)
    return path


# ── skip logic: a figure set already written by the current code from the current inputs ──
REBUILD = False      # True redraws every figure set (e.g. after re-running NB01-NB06)


def code_stamp(*objs, inputs=()):
    """Fingerprint of a figure set: the source of its drawing functions (and of savefig), the
    package, and the newest file among its other inputs (files or folders, searched in full)."""
    import inspect
    parts = [f"{GPKG.stat().st_size}-{GPKG.stat().st_mtime_ns}"]
    parts += [inspect.getsource(o) for o in (savefig,) + objs]
    for q in map(Path, inputs):
        files = [q] if q.is_file() else [f for f in q.rglob("*") if f.is_file()]
        parts.append(str(max((f.stat().st_mtime_ns for f in files), default=0)))
    return hashlib.md5("\n".join(parts).encode()).hexdigest()


def set_is_current(folder, stamp, expected):
    """True when the set was last written under this stamp and every expected file exists: its
    run-all cell then skips drawing. The checks always run."""
    f = Path(folder) / "_stamp.txt"
    return (not REBUILD and f.exists() and f.read_text().strip() == stamp
            and all(Path(q).exists() for q in expected))


def mark_current(folder, stamp):
    (Path(folder) / "_stamp.txt").write_text(stamp)


# ── series check shared by Figs 5, 6 and S4 ───────────────────────────────────
def _series_diff(df, ddf, P, kind):
    """Worst difference between an NB03 series drawn (df: LST by zone, ddf: SUHII) and the mean of the
    package rows P over their cities. Returns (worst, values, one-sided gaps, values undefined in both)."""
    keys = ["year"] if kind == "annual" else ["year", "month"]
    worst, n, gaps, undef = 0.0, 0, 0, 0
    for per in ("day", "night"):
        m = lambda z, col="value_c": P[(P.zone == z) & (P.period == per)].groupby(keys)[col].mean()
        z = lambda zn: df[(df.zone == zn) & (df.period == per)].set_index(keys)
        pairs = [(z(zn).median_lst, m(zp)) for zn, zp in
                 (("urban_core", "uc"), ("bra", "bra"), ("city_region", "cr"))]
        pairs.append((ddf[ddf.period == per].set_index(keys).delta_lst, m("uc") - m("bra")))
        if kind == "annual":
            pairs += [(z("city_region").p05_lst, m("cr", "p05_lst")),
                      (z("city_region").p95_lst, m("cr", "p95_lst"))]
        for a, b in pairs:
            a, b = a.align(b)
            gaps += int((a.isna() ^ b.isna()).sum()); undef += int((a.isna() & b.isna()).sum())
            d = (a - b).abs(); worst = max(worst, float(d.max())); n += len(d)
    return worst, n, gaps, undef


# ── print-size records ──
FIG_REPORT = {}


def _as_page(p):
    """A written figure as a fitz page source: a real PDF, else its SVG."""
    import fitz as _fz
    p = Path(p)
    if p.suffix == ".pdf" and p.open("rb").read(5) == b"%PDF-":
        return _fz.open(str(p))
    svg = p.with_name(p.stem + ".svg") if p.suffix == ".pdf" else p
    return _fz.open(str(svg), filetype="svg")


def fig_report(fig, name):
    """Record the print size (mm) and smallest type (pt) of a figure or a written PDF."""
    if hasattr(fig, "get_size_inches"):
        import matplotlib.text as _mtext
        w, h = fig.get_size_inches() * 25.4
        sizes = [t.get_fontsize() for t in fig.findobj(_mtext.Text) if t.get_text().strip()]
    else:
        d = _as_page(fig)
        pg = d[0]
        w, h = pg.rect.width / 72 * 25.4, pg.rect.height / 72 * 25.4
        sizes = [sp["size"] for b in pg.get_text("dict")["blocks"] for ln in b.get("lines", [])
                 for sp in ln.get("spans", []) if sp.get("text", "").strip()]
        d.close()
    FIG_REPORT[name] = (round(w), round(h), round(min(sizes), 2) if sizes else None)
    return FIG_REPORT[name]


# ── NB05 figures and NB06 tables ──
RESULTS_06T = RESULTS_06 / "tables"
_F05 = RESULTS_05 / "figures"


SOURCES_CSV.write_text("item,path,layer,category,permitted,size,mtime,md5\n", encoding="utf-8")

# ── print geometry: every figure is built at its final size on the page ──
MM = 1.0 / 25.4                      # mm -> inches
W_FULL, W_1_5, W_1COL = 190 * MM, 140 * MM, 90 * MM     # 190 / 140 / 90 mm
H_MAX = 240 * MM                                         # no taller than 240 mm
MIN_PT = 6.5                                             # no text smaller than this


def fig_size_report(paths=None):
    """For every main figure: its size in mm and the smallest type on the page.

    Text is measured from the written PDF, so it reports what the printer sees,
    including figures composed from other PDFs.
    """
    import fitz as _fzr
    rows = []
    for p in sorted(paths or MAIN_FIG.glob("Fig_*.pdf")):
        d = _fzr.open(p)
        pg = d[0]
        w_mm, h_mm = pg.rect.width / 72.0 * 25.4, pg.rect.height / 72.0 * 25.4
        sizes = [sp["size"] for bl in pg.get_text("dict")["blocks"]
                 for ln in bl.get("lines", []) for sp in ln.get("spans", [])
                 if sp.get("text", "").strip()]
        d.close()
        rows.append({"figure": p.stem, "width_mm": round(w_mm, 1),
                     "height_mm": round(h_mm, 1),
                     "min_pt": round(min(sizes), 2) if sizes else np.nan,
                     "n_text": len(sizes)})
    r = pd.DataFrame(rows)
    if len(r):
        r["width_ok"] = r.width_mm <= 190.5
        r["height_ok"] = r.height_mm <= 240.5
        r["font_ok"] = r.min_pt.isna() | (r.min_pt >= MIN_PT - 0.05)
        r["ok"] = r.width_ok & r.height_ok & r.font_ok
    return r


PAD_IN = 0.02                 # the margin savefig adds on each side


def _raise_small_type(fig, min_pt=None):
    """Lift every text artist below the minimum point size.

    Sweeping the source for `fontsize=` missed `labelsize=`, `set_fontsize` and the
    rcParam defaults, so the rule is applied to the drawn artists instead, where
    nothing can hide. Returns the smallest size found before the lift.
    """
    import matplotlib.text as _mtext
    m = min_pt or MIN_PT
    smallest = None
    for t in fig.findobj(_mtext.Text):
        if not t.get_text().strip():
            continue
        sz = t.get_fontsize()
        smallest = sz if smallest is None else min(smallest, sz)
        if sz < m:
            t.set_fontsize(m)
    return smallest


def _fit_width(fig, target_in, tries=5):
    """Shrink the figure until the SAVED page is the target print width.

    bbox_inches="tight" grows the page past figsize whenever a legend or label sits
    outside the axes, and then adds pad_inches on each side, so figsize alone never
    controlled the printed width. Text keeps its point size while the canvas shrinks,
    so this converges without making type smaller.
    """
    inner = target_in - 2 * PAD_IN
    for _ in range(tries):
        fig.canvas.draw()
        bb = fig.get_tightbbox(fig.canvas.get_renderer())
        if bb is None or bb.width <= 0:
            return
        k = inner / bb.width
        if abs(k - 1.0) < 0.002:
            return
        w, h = fig.get_size_inches()
        fig.set_size_inches(w * k, h, forward=True)


def _page_width_mm(p):
    import fitz as _fz
    d = _fz.open(p)
    w = d[0].rect.width / 72.0 * 25.4
    d.close()
    return w


def _scale_pdf_page(p, target_in):
    """Rescale a written PDF page to the target width, keeping its proportions."""
    import fitz as _fz
    src = _fz.open(p)
    sw, sh = src[0].rect.width, src[0].rect.height
    k = (target_in * 72.0) / sw
    out = _fz.open()
    pg = out.new_page(width=sw * k, height=sh * k)
    pg.show_pdf_page(pg.rect, src, 0)
    out.save(str(p) + ".tmp")
    src.close()
    svg = _fz.open(str(p) + ".tmp")[0].get_svg_image()
    out.close()
    Path(str(p) + ".tmp").replace(p)
    return svg


def show_file(paths):
    """Display a written figure (its PDF) with its print size and smallest type."""
    from IPython.display import Image as _Image
    pdf = next(p for p in paths if p.suffix == ".pdf")
    w, h, pt = fig_report(pdf, CURRENT_ITEM)
    d = _as_page(pdf)
    png = d[0].get_pixmap(dpi=120, alpha=False).tobytes("png")
    d.close()
    print(f"Saved: {', '.join(p.name for p in paths)} in {pdf.parent.relative_to(ROOT)}  ·  "
          f"{w} × {h} mm, " + (f"smallest type {pt} pt" if pt else "text drawn as paths"))
    display(_Image(data=png))


def savefig(fig, path_noext, formats=("pdf", "svg"), dpi=300, fit=W_FULL, keep_size=False,
            show=False):
    """Save one figure to vector formats and return the paths written.

    format= is passed explicitly and the written magic bytes are checked: relying on
    the extension alone once produced an SVG inside a .pdf, which no visual check
    would catch.

    Fitting to `fit` inches of print width is tried first by shrinking the canvas,
    which leaves type at its point size. Some figures hold an artist whose extent is
    fixed in inches, so the canvas cannot shrink past it; those are drawn with type
    enlarged by the overshoot and the finished page is then scaled down, which lands
    the printed type back on MIN_PT rather than below it.

    keep_size=True keeps the figure as drawn: tight bounding box with matplotlib's default
    0.1 in padding, no type lifting and no fitting to a print width.

    show=True closes the live figure and displays the written PDF instead, with its print size
    and smallest type, so what the notebook shows is exactly the file in the output folder.
    """
    if keep_size:
        fit = None
    else:
        _raise_small_type(fig)
    if fit:
        _fit_width(fig, fit)
        _raise_small_type(fig)
    out = []
    for ext in formats:
        p = Path(f"{path_noext}.{ext}")
        fig.savefig(p, format=ext, dpi=dpi, bbox_inches="tight",
                    pad_inches=0.1 if keep_size else PAD_IN)
        out.append(p)

    if fit:
        over = _page_width_mm(out[0]) / (fit * 25.4)
        if over > 1.004:
            _raise_small_type(fig, MIN_PT * over)
            _fit_width(fig, fit)
            for ext, p in zip(formats, out):
                fig.savefig(p, format=ext, dpi=dpi, bbox_inches="tight",
                            pad_inches=PAD_IN)
            svg = _scale_pdf_page(out[0], fit)
            for p in out[1:]:
                if p.suffix == ".svg":
                    if not svg.lstrip().startswith("<?xml"):
                        svg = '<?xml version="1.0" encoding="utf-8" standalone="no"?>'                               + chr(10) + svg
                    p.write_text(svg, encoding="utf-8")

    for p in out:
        head = p.open("rb").read(5)
        want = b"%PDF-" if p.suffix == ".pdf" else (b"<?xml" if p.suffix == ".svg" else None)
        if want and head != want:
            raise RuntimeError(f"{p.name} was written as {head!r}, not {p.suffix}")

    if show:
        plt.close(fig)
        show_file(out)
    return out


print("Repo root :", ROOT)
print("Package   :", GPKG.relative_to(ROOT))
print("Output    :", OUT.relative_to(ROOT))
print("Clusters  :", len(CL), "labelled,", int(CL.has_study_cities.sum()), "with study cities")
print("Sources   :", SOURCES_CSV.relative_to(ROOT), "(every read is logged here)")

In [ ]:
# 1) per-city zone GeoDataFrames -------------------------------------------------
zone_results = {}
for d in sorted(p for p in RESULTS_AOI.iterdir() if p.is_dir()):
    s = d.name
    cr  = d / f"{s}_city_region_4326_v5.geojson"
    uc  = d / f"{s}_urban_core_4326_v5.geojson"
    bra = d / f"{s}_bra_4326_v5.geojson"
    if cr.exists() and uc.exists() and bra.exists():
        zone_results[s.replace("_", " ")] = {
            "city_slug": s, "city_dir": d,
            "cr_gdf": gpd.read_file(read_source(cr)), "uc_gdf": gpd.read_file(read_source(uc)),
            "bra_gdf": gpd.read_file(read_source(bra))}
STUDY_CITIES = sorted(zone_results)          # 56 non-absorbed AOIs

# 2) WUP city population table (all candidates) ---------------------------------
wup = pd.read_csv(read_source(ROOT / config.AOI_DATA_PATHS["cities_csv"]))
YEAR_COLS = [c for c in wup.columns if str(c).isdigit()]
for c in YEAR_COLS:
    wup[c] = wup[c].astype(str).str.replace(",", "", regex=False).astype(float)  # thousands
wup_pts = gpd.GeoDataFrame(
    wup, geometry=[Point(xy) for xy in zip(wup["PWCent_Longitude"], wup["PWCent_Latitude"])],
    crs="EPSG:4326")

# 3) WUP cities inside each AOI (for composite / per-AOI trends) -----------------
cities_in_aoi = {}
for city, res in zone_results.items():
    crg = res["cr_gdf"].to_crs("EPSG:4326")
    inside = gpd.sjoin(wup_pts, crg, predicate="within", how="inner")
    inside = inside.drop(columns=[c for c in inside.columns if c.startswith("index_")], errors="ignore")
    cities_in_aoi[city] = {"cities_in_aoi_df": pd.DataFrame(inside.drop(columns="geometry"))}

# 4) authoritative per-AOI population -------------------------------------------
zpop = pd.read_csv(read_source(RESULTS_AOI / "zone_area_summary_with_pop.csv"))
pop_override = {str(r["city"]).strip(): float(r["population_wup_k"]) * 1000
               for _, r in zpop.iterrows() if pd.notna(r["population_wup_k"])}

# 5) active Cluster25 IDs --------------------------------------------------------
lookup = pd.read_csv(read_source(RESULTS_AOI / "city_cluster25_lookup.csv"))
active_clusters = sorted(int(x) for x in lookup["Cluster25"].dropna().unique())

# 6) landscape polygons + reference palette -------------------------------------
landscape_gdf = gpd.read_file(read_source(config.LANDSCAPE_DATA_PATH))
palette = pd.read_csv(read_source(RESULTS_AOI / "cluster25_reference_palette.csv"))

print(f"AOIs (zone_results)      : {len(zone_results)}")
print(f"WUP candidate cities     : {len(wup)}")
print(f"Active Cluster25 IDs (16): {active_clusters}")
print(f"Landscape polygons       : {len(landscape_gdf)}   Palette rows: {len(palette)}")

<a id="nb07-main"></a>

## 1 · Main article

[↑ Contents](#nb07-contents)

<a id="nb07-front"></a>

### Front matter

<a id="nb07-graphical-abstract"></a>

#### Graphical abstract

I prepare the graphical abstract separately; no code here.

<a id="nb07-1-1"></a>

### 1.1 · Methods (manuscript section 2)

[↑ Contents](#nb07-contents)

<a id="nb07-fig-1"></a>

#### Fig. 1 · Study area

I map the 67 Nigerian cities of at least 300,000 inhabitants (UN WUP 2024) on the 24 physiographic landscape typologies, and name the 56 study cities with their 2024 population.

**Data:** package `city_profile`; the UN WUP table, the typology map and the national boundary.  
**Output:** `main_figures/Fig_01_study_area`

In [ ]:
begin("Fig. 1")
_prof1 = pkg("city_profile")
_wup1 = pd.read_csv(read_source(ROOT / config.AOI_DATA_PATHS["cities_csv"]), thousands=",")
_wup1 = _wup1.assign(City_Name=_wup1.City_Name.str.strip()).set_index("City_Name")
_absorbed1 = sorted({n.strip() for s in _prof1.absorbed_cities.dropna() for n in str(s).split(";")})
_clusters1 = sorted(int(c) for c in _prof1.cluster25.unique())
_pop1 = {**dict(zip(_prof1.city, _prof1.population_wup_k * 1000)),
         **(_wup1.loc[_absorbed1, str(STUDY_YEAR)].astype(float) * 1000).to_dict()}
check("Fig. 1 - 67 candidate cities: 56 study cities and 11 absorbed ones, all in the WUP table",
      len(_prof1) == 56 and len(_absorbed1) == 11 and len(_pop1) == 67
      and set(_pop1) <= set(_wup1.index), f"{len(_prof1)} + {len(_absorbed1)} = {len(_pop1)}")
check("Fig. 1 - study-city populations: city_profile.population_wup_k equals the WUP 2024 input",
      np.array_equal(_prof1.population_wup_k.astype(float).values,
                     _wup1.loc[_prof1.city, str(STUDY_YEAR)].astype(float).values),
      f"56 of 56; Lagos {_pop1['Lagos'] / 1e6:.1f} million, Onitsha {_pop1['Onitsha'] / 1e6:.1f} million")
check("Fig. 1 - every candidate city is at or above the 300,000 threshold",
      min(_pop1.values()) >= 300_000, f"smallest {min(_pop1.values()):,.0f} ({min(_pop1, key=_pop1.get)})")
check("Fig. 1 - the legend lists all 24 typology classes with their approved labels",
      len(CL) == 24 and set(_clusters1) <= set(CL.index), f"{len(CL)} classes, {len(_clusters1)} with study cities")

fig1 = viz.create_publication_nigeria_overview(
    aoi_dict=dict.fromkeys(_prof1.city),           # the function uses only the city names
    cities_csv_path=read_source(ROOT / config.AOI_DATA_PATHS["cities_csv"]),
    landscape_path=read_source(config.LANDSCAPE_DATA_PATH),
    nigeria_path=read_source(config.GHSL_POP_CONFIG["boundary_geojson"]),
    cluster25_colors=config.CLUSTER25_COLORS,
    active_cluster_ids=_clusters1,
    study_year=STUDY_YEAR,
    pop_override=_pop1,
    save_path=None, dpi=300, figsize=(8.27, 11.69),
    # the author's changes
    title=None,
    unlabelled_cities=_absorbed1,
    city_legend_label="City (UN WUP 2024, ≥ 300,000 inhabitants)",
    pop_legend_values=[(10_000_000, "10 million"), (3_000_000, "3 million"),
                       (1_000_000, "1 million"), (300_000, "300,000")],
    legend_cluster_ids=sorted(CL.index),
    cluster_labels={i: f"{CL.loc[i].cluster25}  {CL.loc[i].label_short}" for i in CL.index},
    cluster_legend_heading="Landscape typology (climate · geology · elevation):",
    legend_below=True, legend_ncol=3)

paths = savefig(fig1, MAIN_FIG / "Fig_01_study_area", keep_size=True, show=True)

<a id="nb07-fig-2"></a>

#### Fig. 2 · Methodological workflow

I drew the workflow in draw.io; no code here.

<a id="nb07-fig-3"></a>

#### Fig. 3 · Three-zone delineation of five cities

I show the urban core, background reference area and city region of Kano, Abuja, Jos, Lagos and Port Harcourt.

**Data:** NB01 zone, water and road layers; the typology map.  
**Output:** `main_figures/Fig_03_delineation_excerpt`

In [ ]:
begin("Fig. 3")
GRID_CITIES = ["Kano", "Abuja", "Jos", "Lagos", "Port Harcourt"]
fig4 = viz.create_publication_city_grid(
    city_names=GRID_CITIES, zone_results=zone_results, results_aoi_path=read_source(RESULTS_AOI),
    landscape_gdf=landscape_gdf, cluster25_colors=config.CLUSTER25_COLORS,
    storage_crs="EPSG:4326", save_path=None, dpi=300, figsize=(8.27, 5.845),
    single_north_arrow=True)

paths = savefig(fig4, MAIN_FIG / "Fig_03_delineation_excerpt")
plt.show()
print("Saved:", [p.name for p in paths])

<a id="nb07-table-1"></a>

#### Table 1 · Typology composition of the 56 cities

I summarise the study cities by typology cluster: number of cities, their names, the description, the 2024 population and the city-region and urban-core area.

**Data:** package `city_profile`; `cluster_labels.csv`.  
**Output:** `main_tables/Table_01_typology_composition` (.csv, .xlsx)

In [ ]:
begin("Table 1")
_prof1 = pkg("city_profile")
OTHER_CLUSTERS = [6, 8, 15, 21, 23]                  # the single-city clusters, one Other row
_g1 = _prof1.assign(cid=_prof1.cluster25.astype(int)).groupby("cid")
_agg1 = pd.DataFrame({"n": _g1.city.size(), "cities": _g1.city.agg(lambda s: ", ".join(sorted(s))),
                      "pop": _g1.population_wup_merged_k.sum(),
                      "cr": _g1.cr_area_km2.sum(), "uc": _g1.uc_area_km2.sum()})
_multi1 = _agg1.drop(OTHER_CLUSTERS).sort_values("n", ascending=False, kind="stable")
_oth1 = _agg1.loc[OTHER_CLUSTERS]
_num1 = ["WUP pop. (000s)", "CR (km²)", "UC (km²)"]
tab1 = pd.DataFrame(
    [[f"C{c}", int(r["n"]), r["cities"], f"{CL.loc[c].label_short} ({CL.loc[c].elevation_range})",
      float(r["pop"]), float(r["cr"]), float(r["uc"])] for c, r in _multi1.iterrows()]
    + [[f"Other ({len(_oth1)} clusters)", int(_oth1.n.sum()),
        ", ".join(f"{r['cities']} (C{c})" for c, r in _oth1.iterrows()), "Mixed",
        float(_oth1["pop"].sum()), float(_oth1.cr.sum()), float(_oth1.uc.sum())]],
    columns=["Cluster Group", "No. of cities", "Cities", "Physiographic description"] + _num1)

check("Table 1 - rows: 11 clusters by number of cities, then Other",
      list(tab1["Cluster Group"]) == ["C13", "C4", "C9", "C7", "C25", "C11", "C1", "C12", "C16",
                                      "C17", "C19", "Other (5 clusters)"],
      ", ".join(f"{g} {n}" for g, n in zip(tab1["Cluster Group"], tab1["No. of cities"])))
check("Table 1 - the 56 study cities, each in exactly one row",
      tab1["No. of cities"].sum() == 56 and _agg1.n.sum() == _prof1.city.nunique() == 56,
      f"{tab1['No. of cities'].sum()} cities in {len(tab1)} rows")
check("Table 1 - population, CR and UC columns sum to the city_profile totals",
      np.allclose(tab1[_num1].sum().values, [_prof1.population_wup_merged_k.sum(),
                                             _prof1.cr_area_km2.sum(), _prof1.uc_area_km2.sum()]),
      f"{tab1[_num1[0]].sum():,.0f} thousand; {tab1[_num1[1]].sum():,.0f} km² CR; "
      f"{tab1[_num1[2]].sum():,.0f} km² UC")

_uc13, _uct1 = _agg1.loc[13, "uc"], _prof1.uc_area_km2.sum()
FOOT_T1 = (f"C13 alone holds {100 * _uc13 / _uct1:.1f}% of the urban-core (UC) area of the "
           f"{len(_prof1)} study cities ({_uc13:,.0f} of {_uct1:,.0f} km²).")

csv1 = MAIN_TAB / "Table_01_typology_composition.csv"
tab1.assign(**{c: tab1[c].map("{:,.0f}".format) for c in _num1}).to_csv(
    csv1, index=False, encoding="utf-8-sig")

xls1 = MAIN_TAB / "Table_01_typology_composition.xlsx"
wb = xlsxwriter.Workbook(str(xls1)); ws = wb.add_worksheet("Table 1")
hdr  = wb.add_format({"bold": True, "bg_color": "#404040", "font_color": "white",
                      "border": 1, "align": "center", "valign": "vcenter", "text_wrap": True})
cell = wb.add_format({"border": 1, "valign": "vcenter", "text_wrap": True})
num  = wb.add_format({"border": 1, "valign": "vcenter", "num_format": "#,##0"})
foot = wb.add_format({"italic": True, "font_size": 8})
for j, c in enumerate(tab1.columns):
    ws.write(0, j, c, hdr)
for i, r in enumerate(tab1.itertuples(index=False), start=1):
    for j, (c, v) in enumerate(zip(tab1.columns, r)):
        ws.write(i, j, v, num if c in _num1 else cell)
ws.write(len(tab1) + 2, 0, FOOT_T1, foot)
ws.set_column(0, 0, 14); ws.set_column(1, 1, 9); ws.set_column(2, 2, 46)
ws.set_column(3, 3, 44); ws.set_column(4, 6, 11)
wb.close()

print(f"Saved: {csv1.name}, {xls1.name} in {MAIN_TAB.relative_to(ROOT)}")
display(pd.read_csv(csv1, dtype=str, encoding="utf-8-sig"))
print(FOOT_T1)

<a id="nb07-fig-4"></a>

#### Fig. 4 · Stakeholder participants by cluster and city

I draw three rings: the 16 clusters, their participation and the 56 cities, with the 21 cities that sent participants labelled with their counts.

**Data:** package `city_profile`, `stakeholder_perception`.  
**Output:** `main_figures/Fig_04_stakeholder_rings`

In [ ]:
begin("Fig. 4")
_sp4 = pkg("stakeholder_perception")
_cp4 = pkg("city_profile")
_study4 = pd.DataFrame({"city": _cp4.city, "Cluster25": _cp4.cluster25.astype(int),
                        "total_attendees": _cp4.city.map(_sp4.groupby("city").size())
                                           .fillna(0).astype(int)})
_clus4 = (_study4.groupby("Cluster25")
          .agg(**{"N Cities": ("city", "size"), "Attended Workshop": ("total_attendees", "sum")})
          .reset_index().assign(Cluster=lambda d: "C" + d.Cluster25.astype(str)))
_with4 = _study4[_study4.total_attendees > 0]
check("Fig. 4 - the perception cohort: 105 participants, 21 cities, 10 clusters",
      (len(_sp4), _sp4.participant_code.nunique(), _with4.city.nunique(),
       _with4.Cluster25.nunique()) == (105, 105, 21, 10),
      f"{_study4.total_attendees.sum()} participants, {len(_with4)} cities, "
      f"{_with4.Cluster25.nunique()} clusters")
check("Fig. 4 - every participant's city and cluster are in city_profile",
      set(_sp4.city) <= set(_cp4.city)
      and (_sp4.cluster25.astype(int).values
           == _cp4.set_index("city").cluster25.astype(int).reindex(_sp4.city).values).all(),
      "105 of 105")
check("Fig. 4 - the rings hold the 56 study cities in 16 clusters",
      _clus4["N Cities"].sum() == 56 and len(_clus4) == 16
      and _clus4["Attended Workshop"].sum() == len(_sp4),
      ", ".join(f"{r.Cluster} {r['N Cities']}/{r['Attended Workshop']}" for _, r in _clus4.iterrows()))

fig4 = viz.create_city_stakeholder_donut(
    cluster_summary=_clus4[["Cluster", "N Cities", "Attended Workshop"]], study_gdf=_study4,
    cluster25_colors=config.CLUSTER25_COLORS, attended_pids=len(_sp4), figsize=(16, 9),
    show_title=False, legend_count_label="participants")
paths = savefig(fig4, MAIN_FIG / "Fig_04_stakeholder_rings", keep_size=True, show=True)

<a id="nb07-1-2"></a>

### 1.2 · Results 3.1 — long-term and event-scale surface temperature (Objective 1)

[↑ Contents](#nb07-contents)

<a id="nb07-fig-5"></a>

#### Fig. 5 · Annual trends, selected clusters

I draw the annual LST of six clusters with NB03's temporal panel (urban core, background reference area, city-region range and SUHII) and the trend table below. `render_cluster_grid()`, defined here, also draws Fig. 6.

**Data:** NB03 cluster series, checked against package `annual_series`; package `composite_trend_statistics`.  
**Output:** `main_figures/Fig_05_annual_trends`

In [ ]:
begin("Fig. 5")
SELECTED_CLUSTERS = [4, 7, 9, 13, 19, 25]
CL_META = {int(i): (r.climate, int(r.n_cities)) for i, r in CL.iterrows()}

# The table is filled from the published composites, after the NB03 stats behind the panels are
# asserted to agree with them.
_COMP = pkg("composite_trend_statistics")
_COMP = _COMP[_COMP.composite_type == "cluster"]


def assert_trends_match_package(cids, tol=1e-5):
    """The NB03 stats behind the panels and the published composites must agree."""
    worst, n = 0.0, 0
    for cid in cids:
        ent = tp.load_cluster(cid, read_source(STATS03))
        _, _, st, ds = tp.panel_inputs(ent, "annual")
        for zone in ("urban_core", "bra", "delta_uc_bra"):
            src = ds if zone == "delta_uc_bra" else st
            if src is None or src.empty:
                continue
            for period in ("day", "night"):
                row = src[(src["zone"] == zone) & (src["period"] == period)]
                pk = pf.package_trend(_COMP, cid, zone, period)
                if row.empty or pk is None:
                    continue
                a = float(row.iloc[0]["sen_slope_per_decade"])
                b = float(pk.sen_slope_c_per_decade)
                worst = max(worst, abs(a - b)); n += 1
    check("Fig. 5 - NB03 panel statistics match composite_trend_statistics",
          worst <= tol, f"{n} slopes compared, worst difference {worst:.2e}")


def series_vs_package(cids, kind):
    """Worst difference between the NB03 cluster series drawn and the package city series.

    A cluster series is the mean of its cities' values (NB03 §7a); its SUHII is the mean urban
    core minus the mean BRA. The annual CR range (P5-P95) is checked too; the package holds
    no monthly percentiles, so the monthly band stays the NB03 one."""
    P0 = pkg("annual_series") if kind == "annual" else pkg("monthly_series")
    worst, n, gaps = 0.0, 0, 0
    for cid in cids:
        df, ddf, _, _ = tp.panel_inputs(tp.load_cluster(cid, read_source(STATS03)), kind)
        w, k, g, _ = _series_diff(df, ddf, P0[P0.cluster25.astype(int) == cid], kind)
        worst, n, gaps = max(worst, w), n + k, gaps + g
    return worst, n, gaps


assert_trends_match_package(SELECTED_CLUSTERS)
_w5, _n5, _g5 = series_vs_package(SELECTED_CLUSTERS, "annual")
check("Fig. 5 - the annual series drawn equal the package (cluster means of annual_series)",
      _w5 <= 1e-9 and _g5 == 0,
      f"{_n5} values (UC, BRA, CR, CR P5/P95, SUHII), worst difference {_w5:.1e} °C, {_g5} gaps")

_f5 = pf.render_cluster_grid(SELECTED_CLUSTERS, "annual", read_source(STATS03), CL_META,
                             trend_cell=lambda c, z, p: pf.trend_cell_package(_COMP, c, z, p))
paths = savefig(_f5, MAIN_FIG / "Fig_05_annual_trends", keep_size=True, show=True)

<a id="nb07-fig-6"></a>

#### Fig. 6 · Monthly trends, selected clusters

I draw the monthly LST of the same six clusters with the Fig. 5 renderer; run the Fig. 5 cell first.

**Data:** NB03 monthly cluster series and statistics, checked against package `monthly_series`.  
**Output:** `main_figures/Fig_06_monthly_trends`

In [ ]:
begin("Fig. 6")
_w6, _n6, _g6 = series_vs_package(SELECTED_CLUSTERS, "monthly")
check("Fig. 6 - the monthly series drawn equal the package (cluster means of monthly_series)",
      _w6 <= 1e-9 and _g6 == 0,
      f"{_n6} values (UC, BRA, CR, SUHII), worst difference {_w6:.1e} °C, {_g6} gaps")

_f6 = pf.render_cluster_grid(SELECTED_CLUSTERS, "monthly", read_source(STATS03), CL_META)
paths = savefig(_f6, MAIN_FIG / "Fig_06_monthly_trends", keep_size=True, show=True)

<a id="nb07-fig-7"></a>

#### Fig. 7 · Persistent heat events by cluster

I show the mean number of persistent heat events and the mean longest event per city in each cluster, day and night, with the range across its cities and the most common event tier.

**Data:** package `city_year_summary`.  
**Output:** `main_figures/Fig_07_heat_event_bars`

In [ ]:
begin("Fig. 7")
import matplotlib.patches as _mpatch
_cy = pkg("city_year_summary")
_SPEC = [("day_n_era5_persist_evts", "era5_day_n_persistent_events"),
         ("night_n_era5_persist_evts", "era5_night_n_persistent_events"),
         ("day_era5_longest_days", "era5_day_longest_event_days"),
         ("night_era5_longest_days", "era5_night_longest_event_days")]
_g7 = _cy.groupby("cluster25")
_clh = pd.DataFrame({"n_cities": _g7.size()})
for _stem, _col in _SPEC:
    _clh[f"{_stem}_mean"], _clh[f"{_stem}_min"], _clh[f"{_stem}_max"] = (
        _g7[_col].mean(), _g7[_col].min(), _g7[_col].max())
for _per in ("day", "night"):
    _clh[f"{_per}_tier"] = _g7[f"{_per}_selected_event_tier"].agg(lambda s: s.mode().iloc[0])
_clh = _clh.reset_index()
check("Fig. 7 - 56 cities in 16 clusters, the four ERA5 event fields complete",
      len(_cy) == 56 and len(_clh) == 16 and _cy[[c for _, c in _SPEC]].notna().all().all(),
      f"events {int(_cy.era5_day_n_persistent_events.min())}-{int(_cy.era5_day_n_persistent_events.max())} (day), "
      f"{int(_cy.era5_night_n_persistent_events.min())}-{int(_cy.era5_night_n_persistent_events.max())} (night)")
check("Fig. 7 - every cluster has a day and a night event tier for the bar borders",
      _clh[["day_tier", "night_tier"]].isin(["EXCELLENT", "GOOD", "FAIR"]).all().all(),
      "day " + ", ".join(f"C{r.cluster25} {r.day_tier[0]}" for r in _clh.itertuples())
      + "; night " + ", ".join(f"C{r.cluster25} {r.night_tier[0]}" for r in _clh.itertuples()))

_TIER_EC = {"EXCELLENT": "#1a9850", "GOOD": "#f77f00", "FAIR": "#999999"}
_DAY_C, _NGT_C = "#d73027", "#4575b4"

def _bar_group(ax, df_cl, day_col, ngt_col, ylabel, title, bw=0.35):
    _sv = (df_cl.dropna(subset=["cluster25"]).set_index("cluster25")
           [[f"{day_col}_mean", f"{ngt_col}_mean"]].fillna(0)
           .max(axis=1).sort_values(ascending=False))
    cids = _sv.index.tolist(); xs = np.arange(len(cids))
    for xi, cid in enumerate(cids):
        row = df_cl[df_cl["cluster25"] == cid].iloc[0]
        for col, xoff, cc, per in [(day_col, -bw / 2, _DAY_C, "day"), (ngt_col, +bw / 2, _NGT_C, "night")]:
            ec = _TIER_EC.get(str(row.get(f"{per}_tier", "")), "#cccccc")
            m  = float(row.get(f"{col}_mean", 0) or 0)
            lo = float(row.get(f"{col}_min", m) or m)
            hi = float(row.get(f"{col}_max", m) or m)
            ax.bar(xi + xoff, m, width=bw, color=cc, alpha=0.75,
                   edgecolor=ec, linewidth=1.4, zorder=3)
            ax.errorbar(xi + xoff, m, yerr=[[max(m - lo, 0)], [max(hi - m, 0)]],
                        fmt="none", ecolor="#333333", elinewidth=1.0, capsize=2.5, zorder=4)
    ax.set_xticks(xs)
    ax.set_xticklabels([f"C{int(c)}" for c in cids], fontsize=7, rotation=45, ha="right")
    ax.set_xlabel("Cluster", fontsize=8)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_title(title, fontsize=9, pad=4)
    ax.grid(axis="y", linestyle=":", alpha=0.45)
    ax.spines[["top", "right"]].set_visible(False)

fig11, (a11, b11) = plt.subplots(1, 2, figsize=(11.69, 5.85), gridspec_kw={"wspace": 0.28})
_bar_group(a11, _clh, "day_n_era5_persist_evts", "night_n_era5_persist_evts", "Events", "(a)  ERA5 Persistent Events")
_bar_group(b11, _clh, "day_era5_longest_days", "night_era5_longest_days", "Days", "(b)  ERA5 Longest Event")
_leg11 = [
    _mpatch.Patch(color=_DAY_C, alpha=0.75, label="Daytime"),
    _mpatch.Patch(color=_NGT_C, alpha=0.75, label="Night-time"),
    _mpatch.Patch(fc="w", ec="#1a9850", lw=1.5, label="EXCELLENT tier"),
    _mpatch.Patch(fc="w", ec="#f77f00", lw=1.5, label="GOOD tier"),
    _mpatch.Patch(fc="w", ec="#999999", lw=1.5, label="FAIR tier"),
]
# Drop the legend ~1 cm below the plots (clear of the rotated tick labels) with a
# "Legend" header, and keep the descriptive caption as a thin line beneath the box.
fig11.tight_layout(rect=(0, 0.06, 1, 1))
fig11.legend(handles=_leg11, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.017),
             fontsize=8, framealpha=0.9, columnspacing=1.4, handlelength=1.5,
             title="Legend", title_fontsize=9, alignment="center")
fig11.text(0.5, -0.135,
           "Bars = cluster mean across cities  ·  Error bars = city-level min–max  ·  "
           "Bar border = event tier of that period (most common in the cluster)",
           ha="center", va="top", fontsize=7, color="#555", style="italic")
_p11 = savefig(fig11, MAIN_FIG / "Fig_07_heat_event_bars", keep_size=True, show=True)

<a id="nb07-fig-8"></a>

#### Fig. 8 · Event-scale SUHII by city

I show each city's event-scale SUHII from MSG-SEVIRI, day and night, sorted by the daytime value and coloured by cluster.

**Data:** package `city_year_summary`.  
**Output:** `main_figures/Fig_08_seviri_suhii_cities`

In [ ]:
begin("Fig. 8")
_cy8 = pkg("city_year_summary")
_sev8 = pd.concat([pd.DataFrame({"city": _cy8.city, "period": _per,
                                 "suhii": _cy8[f"{_per}_seviri_suhii_c"],
                                 "note": np.where(_cy8[f"{_per}_seviri_suhii_c"].notna(), "ok", "missing")})
                   for _per in ("day", "night")], ignore_index=True)
_clu8 = dict(zip(_cy8.city, _cy8.cluster25.astype(int)))
check("Fig. 8 - SEVIRI SUHII for all 56 study cities, day and night, in 16 clusters",
      len(_cy8) == 56 and (_sev8.note == "ok").sum() == 112 and len(set(_clu8.values())) == 16,
      "national day mean {:.2f} °C [{:.2f} to {:.2f}], night mean {:.2f} °C [{:.2f} to {:.2f}]".format(
          _cy8.day_seviri_suhii_c.mean(), _cy8.day_seviri_suhii_c.min(), _cy8.day_seviri_suhii_c.max(),
          _cy8.night_seviri_suhii_c.mean(), _cy8.night_seviri_suhii_c.min(), _cy8.night_seviri_suhii_c.max()))

fig13 = viz.create_suhii_city_chart(
    df_seviri      = _sev8,
    cluster_lookup = _clu8,
    output_dir     = None,
    dpi            = config.FIGURE_DPI,
    cl_legend_ncol = 4,
    title          = None,
    cl_label       = "C{}",
)
paths = savefig(fig13, MAIN_FIG / "Fig_08_seviri_suhii_cities", keep_size=True, show=True)

<a id="nb07-1-3"></a>

### 1.3 · Results 3.2 — surface hot and cold spots (Objective 2)

[↑ Contents](#nb07-contents)

<a id="nb07-table-2"></a>

#### Table 2 · Significant spots by cluster

I count the spots, the cities and the L50 area of each cluster for every spot type and period, with a national total.

**Data:** package `spots`.  
**Output:** `main_tables/Table_02_significant_spots_by_cluster` (.csv, .xlsx)

In [ ]:
begin("Table 2")
_sp2 = pkg("spots")
_HEAD2 = {"Sig. Hotspots": "Sig. hot spots", "Sig. Cool spots": "Sig. cold spots"}
def _head2(c):
    for a, b in _HEAD2.items():
        c = c.replace(a, b)
    return c

t5 = _pht.build_table5(None, spots=_sp2, cluster_fmt="C{}").rename(columns=_head2)
t5g = t5.copy(); t5g["_group"] = t5g["Cluster"].eq("Total")
_grp2 = [_head2(g) for _, _, g in _pht._T5_GROUPS]
_cl2, _tot2 = t5[t5.Cluster != "Total"], t5[t5.Cluster == "Total"].iloc[0]
check("Table 2 - cluster rows add up to the national row: spots and cities in all four groups",
      all(_cl2[f"{g}|{m}"].sum() == _tot2[f"{g}|{m}"] for g in _grp2
          for m in ("No. of spots", "No. of cities"))
      and sum(_tot2[f"{g}|No. of spots"] for g in _grp2) == len(_sp2),
      "; ".join(f"{g}: {int(_tot2[g + '|No. of spots'])} spots, {int(_tot2[g + '|No. of cities'])} cities"
                for g in _grp2) + f"; {len(_sp2)} spots in all")
_bhg = _sp2.groupby(["city", "period", "spot_type"]).bh_significant
check("Table 2 - BH-FDR survival is one value per city-period-type group",
      (_bhg.nunique() == 1).all(),
      f"{int(_sp2.bh_significant.sum())} of {len(_sp2)} spots, {int(_bhg.first().sum())} of {_bhg.ngroups} groups")
FOOT_T2 = ("Area = total L50 footprint (sq.km). Total row = national aggregate. "
           f"BH-FDR: {int(_sp2.bh_significant.sum())} of the {len(_sp2)} spots, in "
           f"{int(_bhg.first().sum())} of {_bhg.ngroups} city-period-type groups, survive "
           "Benjamini-Hochberg correction; survival is a property of the group, repeated on "
           "each of its spots, not a per-spot test.")

csv2 = MAIN_TAB / "Table_02_significant_spots_by_cluster.csv"
t5.columns = [c.split('|')[-1] if '|' in c else c for c in t5.columns]  # flat for csv
t5.to_csv(csv2, index=False)
_pht._grouped_xlsx(t5g, MAIN_TAB / "Table_02_significant_spots_by_cluster.xlsx",
                   id_cols=['Cluster'], groups=_grp2, subs=_pht._T5_METRICS,
                   title='Table 2: Summary of significant spots occurrence across PLT clusters (56 cities)',
                   footnote=FOOT_T2)

print(f"Saved: {csv2.name}, {csv2.with_suffix('.xlsx').name} in {MAIN_TAB.relative_to(ROOT)}")
_show2 = t5g.drop(columns="_group").set_index("Cluster")
_show2.columns = pd.MultiIndex.from_tuples([tuple(c.split("|")) for c in _show2.columns])
display(_show2)
print(FOOT_T2)

<a id="nb07-fig-9"></a>

#### Fig. 9 · Surface hot and cold spots, all cities, day

I assemble NB05's sheet of the 56 cities' daytime spots with its map legend. The same cell writes Fig. 10.

**Data:** NB05 sheets, checked against package `spots` and `spot_contours`.  
**Output:** `main_figures/Fig_09_spot_maps_day`

In [ ]:
begin("Fig. 9")
import re as _re54mod
from lxml import etree as _ET
import fitz as _fitz
_SVGNS = "http://www.w3.org/2000/svg"
_re54 = _re54mod
_re54_I = _re54mod.IGNORECASE
_re54_W = _re54mod.compile(r'stroke-width:\s*([0-9.]+)')
_ECOMM = _ET.Comment("").__class__

def _gcomment(g):
    for ch in g:
        if isinstance(ch, _ECOMM):
            return (ch.text or "").strip()
    return ""

def _fig1516(src, dst_svg):
    tree = _ET.parse(str(read_source(src))); root = tree.getroot()
    _vb = (root.get("viewBox") or "0 0 575.30448 864.869096").split()
    W, H = float(_vb[2]), float(_vb[3])
    _title = ("Urban Heat Island", "KDE 50% contours")
    drop, arrow = [], None
    for g in root.iter(f"{{{_SVGNS}}}g"):
        gid = g.get("id", ""); c = _gcomment(g)
        if gid.startswith("text_") and any(m in c for m in _title):
            drop.append(g)
        if gid.startswith("axes_"):
            _txts = [ch for ch in g.iter(f"{{{_SVGNS}}}g") if ch.get("id", "").startswith("text_")]
            if _txts and all(_gcomment(t) == "N" for t in _txts):
                arrow = g
    for g in drop:
        g.getparent().remove(g)
    # Water and coastline: NB05 draws these in saturated cyan, which in the creek-laced
    # cities (Port Harcourt, Lagos, Warri) competes with the spots for attention. They
    # are lightened and thinned here, in the same post-processing pass that already
    # edits this SVG, so NB05 itself is untouched.
    # #00e5ff and #0095aa are the COLD-SPOT contour colours, not water: lightening
    # those would wash out the very features the figure is about. Water is the pale
    # blue fill #c6e8ff with the #7ec8e3 stroke, and the fine grey lines are roads.
    _WATER_IN = {"#7ec8e3"}
    _WATER_OUT = "#dcecf3"
    for _el in root.iter():
        _st = _el.get("style")
        if _st:
            _low = _st.lower()
            if any(_w in _low for _w in _WATER_IN):
                for _w in _WATER_IN:
                    _st = _re54.sub(_re54.escape(_w), _WATER_OUT, _st, flags=_re54_I)
                _st = _re54_W.sub(lambda m: "stroke-width:%.2f"
                                  % (float(m.group(1)) * 0.55), _st)
                _el.set("style", _st)
        _sc = _el.get("stroke")
        if _sc and _sc.lower() in _WATER_IN:
            _el.set("stroke", _WATER_OUT)
            if _el.get("stroke-width"):
                _el.set("stroke-width", "%.2f" % (float(_el.get("stroke-width")) * 0.55))
    if arrow is not None:                       # move N-arrow down to the colour-bar level
        dy = ((1 - 0.021) - (1 - 0.972)) * H
        arrow.set("transform", f"translate(-18.000,{dy:.3f}) {arrow.get('transform', '')}".strip())
    # composite the shared map legend into a new bottom strip
    _legW, _legH = 582.559375, 64.8
    root.set("height", f"{H + _legH}pt"); root.set("viewBox", f"0 0 {W} {H + _legH}")
    _defs = root.find(f"{{{_SVGNS}}}defs")
    _lr = _ET.parse(str(read_source(_F05 / "a4_grid_legend.svg"))).getroot()
    _ld = _lr.find(f"{{{_SVGNS}}}defs")
    _have = {c.get("id") for c in _defs if c.get("id")}
    if _ld is not None:
        for c in list(_ld):
            if not (c.get("id") and c.get("id") in _have):
                _defs.append(c)
    _lf = _lr.find(f"{{{_SVGNS}}}g[@id='figure_1']"); _lf.set("id", "legend_fig")
    _wrap = _ET.SubElement(root, f"{{{_SVGNS}}}g")
    _wrap.set("transform", f"translate({(W - _legW) / 2.0:.3f},{H:.3f})"); _wrap.append(_lf)
    dst_svg.parent.mkdir(parents=True, exist_ok=True)
    tree.write(str(dst_svg), xml_declaration=True, encoding="utf-8")
    dst_svg.with_suffix(".pdf").write_bytes(_fitz.open(str(dst_svg)).convert_to_pdf())

# the spots and contours in NB05's tables behind these sheets are the ones the package publishes
# (the sheets of 29 Oct draw only L50: NB05's cache path mislabels the L75 / L90 WKT's CRS)
_sp9, _sc9 = pkg("spots"), pkg("spot_contours")
_nb05sp = pd.read_csv(read_source(RESULTS_05 / "all_cities_spots.csv"))
_nb05ct = pyogrio.read_dataframe(read_source(RESULTS_05 / "master_thermal_spots.gpkg"),
                                 layer="hotspot_contours", read_geometry=False)

for _per, _no, _stem in [("day", "Fig. 9", "Fig_09_spot_maps_day"),
                         ("night", "Fig. 10", "Fig_10_spot_maps_night")]:
    begin(_no)
    _k = ["city", "spot_type"]
    _a, _b = (d[d.period == _per].groupby(_k).size().sort_index() for d in (_nb05sp, _sp9))
    check(f"{_no} - spots per city and type on the NB05 sheet equal the package ({_per})",
          _a.equals(_b), f"{int(_b.sum())} spots in {len(_b)} city-type groups")
    _a, _b = (d[d.period == _per].groupby(_k + ["contour_level"]).size().sort_index()
              for d in (_nb05ct, _sc9))
    check(f"{_no} - NB05 contour table (L50 / L75 / L90) per city and type equals the package ({_per})",
          _a.equals(_b), f"{int(_b.sum())} contours: " + ", ".join(
              f"{lv} {int(n)}" for lv, n in _b.groupby(level="contour_level").sum().items()))
    _src = _F05 / f"all_cities_hotspot_A4_{_per}.svg"
    _dst = MAIN_FIG / f"{_stem}.svg"
    if _src.exists():
        _fig1516(_src, _dst)
        show_file([_dst.with_suffix(".pdf"), _dst])
    else:
        print(f"{_no}: MISSING {_src.name}")

<a id="nb07-fig-10"></a>

#### Fig. 10 · Surface hot and cold spots, all cities, night

The Fig. 9 cell writes the night-time sheet.

**Output:** `main_figures/Fig_10_spot_maps_night`

<a id="nb07-fig-11"></a>

#### Fig. 11 · Spot area by cluster (nested L50 / L75 / L90)

I show each cluster's spot area as nested L90, L75 and L50 segments, for hot and cold spots by day and night.

**Data:** package `spots`, checked against `spot_contours`.  
**Output:** `main_figures/Fig_11_spot_area_by_cluster`

In [ ]:
begin("Fig. 11")
from matplotlib.patches import Patch as _Patch17
_CLR = config.CLUSTER25_COLORS
_sp17, _sc17 = pkg("spots"), pkg("spot_contours")
_g17 = (_sp17.groupby(["spot_type", "period", "cluster25"])
        .agg(L50=("area_l50_km2", "sum"), L75=("area_l75_km2", "sum"),
             L90=("area_l90_km2", "sum"), n=("spot_uid", "size"))
        .reset_index()
        .rename(columns={"spot_type": "Type", "period": "Period", "cluster25": "Cluster"}))
check("Fig. 11 - every spot is in one bar, and L90 <= L75 <= L50 in every bar",
      int(_g17.n.sum()) == len(_sp17)
      and bool(((_g17.L90 <= _g17.L75 + 1e-9) & (_g17.L75 <= _g17.L50 + 1e-9)).all()),
      f"{int(_g17.n.sum())} spots in {len(_g17)} bars")
_lv17 = _sc17.groupby("contour_level").area_km2.sum()
check("Fig. 11 - the L50 / L75 / L90 totals equal spot_contours",
      all(abs(_g17[lv].sum() - _lv17[lv]) < 1e-6 for lv in ("L50", "L75", "L90")),
      ", ".join(f"{lv} {_g17[lv].sum():,.1f} km²" for lv in ("L50", "L75", "L90")))

_PANELS17 = [("hot", "day", "(a)  Hot spot — Day"), ("hot", "night", "(b)  Hot spot — Night"),
             ("cold", "day", "(c)  Cold spot — Day"), ("cold", "night", "(d)  Cold spot — Night")]
fig17, _axes17 = plt.subplots(2, 2, figsize=(11.69, 8.0))
_allcl17 = set()
for _ax, (_typ, _per, _title) in zip(_axes17.flat, _PANELS17):
    _sub = _g17[(_g17.Type == _typ) & (_g17.Period == _per)].copy()
    _sub = _sub[_sub["L50"] > 0].sort_values("L50", ascending=False)
    if _sub.empty:
        _ax.set_visible(False); continue
    _x = np.arange(len(_sub))
    for _xi, _r in zip(_x, _sub.itertuples()):
        _cl = int(_r.Cluster); _col = _CLR.get(_cl, "#888888"); _allcl17.add(_cl)
        _l90, _l75, _l50 = float(_r.L90), float(_r.L75), float(_r.L50)
        _ax.bar(_xi, _l90, color=_col, alpha=1.0, hatch="xxx", edgecolor="white", linewidth=0.3, zorder=3)
        _ax.bar(_xi, max(_l75 - _l90, 0), bottom=_l90, color=_col, alpha=0.72, hatch="//", edgecolor="white", linewidth=0.3, zorder=3)
        _ax.bar(_xi, max(_l50 - _l75, 0), bottom=_l75, color=_col, alpha=0.42, edgecolor="white", linewidth=0.3, zorder=3)
    _ax.set_xticks(_x); _ax.set_xticklabels([f"C{int(c)}" for c in _sub["Cluster"]], rotation=45, ha="right", fontsize=7.5)
    _ax.set_ylabel("Total area (km²)", fontsize=9)
    _ax.set_title(_title, fontsize=11, fontweight="bold", pad=5)
    _ax.grid(axis="y", ls=":", lw=0.4, alpha=0.5); _ax.spines[["top", "right"]].set_visible(False)
fig17.tight_layout(rect=(0, 0.11, 1, 1))
_zh17 = [_Patch17(facecolor="#555", alpha=1.0, hatch="xxx", edgecolor="white", label="L90 dense core"),
         _Patch17(facecolor="#555", alpha=0.72, hatch="//", edgecolor="white", label="L75 inner zone"),
         _Patch17(facecolor="#555", alpha=0.42, edgecolor="white", label="L50 outer extent")]
_lg17a = fig17.legend(handles=_zh17, title="Contour zone", loc="lower left", bbox_to_anchor=(0.02, 0.01),
                      ncol=1, fontsize=8, title_fontsize=9, framealpha=0.9)
_lg17a.get_title().set_fontweight("bold")
_ch17 = [_Patch17(facecolor=_CLR.get(c, "#888"), label=f"C{c}") for c in sorted(_allcl17)]
# _lg17b = fig17.legend(handles=_ch17, title="Cluster", loc="lower center", bbox_to_anchor=(0.60, 0.005),
#                      ncol=8, fontsize=7.5, title_fontsize=9, framealpha=0.9, columnspacing=1.0, handlelength=1.2)
#_lg17b.get_title().set_fontweight("bold")
_p17 = savefig(fig17, MAIN_FIG / "Fig_11_spot_area_by_cluster", keep_size=True, show=True)

<a id="nb07-1-4"></a>

### 1.4 · Results 3.3 — population within spots (Objective 3)

[↑ Contents](#nb07-contents)

<a id="nb07-fig-12"></a>

#### Fig. 12 · Population within hot and cold spots

I map the residents within each city's spots (one circle per city, day and night added) and draw the national and four cluster half-donuts by spot type, period and contour level.

**Data:** package `city_exposure_by_contour`, `city_profile`, `city_zones`; the national boundary.  
**Output:** `main_figures/Fig_12_population_within_spots`

In [ ]:
begin("Fig. 12")
_nigeria20 = gpd.read_file(read_source(ROOT / "data" / "inputs" / "geospatial_features" / "boundary_Nigeria.geojson"))
_hd20 = pkg("city_exposure_by_contour").rename(columns={
    "area_l50_km2": "area_km2_50", "pop_l50_persons": "pop_50", "pop_l75_persons": "pop_75",
    "pop_l90_persons": "pop_90", "cluster25": "cluster"})
_NCLU = pkg("city_profile").groupby("cluster25")["city"].count().to_dict()
_uc20 = pkg("city_zones", geom=True)
_uc20 = _uc20[_uc20.zone == "uc"].to_crs(4326).dissolve("city")
_coords20 = {c: (g.centroid.x, g.centroid.y) for c, g in _uc20.geometry.items()}
check("Fig. 12 - 154 city rows of population and area, with L90 <= L75 <= L50 in each",
      len(_hd20) == 154 and bool(((_hd20.pop_90.fillna(0) <= _hd20.pop_75.fillna(0) + 1e-6)
                                  & (_hd20.pop_75.fillna(0) <= _hd20.pop_50 + 1e-6)).all()),
      f"{len(_hd20)} rows, {_hd20.city.nunique()} cities")
check("Fig. 12 - every study city is placed at its urban-core centroid",
      len(_coords20) == 56 and set(_hd20.city) <= set(_coords20),
      f"{len(_coords20)} cities")
_hotnight20 = _hd20[(_hd20.spot_type == "hot") & (_hd20.period == "night")].pop_50.sum()
check("Fig. 12 - residents in night-time hot spots (L50)",
      round(_hotnight20) == 9_184_401, f"{round(_hotnight20):,}")

fig20, _max20 = pf.population_within_spots(_hd20, _nigeria20, _coords20, _NCLU)
check("Fig. 12 - each map's population key on its own scale, within the range of its circles",
      all(max(v for v, _ in pf.POPULATION_KEY[t]) <= _max20[t] for t in _max20),
      "; ".join(f"{t}: key up to {max(v for v, _ in pf.POPULATION_KEY[t]):,}, largest circle {_max20[t]:,.0f}" for t in _max20))
_p20 = savefig(fig20, MAIN_FIG / "Fig_12_population_within_spots", keep_size=True, show=True)

<a id="nb07-1-5"></a>

### 1.5 · Results 3.4 — stakeholder accounts (Objective 4)

[↑ Contents](#nb07-contents)

<a id="nb07-table-3"></a>

#### Table 3 · Stakeholder accounts against the satellite record

I set the stakeholder accounts of each city, cluster and the nation against the urban-core trends and hottest months, with the agreement and consensus levels.

**Data:** package `stakeholder_perception`, `trend_statistics`, `monthly_climatology`, `monthly_series`.  
**Output:** `main_tables/Table_03_stakeholder_vs_satellite` (.csv, .xlsx)

In [ ]:
begin("Table 3")
_sp3 = pkg("stakeholder_perception")
_ts3 = pkg("trend_statistics")
_mc3 = pkg("monthly_climatology")
_ms3 = pkg("monthly_series")
tab3, _d3, _singles3 = pf.stakeholder_table(_sp3, _ts3, _mc3, _ms3)

_nat3 = tab3.iloc[-1]
check("Table 3 - the perception cohort: 105 participants, 21 cities, 10 clusters",
      (_nat3.N, (tab3.Level == "City").sum(), (tab3.Level == "Cluster").sum()) == (105, 21, 10),
      f"{_nat3.N} participants, {len(_singles3)} single-stakeholder cities")
check("Table 3 - significant urban-core increases, city by city",
      (_nat3.LST_Day_Trend, _nat3.LST_Night_Trend) == ("12/21 increasing", "14/21 increasing"),
      f"day {_nat3.LST_Day_Trend}, night {_nat3.LST_Night_Trend}")
def _cnt3(level, col):
    t = tab3[tab3.Level == level]
    return int(round((t.N * t[col] / 100).sum()))


check("Table 3 - city and cluster rows add up to the same participant counts",
      all(_cnt3("City", c) == _cnt3("Cluster", c) == int(_d3[q].sum())
          for c, q in (("Q61_Annual_Agreement_%", "q61"), ("Q58_Monthly_Agreement_%", "q58"))),
      f"warming {int(_d3.q61.sum())}/105 ({_nat3['Q61_Annual_Agreement_%']} %), "
      f"peak month {int(_d3.q58.sum())}/105 ({_nat3['Q58_Monthly_Agreement_%']} %)")

FOOT_T3 = ("City rows give each city's own urban-core Mann-Kendall trend (significant increase: "
           "p < 0.05); cluster and national rows count the member cities with stakeholders whose "
           "trend is a significant increase, the same per-city basis as the agreement percentages. "
           "Every participant is in the denominators, including answers stored as \"I'd rather not "
           "say\". Diurnal majority: Day, Night or Both, among those who answered; Both where two lead "
           "equally. Peak response: the season named most often (\"I am not sure\" is not a season); "
           "two seasons named equally often are both given; — none named. † One stakeholder: a "
           "single account cannot express agreement within a city.")

csv3 = MAIN_TAB / "Table_03_stakeholder_vs_satellite.csv"
tab3.to_csv(csv3, index=False)
xls3 = MAIN_TAB / "Table_03_stakeholder_vs_satellite.xlsx"
wb = xlsxwriter.Workbook(str(xls3)); ws = wb.add_worksheet("Table 3")
hdr  = wb.add_format({"bold": True, "bg_color": "#404040", "font_color": "white",
                      "border": 1, "align": "center", "valign": "vcenter", "text_wrap": True})
cell = wb.add_format({"border": 1, "valign": "vcenter", "font_size": 9})
foot = wb.add_format({"italic": True, "font_size": 8})
for j, c in enumerate(tab3.columns):
    ws.write(0, j, c, hdr)
for i, (_, r) in enumerate(tab3.iterrows(), start=1):
    for j, c in enumerate(tab3.columns):
        ws.write(i, j, "" if pd.isna(r[c]) else r[c], cell)
ws.write(len(tab3) + 2, 0, FOOT_T3, foot)
ws.set_column(0, 1, 12); ws.set_column(2, 2, 8); ws.set_column(3, 14, 11); ws.set_row(0, 28)
wb.close()

print(f"Saved: {csv3.name}, {xls3.name} in {MAIN_TAB.relative_to(ROOT)}")
display(pd.read_csv(csv3))
print(FOOT_T3)

<a id="nb07-fig-13"></a>

#### Fig. 13 · Stakeholder perception by cluster

I show the answers by cluster: warming over ten years, timing of the most severe heat and the peak season, each with the national donut.

**Data:** package `stakeholder_perception`.  
**Output:** `main_figures/Fig_13_stakeholder_perception`

In [ ]:
begin("Fig. 13")
_pc9 = pkg("stakeholder_perception")
check("Fig. 13 - the perception cohort: 105 participants, 21 cities, 10 clusters",
      len(_pc9) == 105 and _pc9.city.nunique() == 21 and _pc9.cluster25.nunique() == 10,
      f"{len(_pc9)} participants, {_pc9.city.nunique()} cities, {_pc9.cluster25.nunique()} clusters")
_items9 = {"warming": "perceived_warming_10yr", "timing": "perceived_diurnal_severity",
           "peak season": "perceived_peak_period"}
_na9 = "I'd rather not say"
check("Fig. 13 - every participant answers all three items, blanks stored as I'd rather not say",
      int(_pc9[list(_items9.values())].isna().sum().sum()) == 0,
      ", ".join(f"{k} {int((_pc9[c] == _na9).sum())}" for k, c in _items9.items()) + " stored blanks")
_pc9 = _pc9.assign(cluster_num=_pc9.cluster25.astype(int))
_cl_order9 = _pc9["cluster_num"].value_counts().index.tolist()
_w_ct9 = pd.crosstab(_pc9["cluster_num"], _pc9["perceived_warming_10yr"]).reindex(_cl_order9)
_d_ct9 = pd.crosstab(_pc9["cluster_num"], _pc9["perceived_diurnal_severity"]).reindex(_cl_order9)
_p_ct9 = pd.crosstab(_pc9["cluster_num"], _pc9["perceived_peak_period"]).reindex(_cl_order9)

fig13, _fig13_donuts = viz.create_cluster_perception_chart(
    warming_by_cluster = _w_ct9,
    diurnal_by_cluster = _d_ct9,
    warming_total      = _pc9["perceived_warming_10yr"].value_counts(),
    diurnal_total      = _pc9["perceived_diurnal_severity"].value_counts(),
    peak_by_cluster    = _p_ct9,
    peak_total         = _pc9["perceived_peak_period"].value_counts(),
    cluster25_colors   = config.CLUSTER25_COLORS,
    figsize            = (20, 6),
    legend_scale       = 1.0,
    suptitle           = False,
    row_label          = "C{}",
    panel_letters      = ("(a)", "(b)", "(c)"),
)
plt.close(_fig13_donuts)
paths = savefig(fig13, MAIN_FIG / "Fig_13_stakeholder_perception", keep_size=True, show=True)

<a id="nb07-fig-14"></a>

#### Fig. 14 · Hit rate against buffer distance

I show the share of stakeholder-marked points that fall in a detected spot as the buffer grows from 0.5 to 10 km, hot and cold.

**Data:** package `hitrate_sensitivity`.  
**Output:** `main_figures/Fig_14_hitrate_buffer`

In [ ]:
begin("Fig. 14")
sens_df = pkg("hitrate_sensitivity")
_a500 = sens_df[sens_df.buffer_m == 500].groupby(["spot_type", "period"])[["n_hit", "n_total"]].sum()
_want = {("hot", "day"): (4, 55), ("hot", "night"): (3, 55),
         ("cold", "day"): (4, 39), ("cold", "night"): (0, 39)}
_got = {k: (int(_a500.loc[k, "n_hit"]), int(_a500.loc[k, "n_total"])) for k in _want}
check("Fig. 14 - hit rates at 500 m on the cohort points (55 hot / 39 cold)", _got == _want,
      "; ".join(f"{k[0]} {k[1]} {v[0]}/{v[1]}" for k, v in _got.items()))


fig14 = pf.hitrate_buffer(sens_df, "national")
paths = savefig(fig14, MAIN_FIG / "Fig_14_hitrate_buffer", keep_size=True, show=True)

<a id="nb07-1-6"></a>

### 1.6 · Results 3.5 — typology and the thermal anomalies (Objective 5)

[↑ Contents](#nb07-contents)

<a id="nb07-table-4"></a>

#### Table 4 · Typology–anomaly association

I report the association of the typology with spot occurrence, SUHII and area: the test, the effect size and the p-value.

**Data:** NB06 `partA_synthesis_cluster25.csv`.  
**Output:** `main_tables/Table_04_typology_association` (.csv, .xlsx)

In [ ]:
begin("Table 4")
t6 = _pht.build_table6(read_source(RESULTS_06T))
_lab4 = t6[["Spot", "Period"]].replace("", np.nan).ffill()
_sig4 = t6[t6.p.str.contains(r"\*", regex=True)]
check("Table 4 - five of the twelve typology tests reach p < 0.05",
      len(t6) == 12 and len(_sig4) == 5,
      "; ".join(f"{_lab4.loc[i].Spot.lower()} {_lab4.loc[i].Period.lower()} {r.Outcome} p = {r.p}"
                for i, r in _sig4.iterrows()))
FOOT_T4 = ("* p < 0.05; ** p < 0.01. V: large ≥ 0.50; η²: large ≥ 0.14. "
           "MC χ² = Monte-Carlo permutation χ²; KW = Kruskal–Wallis.")
csv4 = MAIN_TAB / "Table_04_typology_association.csv"
t6.to_csv(csv4, index=False)
_pht._simple_xlsx(t6, MAIN_TAB / "Table_04_typology_association.xlsx",
                  title="Table 4: PLT — Thermal anomaly association result",
                  footnote=FOOT_T4, bold_sig_cols=["p"])
print(f"Saved: {csv4.name}, {csv4.with_suffix('.xlsx').name} in {MAIN_TAB.relative_to(ROOT)}")
display(pd.read_csv(csv4, dtype=str).fillna(""))
print(FOOT_T4)

<a id="nb07-table-5"></a>

#### Table 5 · Best single-element attribution (excerpt)

I compare the typology's effect with its strongest single element for six outcomes.

**Data:** NB06 `master_synthesis.csv`.  
**Output:** `main_tables/Table_05_best_element_attribution_excerpt` (.csv, .xlsx)

In [ ]:
begin("Table 5")
_s8_5, t7, _ms5 = _pht.build_supp8_and_table7(read_source(RESULTS_06T))
_occ5 = _ms5.Outcome.str.lower().str.startswith("occ")
_d5 = (np.where(_occ5, _ms5["A: V (Cramér)"], _ms5["A: η²"])
       - np.where(_occ5, _ms5["B: V (Cramér)"], _ms5["B: η²"]) - _ms5["Δ effect size"])
check("Table 5 - the six excerpt outcomes; every Δ is composite minus element",
      len(t7) == 6 and float(np.nanmax(np.abs(_d5))) < 1e-9,
      f"{len(t7)} of {len(_ms5)} outcomes; worst |Δ - (composite - element)| {float(np.nanmax(np.abs(_d5))):.1e}")
FOOT_T5 = ("Composite PLT vs best contributing element. (***/**/* /ns) = element significance. "
           "Δ = composite − element effect size (η² for SUHII/Area; V for Occurrence).")
csv5 = MAIN_TAB / "Table_05_best_element_attribution_excerpt.csv"
t7.to_csv(csv5, index=False)
_pht._simple_xlsx(t7, MAIN_TAB / "Table_05_best_element_attribution_excerpt.xlsx",
                  title="Table 5: Excerpt — best PLT single-element attribution to thermal anomalies",
                  footnote=FOOT_T5)
print(f"Saved: {csv5.name}, {csv5.with_suffix('.xlsx').name} in {MAIN_TAB.relative_to(ROOT)}")
display(pd.read_csv(csv5, dtype=str))
print(FOOT_T5)

<a id="nb07-fig-15"></a>

#### Fig. 15 · Effect sizes (forest plot)

I plot the effect sizes of the typology against its three elements for occurrence, SUHII and area.

**Data:** NB06 part A and part B tables.  
**Output:** `main_figures/Fig_15_association_effect_sizes`

In [ ]:
begin("Fig. 15")
_star = _pht._star          # '***' / '**' / '*' from NB06's star text; '' for ns
_pa = pd.read_csv(read_source(RESULTS_06T / 'partA_synthesis_cluster25.csv'))
_pb = {(_st, _pe): pd.read_csv(read_source(RESULTS_06T / f'partB_{_st}_{_pe}.csv'))
       for _st in pf.SPOTS for _pe in pf.PERIODS}
_paSRC, _pbR, _pbS = pf.association_results(_pa, _pb)
_nsig15 = sum(bool(v["st"]) for o in _paSRC.values() for v in o.values())
check("Fig. 15 - the composite typology: 12 tests, 5 at p < 0.05 (as Table 4); 36 element tests",
      sum(len(o) for o in _paSRC.values()) == 12 and _nsig15 == 5 and len(_pbR) == 36,
      f"{_nsig15} of 12 composite tests significant; {len(_pbR)} element effect sizes, "
      f"{sum(map(bool, _pbS.values()))} significant after Holm")
# NB06 writes the same stars a second time in master_synthesis (A: sig; B: sig for the best element)
_ms15 = pd.read_csv(read_source(RESULTS_06T / 'master_synthesis.csv'))
_okey = {v: k for k, v in pf.OUTCOME_LABEL.items()}
_bad15 = []
for _, _r in _ms15.iterrows():
    _st, _pe, _o = str(_r['Spot Type']).lower(), str(_r['Period']).lower(), _okey[_r['Outcome']]
    if _paSRC[_o][(_st, _pe)]['st'] != _star(_r['A: sig']):
        _bad15.append(f"{_st} {_pe} {_o} composite")
    if _pbS[(_st, _pe, pf.ELEMENT_KEY[_r['B: Best element']], _o)] != _star(_r['B: sig']):
        _bad15.append(f"{_st} {_pe} {_o} {_r['B: Best element']}")
check("Fig. 15 - every star drawn equals NB06's own (partA / partB against master_synthesis)",
      len(_ms15) == 12 and not _bad15, "; ".join(_bad15) or "12 composite and 12 best-element stars agree")
fig19 = pf.association_effect_sizes(_paSRC, _pbR, _pbS)
_p19 = savefig(fig19, MAIN_FIG / "Fig_15_association_effect_sizes", keep_size=True, show=True)

<a id="nb07-supp-figures"></a>

## 2 · Supplementary figures

[↑ Contents](#nb07-contents)

<a id="nb07-s1"></a>

#### S1 · Population trajectories: S1.0 national, S1.1 cities, S1.2 clusters

I draw the UN WUP population, 1975–2050, for the nation, each study city and each cluster. The first cell shows the national chart, Uyo and C13; the second writes all charts.

**Data:** the UN WUP table; package `city_profile`.  
**Output:** `supplementary_figures/S1_population_trajectories/`

In [ ]:
begin("S1")
import textwrap
S1_EXAMPLE = {"city": "Uyo", "cluster": 13}

_prof1 = pkg("city_profile")
_abs1 = {a.strip(): r.city for _, r in _prof1.dropna(subset=["absorbed_cities"]).iterrows()
         for a in str(r.absorbed_cities).split(";")}
_mk1 = _prof1.set_index("city").population_wup_merged_k
_pk1 = _prof1.set_index("city").population_wup_k
_cl1 = _prof1.groupby(_prof1.cluster25.astype(int)).city.apply(sorted)
_start1 = {r["City_Name"]: next(y for y in YEAR_COLS if pd.notna(r[y])) for _, r in wup.iterrows()}


def _rollup(city):
    """The city's own WUP row and the rows of the cities it absorbed."""
    return (wup[wup["City_Name"].isin([city] + [a for a, q in _abs1.items() if q == city])]
            .rename(columns={"City_Name": "Urban Agglomeration"}))


def _cluster_rows(cid):
    """One row per study city of the cluster, absorbed cities summed in (missing stays missing)."""
    return pd.DataFrame([{**{y: _rollup(c)[y].sum(min_count=1) for y in YEAR_COLS},
                          "Urban Agglomeration": c} for c in _cl1[cid]])


def _drawn(df):
    """The chart's rows without the years in which no row has a value (summed, they would be 0)."""
    return df.drop(columns=[y for y in YEAR_COLS if df[y].isna().all()])


def _late(names, first, place=True):
    """'City (first year)' for each WUP city whose series starts after the chart's first year."""
    return [f"{n} ({_start1[n]}, in {_abs1[n]})" if place and n in _abs1 else f"{n} ({_start1[n]})"
            for n in sorted(names) if int(_start1[n]) > int(first)]


def _footnote(fig, entries, first, fs):
    if entries:
        txt = (f"Series starting after {first} (first year with data in brackets): "
               + ", ".join(entries) + ".")
        fig.text(0.01, 0.0, textwrap.fill(txt, int(fig.get_size_inches()[0] * 72 / (fs * 0.52))),
                 fontsize=fs, ha="left", va="top", color="#333333")


def _members(cid):
    return [m for c in _cl1[cid] for m in _rollup(c)["Urban Agglomeration"]]


S1_DIR = SUP_FIG / "S1_population_trajectories"
CITY_TREND_DIR = S1_DIR / "S1.1_city_trends"
CLUS_TREND_DIR = S1_DIR / "S1.2_cluster_trends"
for _d in (S1_DIR, CITY_TREND_DIR, CLUS_TREND_DIR):
    _d.mkdir(exist_ok=True)
S11 = [(f"S1.1.{i}", c) for i, c in enumerate([c for c in wup["City_Name"] if c in _pk1.index], start=1)]
S12 = [(f"S1.2.{i}", cid) for i, cid in enumerate(sorted(_cl1.index), start=1)]

# ── checks ──
_r2024 = pd.Series({c: _rollup(c)["2024"].sum() for _, c in S11})
check("S1.0 / S1.1 - each study city's 2024 population, with the cities it absorbed, equals "
      "city_profile.population_wup_merged_k",
      len(S11) == 56 and (_r2024 - _mk1.reindex(_r2024.index)).abs().max() == 0,
      f"{len(S11)} cities, {len(_abs1)} absorbed; total {_r2024.sum():,.0f} thousand")
check("S1.1 - the 67 WUP candidates: 56 study cities (own 2024 = city_profile.population_wup_k) "
      "and 11 absorbed, each in its study city's chart",
      len(wup) == 67 and all(wup.set_index("City_Name").loc[c, "2024"] == _pk1[c] for _, c in S11)
      and sorted(set(wup["City_Name"]) - set(_pk1.index)) == sorted(_abs1)
      and set(_abs1.values()) <= set(_pk1.index),
      f"{len(S11)} study cities, {len(_abs1)} absorbed into {len(set(_abs1.values()))}")
_bad12 = [f"C{cid}" for _, cid in S12 if _cluster_rows(cid)["2024"].sum() != _mk1[_cl1[cid]].sum()]
check("S1.2 - the 16 clusters with study cities (as NB01's lookup), each 2024 total equal to the "
      "sum of its cities' population_wup_merged_k",
      sorted(_cl1.index) == active_clusters and not _bad12,
      ", ".join(_bad12) or f"{len(S12)} clusters")
_charts1 = ([(c, _drawn(_rollup(c)), list(_rollup(c)["Urban Agglomeration"])) for _, c in S11]
            + [(f"C{cid}", _drawn(_cluster_rows(cid)), _members(cid)) for _, cid in S12])
_yrs = lambda df: [y for y in df.columns if str(y).isdigit()]
_zero1 = [n for n, df, _ in _charts1 if (df[_yrs(df)].sum() <= 0).any()]
_foot1 = {n: len(_late(m, _yrs(df)[0])) for n, df, m in _charts1 if _late(m, _yrs(df)[0])}
_nat1 = _late(wup["City_Name"], YEAR_COLS[0])
check("S1 - no year is drawn without data, and each chart footnotes the member cities that start later",
      not _zero1 and len(_nat1) == int(wup[YEAR_COLS[0]].isna().sum()),
      f"drawn as 0: {', '.join(_zero1) or 'none'}; footnoted: national {len(_nat1)}, "
      + ", ".join(f"{n} {k}" for n, k in _foot1.items()) + "; charts starting after "
      + f"{YEAR_COLS[0]}: " + ", ".join(f"{n} {_yrs(df)[0]}" for n, df, _ in _charts1
                                       if _yrs(df)[0] != YEAR_COLS[0]))


# ── drawing ──
def plot_city_trend(city, save_noext):
    df = _drawn(_rollup(city))
    fig = viz.create_population_trends_plot({"cities_in_aoi_df": df}, city, figsize=(8, 5))
    _footnote(fig, _late(df["Urban Agglomeration"], _yrs(df)[0], place=False), _yrs(df)[0], 7.5)
    p = savefig(fig, save_noext, keep_size=True)
    plt.close(fig); return p


def plot_cluster_trend(cid, save_noext):
    df = _drawn(_cluster_rows(cid))
    n, n_abs = len(_cl1[cid]), sum(q in _cl1[cid] for q in _abs1.values())
    title = f"C{cid} — {n} {'city' if n == 1 else 'cities'}" + (f" (+{n_abs} absorbed)" if n_abs else "")
    fig = viz.create_population_trends_plot({"cities_in_aoi_df": df}, title, figsize=(9, 5))
    _footnote(fig, _late(_members(cid), _yrs(df)[0]), _yrs(df)[0], 7.5)
    p = savefig(fig, save_noext, keep_size=True)
    plt.close(fig); return p


def s11_path(tag, city):
    return CITY_TREND_DIR / f"Supp_Fig_{tag}_{city.replace(' ', '_').replace('/', '-')}"


def s12_path(tag, cid):
    return CLUS_TREND_DIR / f"Supp_Fig_{tag}_C{cid}"


# ── S1.0 · national composite ──
_compat1 = {c: {"cities_in_aoi_df": _rollup(c)} for _, c in S11}
fig_s10, stats_s10 = viz.create_composite_population_trend(
    _compat1, save_dir=None, figsize=(11.69, 5.845), dpi=300,
    mode="aoi", target_cities=list(_compat1))
check("S1.0 - the 2024 total drawn equals the package (sum of population_wup_merged_k)",
      stats_s10["pop_2024_k"] == _mk1.sum() and stats_s10["n_aois"] == 56,
      "{pop_2024_k:,.0f} thousand over {n_aois} cities".format(**stats_s10))
_footnote(fig_s10, _nat1, YEAR_COLS[0], 9.75)
savefig(fig_s10, S1_DIR / "Supp_Fig_S1.0_national_population_trend", keep_size=True, show=True)

# ── S1.1 and S1.2 · one of each, shown here ──
_t, _c = next((t, c) for t, c in S11 if c == S1_EXAMPLE["city"])
_ex = plot_city_trend(_c, s11_path(_t, _c))[0]
show_file([_ex])
_t, _cid = next((t, c) for t, c in S12 if c == S1_EXAMPLE["cluster"])
_ex = plot_cluster_trend(_cid, s12_path(_t, _cid))[0]
show_file([_ex])

In [ ]:
begin("S1")
# ── S1 · every figure of S1.1 (56 study cities) and S1.2 (16 clusters) to the folder, none shown; skipped when current ──
_stamp1 = code_stamp(
    plot_city_trend, plot_cluster_trend, _rollup, _cluster_rows, _drawn, _late,
    _footnote, _members, viz.create_population_trends_plot, inputs=[ROOT / config.AOI_DATA_PATHS["cities_csv"]])
_files1 = [Path(f"{s11_path(t, c)}.pdf") for t, c in S11] + [Path(f"{s12_path(t, c)}.pdf") for t, c in S12]
if set_is_current(S1_DIR, _stamp1, _files1):
    print("S1: the set is current (same drawing code and inputs), not redrawn; REBUILD = True redraws")
else:
    for _t, _c in S11:
        plot_city_trend(_c, s11_path(_t, _c))
    for _t, _cid in S12:
        plot_cluster_trend(_cid, s12_path(_t, _cid))
    mark_current(S1_DIR, _stamp1)
    print(f"S1.1: {len(S11)} city figures; S1.2: {len(S12)} cluster figures; "
          f"in {S1_DIR.relative_to(OUT)}")
pd.DataFrame([{"supp_fig": t, "city": c,
               "absorbed_cities": "; ".join(sorted(a for a, q in _abs1.items() if q == c)),
               "first_year": _yrs(_drawn(_rollup(c)))[0], "pop_2024_k": _mk1[c]}
              for t, c in S11]).to_csv(CITY_TREND_DIR / "_index_S1.1_city_trends.csv", index=False)
pd.DataFrame([{"supp_fig": t, "cluster": f"C{c}", "n_cities": len(_cl1[c]), "cities": "; ".join(_cl1[c]),
               "first_year": _yrs(_drawn(_cluster_rows(c)))[0],
               "pop_2024_k": _cluster_rows(c)["2024"].sum()}
              for t, c in S12]).to_csv(CLUS_TREND_DIR / "_index_S1.2_cluster_trends.csv", index=False)

<a id="nb07-s2"></a>

#### S2 · City delineation maps

I map each city's three zones on the typology, with water, roads and places. The first cell shows Uyo; the second writes all 56 maps.

**Data:** NB01 zone and feature layers; the typology map; package `city_profile`.  
**Output:** `supplementary_figures/S2_city_delineation_maps/`

In [ ]:
begin("S2")
S2_EXAMPLE = "Uyo"

_prof2 = pkg("city_profile").set_index("city")
_land2 = landscape_gdf.to_crs("EPSG:4326")
_land2["geometry"] = _land2.geometry.buffer(0)
S2_DIR = SUP_FIG / "S2_city_delineation_maps"
S2_DIR.mkdir(exist_ok=True)
S2 = [(f"S2.{i}", c) for i, c in enumerate(STUDY_CITIES, start=1)]


def _area2(gdf):
    """A zone's area as the map title gives it (km², ESRI:102022)."""
    return gdf.to_crs("ESRI:102022").geometry.buffer(0).area.sum() / 1e6


_zcol2 = {"uc_gdf": "uc_area_km2", "bra_gdf": "bra_area_km2", "cr_gdf": "cr_area_km2"}
_dev2 = max(abs(round(_area2(zone_results[c][z]), 4) - _prof2.loc[c, col])
            for _, c in S2 for z, col in _zcol2.items())
check("S2 - the 56 study cities, and each map's UC / BRA / CR areas equal city_profile "
      "(to the package's four decimals)",
      sorted(STUDY_CITIES) == sorted(_prof2.index) and _dev2 < 1e-9,
      f"{len(S2)} cities, 168 zone areas; largest difference {_dev2:.1e} km²")


def _layer2(p):
    """A cached NB01 layer (water, roads, places), or None when the city has none."""
    if not p.exists():
        return None
    g = gpd.read_file(read_source(p))
    return None if g.empty else g


def _repair2(g):
    out = g.to_crs("EPSG:4326").copy()
    out["geometry"] = out.geometry.buffer(0)
    return out


def s2_path(tag, city):
    return S2_DIR / f"Supp_Fig_{tag}_{zone_results[city]['city_slug']}"


def plot_city_map(city, save_noext):
    z, d, s = zone_results[city], zone_results[city]["city_dir"], zone_results[city]["city_slug"]
    cr, uc, bra = _repair2(z["cr_gdf"]), _repair2(z["uc_gdf"]), _repair2(z["bra_gdf"])
    fig = viz.create_geospatial_features_map(
        city_name=city,
        aoi_gdf=gpd.GeoDataFrame(geometry=[cr.geometry.union_all().convex_hull], crs="EPSG:4326"),
        cr_gdf=cr, main_polygon_gdf=uc, adjacent_polygons_gdf=bra,
        landscape_gdf=_land2, cluster25_colors=config.CLUSTER25_COLORS,
        water_gdf=_layer2(d / f"{s}_water_bodies_4326.geojson"),
        roads_gdf=_layer2(d / "osm" / f"{s}_roads_4326.geojson"),
        places_gdf=_layer2(d / "osm" / f"{s}_places_4326.geojson"),
        place_types_filter=["town", "suburb"], label_buffer_km=1.4, figsize=(12, 10))
    p = savefig(fig, save_noext, keep_size=True)
    plt.close(fig); return p


# ── one city, shown here ──
_t, _c = next((t, c) for t, c in S2 if c == S2_EXAMPLE)
_ex = plot_city_map(_c, s2_path(_t, _c))[0]
show_file([_ex])

In [ ]:
begin("S2")
# ── S2 · all 56 city maps to the folder, none shown; skipped when current ──
_stamp2 = code_stamp(
    plot_city_map, _layer2, _repair2, viz.create_geospatial_features_map, inputs=[RESULTS_AOI, config.LANDSCAPE_DATA_PATH])
_files2 = [Path(f"{s2_path(t, c)}.pdf") for t, c in S2]
if set_is_current(S2_DIR, _stamp2, _files2):
    print("S2: the set is current (same drawing code and inputs), not redrawn; REBUILD = True redraws")
else:
    for _t, _c in S2:
        plot_city_map(_c, s2_path(_t, _c))
    mark_current(S2_DIR, _stamp2)
    print(f"S2: {len(S2)} city maps in {S2_DIR.relative_to(OUT)}")
pd.DataFrame([{"supp_fig": t, "city": c, "cluster": f"C{int(_prof2.loc[c, 'cluster25'])}",
               **{col: _prof2.loc[c, col] for col in _zcol2.values()}}
              for t, c in S2]).to_csv(S2_DIR / "_index_S2_city_delineation_maps.csv", index=False)

<a id="nb07-s3"></a>

#### S3 · Stakeholder sample maps: S3.1 by cluster, S3.2 by city

I map the registered stakeholders and their professions by cluster and by city, with the attended and registered counts.

**Data:** NB02's stakeholder loader; the UN WUP table and the boundary; package `city_profile`, `stakeholder_perception`.  
**Output:** `supplementary_figures/S3_stakeholder_maps/`

In [ ]:
begin("S3")
import contextlib, io, logging
import data_io
logging.getLogger("adjustText").setLevel(logging.ERROR)   # its notice on arrow styles

_prof3 = pkg("city_profile").set_index("city")
_pp3 = pkg("stakeholder_perception")
for _p in (config.get_aoi_data_path("boundary_geojson"), config.get_aoi_data_path("cities_csv"),
           RESULTS_AOI / "study_cities_final.csv", RESULTS_AOI / "city_cluster25_lookup.csv",
           config.INPUT_DIR / "stakeholder_survey" / "master_composite.csv"):
    read_source(_p)                                      # the files NB02's loader opens
with contextlib.redirect_stdout(io.StringIO()):          # its long filter report is not shown
    _bnd3, _wup3, _sv3 = data_io.load_stakeholder_datasets(config)

_cl3 = _prof3.cluster25.astype(int)
_reg3, _ws3, _att3, _wp3, _cwp3, _regc3, _cities3 = data_io.stakeholder_counts(_sv3, _wup3, _cl3, config)
_pcol3 = pf.profession_colours(_wp3.profession)

# ── checks ──
check("S3 - attendance: NB02's workshop attendees equal the package's participants, city by city",
      _att3.sort_index().equals(_pp3.groupby("city").size().sort_index()) and _ws3.pid.nunique() == len(_pp3),
      f"{_ws3.pid.nunique()} participants, {len(_att3)} cities, {_cwp3.Cluster25.nunique()} clusters")
_cache3 = pd.read_csv(read_source(ROOT / "Results/02_stakeholder_analysis/Data/Cities_Stakeholder_Summary_2024.csv"))
_ctab3 = pd.read_csv(read_source(ROOT / "Results/02_stakeholder_analysis/cluster25_stakeholder_distribution.csv"))
check("S3 - registrations: per city and per cluster as NB02's cached tables, all in study cities",
      (_reg3.reindex(_cache3.city).fillna(0).astype(int).values == _cache3.registered_count.values).all()
      and (_regc3.reindex(_ctab3.Cluster.str[1:].astype(int)).fillna(0).astype(int).values
           == _ctab3.Registered.values).all() and set(_reg3.index) <= set(_prof3.index),
      f"{_sv3.pid.nunique()} registered stakeholders, {_reg3.sum()} city registrations in {len(_reg3)} cities")
_wpc3 = pd.read_csv(read_source(ROOT / "Results/02_stakeholder_analysis/Data/Workshop_Profession_Breakdown_2024.csv"))
check("S3 - professions: as NB02's cached breakdown, each city's and cluster's adding up to its attendance",
      _wp3.set_index(["city", "profession"])["count"].sort_index()
      .equals(_wpc3.set_index(["city", "profession"])["count"].sort_index())
      and _wp3.groupby("city")["count"].sum().equals(_att3.rename_axis("city").rename("count"))
      and _cwp3["count"].sum() == len(_pp3),
      f"{_wp3.profession.nunique()} professions")
check("S3 - map population: the 67 WUP cities, the 56 study cities' 2024 = city_profile.population_wup_k",
      len(_cities3) == 67 and all(_cities3.set_index("city").loc[c, "population"] == _prof3.loc[c, "population_wup_k"]
                                  for c in _prof3.index), f"{len(_cities3)} cities")
_ncity3 = _cl3.value_counts()                            # study cities per cluster
check("S3 - bar labels: each cluster's number of study cities (city_profile) as NB02's cluster table",
      (_ncity3.reindex(_ctab3.Cluster.str[1:].astype(int)).fillna(0).astype(int).values
       == _ctab3["N Cities"].values).all(),
      ", ".join(f"C{c} {_ncity3[c]}" for c in _regc3.index))

# ── the two figures ──
S3_DIR = SUP_FIG / "S3_stakeholder_maps"
S3_DIR.mkdir(exist_ok=True)
_args3 = dict(cities_gdf=_cities3, boundary_gdf=_bnd3, workshop_prof=_wp3, profession_colors=_pcol3,
              survey_df=_sv3, workshop_total=_att3, global_registered=_sv3.pid.nunique(),
              global_attendees=_ws3.pid.nunique(), figsize=(11.7, 5.85),
              panel_letters=("(a)", "(b)"), bar_legend_title="Discipline/Profession")
for _no, _name, _v3, _kw in (
        ("S3.1", "Supp_Fig_S3.1_cluster_stakeholder_map", "Supp_Fig_3.1_cluster_stakeholder_map.svg",
         dict(cluster_workshop_prof=_cwp3, registered_by=_regc3.to_dict(),
              bar_labels={c: f"C{c} ({n})" for c, n in _ncity3.items()})),
        ("S3.2", "Supp_Fig_S3.2_city_stakeholder_map", "Supp_Fig_3.2_city_stakeholder_map.pdf",
         dict(registered_by=_reg3.to_dict(), bar_labels={c: f"{c} (C{_cl3[c]})" for c in _att3.index}))):
    _f = viz.create_stakeholder_map_with_bar_chart(**_args3, **_kw)
    _ex = savefig(_f, S3_DIR / _name, keep_size=True)[0]
    plt.close(_f)
    show_file([_ex])

<a id="nb07-s4"></a>

#### S4 · Long-term trends: S4.1 clusters, S4.2 cities, S4.3 national

I draw the annual and monthly LST trends of every cluster, city and the nation. The first cell shows the national chart, C13 and Uyo; the second writes all charts.

**Data:** NB03 series and statistics, checked against package `annual_series`, `monthly_series`, `trend_statistics` and `composite_trend_statistics`.  
**Output:** `supplementary_figures/S4_temporal_trends/`

In [ ]:
begin("S4")
S4_EXAMPLE = {"city": "Uyo", "cluster": 13}

_cp4s = pkg("city_profile")
_A4, _M4 = pkg("annual_series"), pkg("monthly_series")
_T4, _C4 = pkg("trend_statistics"), pkg("composite_trend_statistics")
_slug4 = dict(zip(_cp4s.city, _cp4s.city_slug))
S41 = [(f"S4.1.{i}", cid) for i, cid in enumerate(active_clusters, start=1)]
S42 = [(f"S4.2.{i}", c) for i, c in enumerate(STUDY_CITIES, start=1)]
_ZP4 = {"urban_core": "uc", "bra": "bra", "city_region": "cr", "delta_uc_bra": "suhii"}


def load_city4(city):
    return tp.load_city(_slug4[city], read_source(RESULTS_03 / _slug4[city]))


def load_cluster4(cid):
    return tp.load_cluster(cid, read_source(STATS03))


def _trend_diff4(ent, pk):
    """Largest Sen-slope and MK-p differences between an entity's NB03 annual stats and the package."""
    ds = dp = 0.0; n = 0; miss = 0
    for _, r in pd.concat([ent["ann_stats"], ent["suh_ann_stats"]], ignore_index=True).iterrows():
        q = pk[(pk.zone == _ZP4[r.zone]) & (pk.period == r.period)]
        if q.empty:
            miss += 1; continue
        ds = max(ds, abs(float(r.sen_slope_per_decade) - float(q.iloc[0].sen_slope_c_per_decade)))
        dp = max(dp, abs(float(r.mk_p) - float(q.iloc[0].mk_p_value))); n += 1
    return ds, dp, n, miss


# ── checks: the series drawn and the annual trends equal the package ──
_nat4 = tp.load_national([(_slug4[c], read_source(RESULTS_03 / _slug4[c])) for c in STUDY_CITIES])
_res4 = {"annual": [0.0, 0, 0, 0], "monthly": [0.0, 0, 0, 0]}
_tr4 = [0.0, 0.0, 0, 0]
_ents4 = ([(load_city4(c), _A4[_A4.city == c], _M4[_M4.city == c], _T4[_T4.city == c]) for c in STUDY_CITIES]
          + [(load_cluster4(cid), _A4[_A4.cluster25.astype(int) == cid], _M4[_M4.cluster25.astype(int) == cid],
              _C4[(_C4.composite_type == "cluster") & (_C4.composite_key == f"C{cid}")]) for cid in active_clusters]
          + [(_nat4, _A4, _M4, _C4[_C4.composite_type == "national"])])
for _e, _pa, _pm, _pt in _ents4:
    for _kind, _P in (("annual", _pa), ("monthly", _pm)):
        _df, _ddf, _, _ = tp.panel_inputs(_e, _kind)
        _r = _series_diff(_df, _ddf, _P, _kind)
        _res4[_kind] = [max(_res4[_kind][0], _r[0])] + [a + b for a, b in zip(_res4[_kind][1:], _r[1:])]
    _r = _trend_diff4(_e, _pt)
    _tr4 = [max(_tr4[0], _r[0]), max(_tr4[1], _r[1]), _tr4[2] + _r[2], _tr4[3] + _r[3]]
for _kind in ("annual", "monthly"):
    _w, _n, _g, _u = _res4[_kind]
    check(f"S4 - the {_kind} series drawn equal the package (56 cities, 16 clusters, national)",
          _w <= 1e-9 and _g == 0,
          f"{_n} values (UC, BRA, CR{', CR P5/P95' if _kind == 'annual' else ''}, SUHII), worst difference "
          f"{_w:.1e} °C, {_g} one-sided gaps, {_u} SUHII months without a UC or BRA value in either")
check("S4 - the annual trends (Sen slope, MK p) equal trend_statistics and composite_trend_statistics",
      _tr4[0] <= 1e-9 and _tr4[1] <= 1e-9 and _tr4[3] == 0,
      f"{_tr4[2]} zone-period trends, worst slope difference {_tr4[0]:.1e} °C/decade, worst p {_tr4[1]:.1e}")


# ── drawing: NB03's chart and titles (clusters written "C13") ──
S4_DIR = SUP_FIG / "S4_temporal_trends"
S41_DIR, S42_DIR = S4_DIR / "S4.1_cluster_trends", S4_DIR / "S4.2_city_trends"
for _d in (S4_DIR, S41_DIR, S42_DIR):
    _d.mkdir(exist_ok=True)


def _cluster_title4(cid):
    n = int((_cp4s.cluster25.astype(int) == cid).sum())
    return f"C{cid} — {CL.loc[cid, 'climate']} / {CL.loc[cid, 'landcover']} ({n} {'city' if n == 1 else 'cities'})"


def plot_trend4(ent, title, save_noext):
    fig = viz.create_temporal_chart(city=title, df_ann=ent["ann"], df_mon=ent["mon"],
                                    df_del_ann=ent["suh_ann"], df_del_mon=ent["suh_mon"],
                                    ann_stats=ent["ann_stats"], mon_stats=ent["mon_stats"],
                                    del_ann_stats=ent["suh_ann_stats"], del_mon_stats=ent["suh_mon_stats"],
                                    save_path=None, dpi=300)
    p = savefig(fig, save_noext, keep_size=True)
    plt.close(fig); return p


def s41_path(tag, cid):
    return S41_DIR / f"Supp_Fig_{tag}_C{cid}"


def s42_path(tag, city):
    return S42_DIR / f"Supp_Fig_{tag}_{_slug4[city]}"


# ── S4.3 national, one cluster and one city, shown here ──
_ex = plot_trend4(_nat4, f"Nigerian Cities across {len(STUDY_CITIES)} City-Regions",
                  S4_DIR / "Supp_Fig_S4.3_national")[0]
show_file([_ex])
_t, _cid = next((t, c) for t, c in S41 if c == S4_EXAMPLE["cluster"])
_ex = plot_trend4(load_cluster4(_cid), _cluster_title4(_cid), s41_path(_t, _cid))[0]
show_file([_ex])
_t, _c = next((t, c) for t, c in S42 if c == S4_EXAMPLE["city"])
_ex = plot_trend4(load_city4(_c), _c, s42_path(_t, _c))[0]
show_file([_ex])

In [ ]:
begin("S4")
# ── S4 · every chart of S4.1 (16 clusters) and S4.2 (56 cities) to the folder, none shown; skipped when current ──
_stamp4 = code_stamp(
    plot_trend4, load_city4, load_cluster4, _cluster_title4,
    viz.create_temporal_chart, tp.load_city, tp.load_cluster, inputs=[RESULTS_03])
_files4 = [Path(f"{s41_path(t, c)}.pdf") for t, c in S41] + [Path(f"{s42_path(t, c)}.pdf") for t, c in S42]
if set_is_current(S4_DIR, _stamp4, _files4):
    print("S4: the set is current (same drawing code and inputs), not redrawn; REBUILD = True redraws")
else:
    for _t, _cid in S41:
        plot_trend4(load_cluster4(_cid), _cluster_title4(_cid), s41_path(_t, _cid))
    for _t, _c in S42:
        plot_trend4(load_city4(_c), _c, s42_path(_t, _c))
    mark_current(S4_DIR, _stamp4)
    print(f"S4.1: {len(S41)} cluster charts; S4.2: {len(S42)} city charts; in {S4_DIR.relative_to(OUT)}")
pd.DataFrame([{"supp_fig": t, "cluster": f"C{c}", "title": _cluster_title4(c)} for t, c in S41]).to_csv(
    S41_DIR / "_index_S4.1_cluster_trends.csv", index=False)
pd.DataFrame([{"supp_fig": t, "city": c, "cluster": f"C{int(_cp4s.set_index('city').loc[c, 'cluster25'])}"}
              for t, c in S42]).to_csv(S42_DIR / "_index_S4.2_city_trends.csv", index=False)

<a id="nb07-s5"></a>

#### S5 · Daily series and persistent heat events: S5.0 panel, S5.1 cities, S5.2 clusters, S5.3 national

I draw the daily MODIS and ERA5-Land series with the persistent heat events. The first cell shows the three-level panel, the national chart, Uyo and C13; the second writes all charts.

**Data:** NB04 daily series and cluster composites, checked against package `heat_event_catalogue`.  
**Output:** `supplementary_figures/S5_daily_phe/`

In [ ]:
begin("S5")
import matplotlib.dates as _mdates
from matplotlib.gridspec import GridSpecFromSubplotSpec as _GSS
from daily_analysis_utils import load_daily_csv, run_city_event_pipeline

S5_EXAMPLE = {"city": "Uyo", "cluster": 13}

RAW04 = RESULTS_04 / "raw_extractions"; CLC04 = RESULTS_04 / "cluster_composites"
_YR = STUDY_YEAR
_TIER_C = {"EXCELLENT": "#1a9850", "GOOD": "#f77f00", "FAIR": "#999999", "ACCEPTABLE": "#999999"}

def _period_df(comp, period):
    col = f"{period}_lst_median"
    if comp is None or col not in comp.columns:
        return None
    d = comp[["date", col]].rename(columns={col: "lst_median"}).copy()
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    return d.dropna(subset=["date", "lst_median"]).sort_values("date").reset_index(drop=True)

def _audits(mdf, edf, name):
    return (run_city_event_pipeline(_period_df(mdf, "day"),   _period_df(edf, "day"),   name, "day"),
            run_city_event_pipeline(_period_df(mdf, "night"), _period_df(edf, "night"), name, "night"))

def _national_daily(ds):
    frames = []
    for p in RAW04.glob(f"*_{ds}_UC_daily_{_YR}.csv"):
        d = pd.read_csv(read_source(p)); d["date"] = pd.to_datetime(d["date"], errors="coerce"); frames.append(d)
    allc = pd.concat(frames)
    num = [c for c in allc.columns if c.startswith(("day_lst", "night_lst"))]
    return allc.groupby("date", as_index=False)[num].mean()

def _daily_panel(ax, df, ds_label, ad, an, ds_type, level_lbl=None, fixed_ylim=None):
    DAY_C, DAY_F = "#d73027", "#fdae61"; NGT_C, NGT_F = "#1C1C1C", "#9a9a9a"
    SEL_D, SEL_N = "#a50026", "#000000"
    if df is None:
        ax.text(0.5, 0.5, f"No {ds_label}", transform=ax.transAxes, ha="center", va="center", color="grey"); return
    df = df.copy(); df["date"] = pd.to_datetime(df["date"]); df = df.sort_values("date")
    if {"day_lst_min", "day_lst_max"} <= set(df.columns):
        ax.fill_between(df["date"], df["day_lst_min"], df["day_lst_max"], color=DAY_F, alpha=.35, lw=0, label="Day range")
    ax.plot(df["date"], df["day_lst_median"], color=DAY_C, lw=1.0, label="Day median", zorder=3)
    if {"night_lst_min", "night_lst_max"} <= set(df.columns):
        ax.fill_between(df["date"], df["night_lst_min"], df["night_lst_max"], color=NGT_F, alpha=.45, lw=0, label="Night range")
    ax.plot(df["date"], df["night_lst_median"], color=NGT_C, lw=1.0, label="Night median", zorder=3)
    key = f"{ds_type}_p90_celsius"
    td = ad.get("thresholds", {}).get(key); tn = an.get("thresholds", {}).get(key)
    if td is not None:
        ax.axhline(td, color=DAY_C, ls="--", lw=.9, alpha=.7, label=f"Day P90 {td:.1f}°", zorder=4)
        h = df[df["day_lst_median"] >= td]; ax.scatter(h["date"], h["day_lst_median"], color=DAY_C, s=9, zorder=5, alpha=.6)
    if tn is not None:
        ax.axhline(tn, color=NGT_C, ls="--", lw=.9, alpha=.7, label=f"Night P90 {tn:.1f}°", zorder=4)
        h = df[df["night_lst_median"] >= tn]; ax.scatter(h["date"], h["night_lst_median"], color=NGT_C, s=9, zorder=5, alpha=.6)
    seen = set()
    for e in ad.get(f"{ds_type}_persistent_events", []) + an.get(f"{ds_type}_persistent_events", []):
        k = (e.get("start_date"), e.get("end_date"))
        if k in seen: continue
        seen.add(k)
        try:
            ax.axvspan(pd.to_datetime(e["start_date"]), pd.to_datetime(e["end_date"]) + pd.Timedelta(days=1),
                       color="grey", alpha=.12, lw=0, zorder=1)
        except Exception: pass
    # ── fix the y-scale FIRST so the selection bracket sits correctly ──
    if fixed_ylim is not None:
        ax.set_ylim(*fixed_ylim)
    else:
        lo = float(np.nanmin(df["night_lst_min"] if "night_lst_min" in df else df["night_lst_median"]))
        hi = float(np.nanmax(df["day_lst_max"] if "day_lst_max" in df else df["day_lst_median"]))
        ax.set_ylim(lo - 2, hi + 2)
    y0, y1 = ax.get_ylim(); gap = (y1 - y0) * .09
    def _br(sd, ed, col, above, tier=None):
        """Draw the selected-event bracket (⊢——⊣) and annotate its tier below it."""
        try: s, e = pd.to_datetime(sd), pd.to_datetime(ed)
        except Exception: return
        bh = (y1 - y0) * .025; b = above - gap
        ax.plot([s, e], [b, b], color=col, lw=2.0, zorder=8)
        ax.plot([s, s], [b - bh, b + bh], color=col, lw=2.0, zorder=9); ax.plot([e, e], [b - bh, b + bh], color=col, lw=2.0, zorder=9)
        if tier:
            ax.annotate(tier, xy=(s + (e - s) / 2, b - bh), xytext=(0, -1.5),
                        textcoords="offset points", ha="center", va="top",
                        fontsize=4.8, fontweight="bold", color=_TIER_C.get(tier, col), zorder=10)
    sd, sn = ad.get("selected_event"), an.get("selected_event")
    if sd and td is not None: _br(sd["start_date"], sd["end_date"], SEL_D, td, sd.get("tier", "FAIR"))
    if sn and tn is not None: _br(sn["start_date"], sn["end_date"], SEL_N, tn, sn.get("tier", "FAIR"))
    ax.set_ylabel((f"{level_lbl}\n" if level_lbl else "") + f"{ds_label} (°C)", fontsize=6.5)
    ax.tick_params(labelsize=5.5); ax.grid(axis="y", ls=":", lw=.4, alpha=.5)
    # legend — no per-tier entries (tier is annotated under each selection bracket)
    hnd, lab = ax.get_legend_handles_labels()
    ax.legend(hnd, lab, loc="upper right", fontsize=5.0, ncol=3, framealpha=.85,
              handlelength=1.1, columnspacing=.7, borderaxespad=.2)

# ── S5.1 cities and S5.2 clusters: NB04's chart and event pipeline (NB04 cells 27, 37, 39) ──
_PCT5, _RUN5 = config.DAILY_LST_CONFIG["hot_percentile"], config.DAILY_LST_CONFIG["persistent_threshold"]
_cp5, _H5 = pkg("city_profile"), pkg("heat_event_catalogue")
_cl5 = _cp5.groupby(_cp5.cluster25.astype(int)).city.apply(sorted)
S51 = [(f"S5.1.{i}", c) for i, c in enumerate(STUDY_CITIES, start=1)]
S52 = [(f"S5.2.{i}", cid) for i, cid in enumerate(active_clusters, start=1)]
S5_DIR = SUP_FIG / "S5_daily_phe"
S51_DIR, S52_DIR = S5_DIR / "S5.1_city_daily", S5_DIR / "S5.2_cluster_daily"
for _d in (S5_DIR, S51_DIR, S52_DIR):
    _d.mkdir(exist_ok=True)


def raw5(city, ds):
    """A city's daily urban-core series (NB04 names the files by city name)."""
    return read_source(RAW04 / f"{city}_{ds}_UC_daily_{_YR}.csv")


def comp5(cid, ds):
    return read_source(CLC04 / f"Cluster{cid:02d}_{ds}_composite_{_YR}.csv")


def city_audits5(city):
    """NB04 cell 27: the event pipeline on each period's daily series."""
    return tuple(run_city_event_pipeline(load_daily_csv(raw5(city, "MODIS"), per),
                                         load_daily_csv(raw5(city, "ERA5"), per),
                                         city, per, percentile=_PCT5, min_consecutive=_RUN5)
                 for per in ("day", "night"))


def plot_city5(city, save_noext):
    ad, an = city_audits5(city)
    fig = viz.create_nb04_event_plot(modis_df=pd.read_csv(raw5(city, "MODIS")),
                                     era5_df=pd.read_csv(raw5(city, "ERA5")), city_name=city,
                                     study_year=_YR, audit_day=ad, audit_night=an,
                                     output_path=None, show_inline=False)
    p = savefig(fig, save_noext, keep_size=True)
    plt.close(fig); return p


def plot_cluster5(cid, save_noext):
    m, e = pd.read_csv(comp5(cid, "MODIS")), pd.read_csv(comp5(cid, "ERA5"))
    ad, an = _audits(m, e, f"C{cid}")
    fig = viz.create_nb04_event_plot(modis_df=m, era5_df=e, city_name=f"C{cid} Composite",
                                     study_year=_YR, audit_day=ad, audit_night=an,
                                     output_path=None, show_inline=False)
    p = savefig(fig, save_noext, keep_size=True)
    plt.close(fig); return p


def s51_path(tag, city):
    return S51_DIR / f"Supp_Fig_{tag}_{city.replace(' ', '_')}"


def s52_path(tag, cid):
    return S52_DIR / f"Supp_Fig_{tag}_C{cid}"


def _events5(a):
    return sorted((str(pd.Timestamp(x["start_date"]).date()), str(pd.Timestamp(x["end_date"]).date()),
                   x.get("tier")) for x in a.get("era5_persistent_events", []))


def _sel5(a):
    s = a.get("selected_event")
    return ((str(pd.Timestamp(s["start_date"]).date()), str(pd.Timestamp(s["end_date"]).date()),
             s.get("tier")) if s else None)


def _vs_catalogue5(city, per, a):
    """The audit's events, selected event and ERA5-Land P90 against heat_event_catalogue."""
    q = _H5[(_H5.city == city) & (_H5.period == per)]
    qs = q[q.is_selected.astype(bool)]
    want_sel = ((str(qs.start_date.iloc[0])[:10], str(qs.end_date.iloc[0])[:10], qs.tier.iloc[0])
                if len(qs) else None)
    p90 = a.get("thresholds", {}).get("era5_p90_celsius")
    ok_p90 = q.empty or (p90 is not None
                         and round(float(p90), 2) == round(float(q.era5_p90_threshold_c.iloc[0]), 2))
    ok = (_events5(a) == sorted(zip(q.start_date.astype(str).str[:10], q.end_date.astype(str).str[:10], q.tier))
          and _sel5(a) == want_sel and ok_p90)
    return ok, len(q), want_sel is not None


# ── checks ──
_bad5, _n5, _s5 = [], 0, 0
for _c in STUDY_CITIES:
    for _per, _a in zip(("day", "night"), city_audits5(_c)):
        _ok, _k, _sel = _vs_catalogue5(_c, _per, _a)
        _n5 += _k; _s5 += _sel
        if not _ok:
            _bad5.append(f"{_c} {_per}")
check("S5.1 - every city's events, selected event and ERA5-Land P90 equal heat_event_catalogue",
      not _bad5, ", ".join(_bad5[:6]) or f"{len(STUDY_CITIES)} cities: {_n5} events, {_s5} selected")
_w5c, _n5c = 0.0, 0
for _cid in active_clusters:
    for _ds in ("MODIS", "ERA5"):
        _cmp = pd.read_csv(comp5(_cid, _ds)); _cmp["date"] = pd.to_datetime(_cmp["date"])
        _fr = []
        for _c in _cl5[_cid]:
            _d = pd.read_csv(raw5(_c, _ds)); _d["date"] = pd.to_datetime(_d["date"], errors="coerce"); _fr.append(_d)
        _num = [c for c in _cmp.columns if c.startswith(("day_lst", "night_lst"))]
        _dd = (_cmp.set_index("date")[_num]
               - pd.concat(_fr).groupby("date")[_num].mean().reindex(_cmp.date)).abs()
        _w5c = max(_w5c, float(np.nanmax(_dd.values))); _n5c += _dd.size
check("S5.2 - each NB04 cluster composite is the mean of its member cities (city_profile)",
      _w5c <= 1e-9 and len(_cl5) == len(active_clusters),
      f"{len(_cl5)} clusters, {_n5c} values, worst difference {_w5c:.1e} °C")


# ── S5.0 · the three-level panel: Lagos, C13, national ──
_LG_M = pd.read_csv(read_source(RAW04 / f"Lagos_MODIS_UC_daily_{_YR}.csv")); _LG_E = pd.read_csv(read_source(RAW04 / f"Lagos_ERA5_UC_daily_{_YR}.csv"))
_C13_M = pd.read_csv(read_source(CLC04 / f"Cluster13_MODIS_composite_{_YR}.csv")); _C13_E = pd.read_csv(read_source(CLC04 / f"Cluster13_ERA5_composite_{_YR}.csv"))
_NAT_M = _national_daily("MODIS"); _NAT_E = _national_daily("ERA5")
_NAT_AUD = _audits(_NAT_M, _NAT_E, "National")
_LEVELS = [("Lagos (city)", _LG_M, _LG_E, city_audits5("Lagos")),
           ("C13", _C13_M, _C13_E, _audits(_C13_M, _C13_E, "C13")),
           ("National", _NAT_M, _NAT_E, _NAT_AUD)]

# ── uniform temperature scale across ALL six sub-panels ──
def _series_range(d):
    lo = float(np.nanmin(d["night_lst_min"] if "night_lst_min" in d else d["night_lst_median"]))
    hi = float(np.nanmax(d["day_lst_max"] if "day_lst_max" in d else d["day_lst_median"]))
    return lo, hi
_ranges = [_series_range(d) for d in (_LG_M, _LG_E, _C13_M, _C13_E, _NAT_M, _NAT_E) if d is not None]
_GYLIM = (min(r[0] for r in _ranges) - 2, max(r[1] for r in _ranges) + 2)

fig10 = plt.figure(figsize=(8.27, 9.6))
_outer = fig10.add_gridspec(3, 1, hspace=0.14, top=0.985, bottom=0.05, left=0.10, right=0.97)
for _i, (_lbl, _m, _e, (_ad, _an)) in enumerate(_LEVELS):
    _in = _GSS(2, 1, subplot_spec=_outer[_i], hspace=0.06)
    _axm = fig10.add_subplot(_in[0]); _axe = fig10.add_subplot(_in[1], sharex=_axm)
    _daily_panel(_axm, _m, "MODIS", _ad, _an, "modis", level_lbl=_lbl, fixed_ylim=_GYLIM)
    _daily_panel(_axe, _e, "ERA5-Land", _ad, _an, "era5", fixed_ylim=_GYLIM)
    _axm.tick_params(labelbottom=False)
    _axe.xaxis.set_major_locator(_mdates.MonthLocator()); _axe.xaxis.set_major_formatter(_mdates.DateFormatter("%b"))
    _axe.tick_params(axis="x", labelsize=5.5)
_lg5 = _LEVELS[0][3]
check("S5.0 - the panel's Lagos events equal heat_event_catalogue; the national level is the mean of "
      "the 56 cities",
      all(_vs_catalogue5("Lagos", _per, _a)[0] for _per, _a in zip(("day", "night"), _lg5))
      and len(list(RAW04.glob(f"*_MODIS_UC_daily_{_YR}.csv"))) == len(STUDY_CITIES),
      f"Lagos selected: day {_sel5(_lg5[0])[2]}, night {_sel5(_lg5[1])[2]}; "
      f"national from {len(STUDY_CITIES)} cities")
_ex = savefig(fig10, S5_DIR / "Supp_Fig_S5.0_daily_phe_panel", keep_size=True)[0]
plt.close(fig10)
show_file([_ex])

# ── S5.3 national: NB04's chart on the national composite of the panel ──
def plot_national5(save_noext):
    fig = viz.create_nb04_event_plot(modis_df=_NAT_M, era5_df=_NAT_E,
                                     city_name=f"Nigerian Cities across {len(STUDY_CITIES)} City-Regions",
                                     study_year=_YR, audit_day=_NAT_AUD[0], audit_night=_NAT_AUD[1],
                                     output_path=None, show_inline=False)
    p = savefig(fig, save_noext, keep_size=True)
    plt.close(fig); return p


show_file([plot_national5(S5_DIR / "Supp_Fig_S5.3_national")[0]])

# ── one city and one cluster, shown here ──
_t, _c = next((t, c) for t, c in S51 if c == S5_EXAMPLE["city"])
_ex = plot_city5(_c, s51_path(_t, _c))[0]
show_file([_ex])
_t, _cid = next((t, c) for t, c in S52 if c == S5_EXAMPLE["cluster"])
_ex = plot_cluster5(_cid, s52_path(_t, _cid))[0]
show_file([_ex])

In [ ]:
begin("S5")
# ── S5 · every chart of S5.1 (56 cities) and S5.2 (16 clusters) to the folder, none shown; skipped when current ──
_stamp5 = code_stamp(
    plot_city5, plot_cluster5, city_audits5, raw5, comp5, _audits, _period_df,
    viz.create_nb04_event_plot, run_city_event_pipeline, load_daily_csv, inputs=[RAW04, CLC04])
_files5 = [Path(f"{s51_path(t, c)}.pdf") for t, c in S51] + [Path(f"{s52_path(t, c)}.pdf") for t, c in S52]
if set_is_current(S5_DIR, _stamp5, _files5):
    print("S5: the set is current (same drawing code and inputs), not redrawn; REBUILD = True redraws")
else:
    for _t, _c in S51:
        plot_city5(_c, s51_path(_t, _c))
    for _t, _cid in S52:
        plot_cluster5(_cid, s52_path(_t, _cid))
    mark_current(S5_DIR, _stamp5)
    print(f"S5.1: {len(S51)} city charts; S5.2: {len(S52)} cluster charts; in {S5_DIR.relative_to(OUT)}")
pd.DataFrame([{"supp_fig": t, "city": c, "cluster": f"C{int(_cp5.set_index('city').loc[c, 'cluster25'])}"}
              for t, c in S51]).to_csv(S51_DIR / "_index_S5.1_city_daily.csv", index=False)
pd.DataFrame([{"supp_fig": t, "cluster": f"C{c}", "cities": "; ".join(_cl5[c])}
              for t, c in S52]).to_csv(S52_DIR / "_index_S5.2_cluster_daily.csv", index=False)

<a id="nb07-s6"></a>

#### S6 · Event frequency by city and cluster, day and night

I count each city's persistent heat events, grouped by cluster, day above night, coloured by the tier of the selected event.

**Data:** package `heat_event_catalogue`, `city_profile`.  
**Output:** `supplementary_figures/S6_event_frequency/`

In [ ]:
begin("S6")
_H6, _cp6 = pkg("heat_event_catalogue"), pkg("city_profile")
_lookup6 = dict(zip(_cp6.city, _cp6.cluster25.astype(int)))


def _records6(period):
    """Each city's ERA5-Land persistent events and selected event, in the form of NB04's audits."""
    recs = {}
    for city in _lookup6:
        q = _H6[(_H6.city == city) & (_H6.period == period)]
        sel = q[q.is_selected.astype(bool)]
        recs[city] = {"era5_persistent_events": q.to_dict("records"),
                      "selected_event": ({"tier": sel.tier.iloc[0], "duration": int(sel.duration_days.iloc[0])}
                                         if len(sel) else None)}
    return recs


_rec6 = {per: _records6(per) for per in ("day", "night")}
check("S6 - every bar is heat_event_catalogue's: the city's events, its selected event's tier and duration",
      len(_lookup6) == 56 and set(_H6.city) <= set(_lookup6)
      and sum(len(r["era5_persistent_events"]) for d in _rec6.values() for r in d.values()) == len(_H6),
      "; ".join(f"{per}: {sum(len(r['era5_persistent_events']) for r in d.values())} events, "
                f"{sum(r['selected_event'] is not None for r in d.values())} selected" for per, d in _rec6.items()))

# ── one figure: day above night, one legend below, one title ──
S6_DIR = SUP_FIG / "S6_event_frequency"
S6_DIR.mkdir(exist_ok=True)
fig6, _axs6 = plt.subplots(2, 1, figsize=(max(8, len(_lookup6) * 0.45), 10.5), sharex=True, sharey=True)
for _ax, _per, _let in zip(_axs6, ("day", "night"), ("(a)", "(b)")):
    viz.create_cluster_event_frequency_chart(cluster_event_summary=None, city_event_records=_rec6[_per],
                                             cluster_lookup=_lookup6, period=_per, cluster_fmt="C{}",
                                             ax=_ax, show_legend=(_per == "night"), title="")
    _ax.set_title(f"{_let} {_per.capitalize()}", loc="left", fontsize=10, fontweight="bold")
_axs6[0].set_xlabel("")
_axs6[0].tick_params(labelbottom=False)
_leg6 = _axs6[1].get_legend()                    # NB04's tier legend, moved below both panels
fig6.legend(handles=_leg6.legend_handles, title=_leg6.get_title().get_text(), loc="lower center",
            ncol=len(_leg6.legend_handles), fontsize=8, title_fontsize=8, bbox_to_anchor=(0.5, 0.0))
_leg6.remove()
fig6.suptitle("ERA5 Persistent Hot-Event Count per City — Day and Night\n"
              "Bar colour = tier of selected SEVIRI event  |  Label = selected duration (days)", fontsize=10)
fig6.tight_layout(rect=(0, 0.05, 1, 0.97))
_ex = savefig(fig6, S6_DIR / "Supp_Fig_S6_event_frequency", keep_size=True)[0]
plt.close(fig6)
show_file([_ex])

<a id="nb07-s7"></a>

#### S7 · Agreement between thermal products: S7.1 MODIS vs ERA5-Land, S7.2 SEVIRI vs ERA5-Land

I compare the daily MODIS and ERA5-Land values of all cities, and each city's SEVIRI and ERA5-Land means over its selected event.

**Data:** NB04 daily series; package `city_year_summary`.  
**Output:** `supplementary_figures/S7_thermal_product_agreement/`

In [ ]:
begin("S7")
from daily_analysis_utils import load_daily_csv, compute_correlation_metrics

_cy7 = pkg("city_year_summary").set_index("city")
_RAW7 = RESULTS_04 / "raw_extractions"
S7_DIR = SUP_FIG / "S7_thermal_product_agreement"
S7_DIR.mkdir(exist_ok=True)
_MET7 = (("n", "modis_era5land_n"), ("r", "modis_era5land_r"), ("bias", "era5land_minus_modis_bias_c"),
         ("rmse", "modis_era5land_rmse_c"))

# ── S7.1 · MODIS against ERA5-Land, every city's daily pairs pooled ──
_pooled7 = {"day": {"era5": [], "modis": []}, "night": {"era5": [], "modis": []}}
_dev7 = 0.0
for _city in _cy7.index:
    for _p in ("day", "night"):
        _m = load_daily_csv(read_source(_RAW7 / f"{_city}_MODIS_UC_daily_{STUDY_YEAR}.csv"), _p)
        _e = load_daily_csv(read_source(_RAW7 / f"{_city}_ERA5_UC_daily_{STUDY_YEAR}.csv"), _p)
        _mg = pd.merge(_m[["date", "lst_median"]].rename(columns={"lst_median": "modis"}),
                       _e[["date", "lst_median"]].rename(columns={"lst_median": "era5"}),
                       on="date", how="inner").dropna()
        _mc = compute_correlation_metrics(_mg["modis"], _mg["era5"])
        _dev7 = max([_dev7] + [abs(float(_mc[k]) - float(_cy7.loc[_city, f"{_p}_{c}"])) for k, c in _MET7])
        _pooled7[_p]["era5"] += _mg["era5"].tolist()
        _pooled7[_p]["modis"] += _mg["modis"].tolist()
for _p in ("day", "night"):
    _pooled7[_p]["metrics"] = compute_correlation_metrics(pd.Series(_pooled7[_p]["modis"]),
                                                          pd.Series(_pooled7[_p]["era5"]))
check("S7.1 - every city's MODIS / ERA5-Land daily pairs give the package's n, r, bias and RMSE",
      _dev7 <= 1e-9,
      f"{len(_cy7)} cities x 2 periods, worst difference {_dev7:.1e}; pooled: "
      + "; ".join(f"{p} n {_pooled7[p]['metrics']['n']:,}, bias (ERA5-Land - MODIS) "
                  f"{_pooled7[p]['metrics']['bias']:+.3f} °C" for p in ("day", "night")))
_f = viz.create_era5_modis_hexbin(paired_data=_pooled7,
                                  title=f"All Cities Pooled - ERA5-Land vs MODIS ({STUDY_YEAR})",
                                  output_path=None, dpi=config.FIGURE_DPI, show_inline=False,
                                  bias_label="Bias (ERA5-Land − MODIS)")
_ex = savefig(_f, S7_DIR / "Supp_Fig_S7.1_era5land_vs_modis_hexbin", keep_size=True)[0]
plt.close(_f)
show_file([_ex])

# ── S7.2 · SEVIRI event composite against the ERA5-Land event-window mean, one point per city ──
_rows7 = {p: [{"city": c, "era5_mean": float(r[f"{p}_era5land_event_mean_c"]),
               "seviri_mean": float(r[f"{p}_seviri_uc_mean_c"]), "tier": r[f"{p}_selected_event_tier"]}
              for c, r in _cy7.iterrows()
              if pd.notna(r[f"{p}_era5land_event_mean_c"]) and pd.notna(r[f"{p}_seviri_uc_mean_c"])]
          for p in ("day", "night")}


def _r2_7(rows):
    """r² of the OLS fit the chart draws (SEVIRI on ERA5-Land)."""
    x, y = np.array([r["era5_mean"] for r in rows]), np.array([r["seviri_mean"] for r in rows])
    m, b = np.polyfit(x, y, 1)
    return 1 - ((y - (m * x + b)) ** 2).sum() / ((y - y.mean()) ** 2).sum()


check("S7.2 - one point per city from city_year_summary (SEVIRI urban-core mean, ERA5-Land event mean, tier)",
      all(len(v) == len(_cy7) for v in _rows7.values()),
      "; ".join(f"{p}: {len(v)} cities, r² {_r2_7(v):.3f}" for p, v in _rows7.items()))
_f = viz.create_seviri_era5_scatter(day_rows=_rows7["day"], night_rows=_rows7["night"], zone="urban_core",
                                    output_dir=None, dpi=config.FIGURE_DPI, tier_title="Event tier")
_ex = savefig(_f, S7_DIR / "Supp_Fig_S7.2_seviri_vs_era5land_event_scatter", keep_size=True)[0]
plt.close(_f)
show_file([_ex])

<a id="nb07-s8"></a>

#### S8 · Gi* z-score distributions, S8.1–S8.56

I plot each city's Gi* z-score distribution, day and night. The first cell shows Uyo; the second writes all 56.

**Data:** NB05 Gi* grids, checked against package `spots`.  
**Output:** `supplementary_figures/S8_gistar_distributions/`

In [ ]:
begin("S8")
S8_EXAMPLE = "Uyo"
_sp8 = pkg("spots")
S8_DIR = SUP_FIG / "S8_gistar_distributions"
S8_DIR.mkdir(exist_ok=True)
S8 = [(f"S8.{i}", c) for i, c in enumerate(STUDY_CITIES, start=1)]


def _mask8(city, period):
    return RESULTS_05 / city / f"{city}_masks_{period}.npz"


def zgrids8(city):
    """The city's day and night Gi★ z-score grids (NB05's cached masks)."""
    return {per: np.load(read_source(_mask8(city, per)))["z_grid"] for per in ("day", "night")}


def plot_gistar8(city, save_noext):
    fig = viz.create_gistar_distribution(city, zgrids8(city))
    p = savefig(fig, save_noext, keep_size=True)
    plt.close(fig); return p


def s8_path(tag, city):
    return S8_DIR / f"Supp_Fig_{tag}_{city.replace(' ', '_')}"


# ── check: every grid present, and every spot in the package has Gi★ pixels beyond ±1.96 on its side ──
_bad8, _n8 = [], 0
for _c in STUDY_CITIES:
    _g = zgrids8(_c)
    for _per in ("day", "night"):
        for _st, _side in (("hot", _g[_per] > 1.96), ("cold", _g[_per] < -1.96)):
            if ((_sp8.city == _c) & (_sp8.period == _per) & (_sp8.spot_type == _st)).any():
                _n8 += 1
                if not _side.any():
                    _bad8.append(f"{_c} {_per} {_st}")
check("S8 - a day and a night Gi★ grid for each of the 56 cities; every city-period with a spot in the "
      "package has Gi★ pixels beyond ±1.96 on that side",
      len(S8) == 56 and not _bad8,
      ", ".join(_bad8[:6]) or f"{len(S8) * 2} grids; {_n8} city-period-type combinations with spots, all supported")

# ── one city, shown here ──
_t, _c = next((t, c) for t, c in S8 if c == S8_EXAMPLE)
_ex = plot_gistar8(_c, s8_path(_t, _c))[0]
show_file([_ex])

In [ ]:
begin("S8")
# ── S8 · every city's Gi★ distribution, S8.1–S8.56, to the folder, none shown; skipped when current ──
_stamp8 = code_stamp(plot_gistar8, zgrids8, viz.create_gistar_distribution,
                     inputs=[_mask8(c, p) for c in STUDY_CITIES for p in ("day", "night")])
_files8 = [Path(f"{s8_path(t, c)}.pdf") for t, c in S8]
if set_is_current(S8_DIR, _stamp8, _files8):
    print("S8: the set is current (same drawing code and inputs), not redrawn; REBUILD = True redraws")
else:
    for _t, _c in S8:
        plot_gistar8(_c, s8_path(_t, _c))
    mark_current(S8_DIR, _stamp8)
    print(f"S8: {len(S8)} city Gi★ distributions in {S8_DIR.relative_to(OUT)}")
_cl8 = pkg("city_profile").set_index("city").cluster25.astype(int)
pd.DataFrame([{"supp_fig": t, "city": c, "cluster": f"C{_cl8[c]}"} for t, c in S8]).to_csv(
    S8_DIR / "_index_S8_gistar_distributions.csv", index=False)

<a id="nb07-s9"></a>

#### S9 · City spot area, ranked: S9.1 hot spots, S9.2 cold spots

I rank the cities by spot area, with the nested contour levels and the BH-FDR result.

**Data:** package `spots`, `city_profile`.  
**Output:** `supplementary_figures/S9_city_spot_area_ranked/`

In [ ]:
begin("S9")
_sp9 = pkg("spots").rename(columns={"area_l50_km2": "area_km2_50", "area_l75_km2": "area_km2_75",
                                     "area_l90_km2": "area_km2_90"})
_cp9 = pkg("city_profile")
_a9 = _sp9[["area_km2_50", "area_km2_75", "area_km2_90"]].fillna(0)
check("S9 - every bar is the package's spot areas, L90 <= L75 <= L50 in each spot",
      len(_sp9) == 185 and bool(((_a9.area_km2_90 <= _a9.area_km2_75 + 1e-9)
                                 & (_a9.area_km2_75 <= _a9.area_km2_50 + 1e-9)).all()),
      f"{len(_sp9)} spots; cities with spots: "
      + "; ".join(f"{st} {per} {n}" for (st, per), n in _sp9.groupby(["spot_type", "period"]).city.nunique().items()))

S9_DIR = SUP_FIG / "S9_city_spot_area_ranked"
S9_DIR.mkdir(exist_ok=True)
for _no, _st in (("S9.1", "hot"), ("S9.2", "cold")):
    _word = _st.capitalize()
    _f = viz.create_city_spot_area_ranked(
        spots=_sp9, cities=sorted(_cp9.city), city_to_cluster=dict(zip(_cp9.city, _cp9.cluster25.astype(int))),
        cluster_colors=config.CLUSTER25_COLORS, spot_type_filter=_st,
        chart_title=f"{_word} Spot Area per City — Day & Night  (cities with spots only)\n"
                    "* = BH-FDR  |  colour = cluster  |  hatch = contour zone",
        legend_clusters=active_clusters,
        panel_title_fmt="{period} — {spot} Spot Area\n(* = BH-FDR  |  colour = cluster  |  hatch = contour zone)",
        legend_title="Cluster  |  Contour zone")
    _ex = savefig(_f, S9_DIR / f"Supp_Fig_{_no}_city_{_st}_spot_area_ranked", keep_size=True)[0]
    plt.close(_f)
    show_file([_ex])

<a id="nb07-s10"></a>

#### S10 · Population within spots: national maps and half-donuts

I map the residents within hot and cold spots nationally and draw the half-donuts of every cluster, city and the nation. The first cell shows the hot-spot map and the national, C13 and Uyo half-donuts; the second writes the rest.

**Data:** package `city_exposure_by_contour`, `spots`, `city_profile`, `city_zones`; the typology map and the national boundary.  
**Output:** `supplementary_figures/S10_population_within_spots/`

In [ ]:
begin("S10")
S10_EXAMPLE = {"city": "Uyo", "cluster": 13}
_hd10 = pkg("city_exposure_by_contour").rename(columns={
    "area_l50_km2": "area_km2_50", "area_l75_km2": "area_km2_75", "area_l90_km2": "area_km2_90",
    "pop_l50_persons": "population", "pop_l75_persons": "pop_75", "pop_l90_persons": "pop_90"})
_hd10["cluster25"] = _hd10["cluster25"].astype(int)
_bh10 = pkg("spots").groupby(["city", "spot_type", "period"]).bh_significant.any()
_hd10["bh_significant"] = [bool(_bh10.get((c, st, per), False))
                           for c, st, per in zip(_hd10.city, _hd10.spot_type, _hd10.period)]
_cp10 = pkg("city_profile")
_cl10 = dict(zip(_cp10.city, _cp10.cluster25.astype(int)))
_uc10 = pkg("city_zones", geom=True)
_uc10 = _uc10[_uc10.zone == "uc"].to_crs(4326).dissolve("city")
_cent10 = {c: (g.centroid.x, g.centroid.y) for c, g in _uc10.geometry.items()}
_nig10 = gpd.read_file(read_source(ROOT / "data" / "inputs" / "geospatial_features" / "boundary_Nigeria.geojson"))

# ── checks: as Fig. 12 ──
_p10 = _hd10[["population", "pop_75", "pop_90"]].fillna(0)
check("S10 - 154 city rows of population and area, with L90 <= L75 <= L50 in each",
      len(_hd10) == 154 and bool(((_p10.pop_90 <= _p10.pop_75 + 1e-6) & (_p10.pop_75 <= _p10.population + 1e-6)).all()),
      f"{len(_hd10)} rows, {_hd10.city.nunique()} cities, {_hd10.cluster25.nunique()} clusters")
check("S10 - every study city is placed at its urban-core centroid; residents in night-time hot spots (L50)",
      len(_cent10) == 56 and set(_hd10.city) <= set(_cent10)
      and round(_hd10[(_hd10.spot_type == "hot") & (_hd10.period == "night")].population.sum()) == 9_184_401,
      f"{len(_cent10)} cities; hot night "
      f"{round(_hd10[(_hd10.spot_type == 'hot') & (_hd10.period == 'night')].population.sum()):,}")

# ── drawing: NB05's map and half-donut, with "population within" and "surface hot and cold spots" ──
S10_DIR = SUP_FIG / "S10_population_within_spots"
S102_DIR, S103_DIR = S10_DIR / "S10.2_cluster_halfdonuts", S10_DIR / "S10.3_city_halfdonuts"
for _d in (S10_DIR, S102_DIR, S103_DIR):
    _d.mkdir(exist_ok=True)
S10_CL = [(f"S10.2.{i}", c) for i, c in enumerate(sorted(_hd10.cluster25.unique()), start=1)]
S10_CT = [(f"S10.3.{i}", c) for i, c in enumerate(sorted(_hd10.city.unique()), start=1)]
_RING10 = "Left=Hot  Right=Cold  ·  Ring1=Type  Ring2=Day/Night  Ring3=Contour"


def _save10(fig, save_noext):
    p = savefig(fig, save_noext, keep_size=True)
    plt.close(fig); return p


def plot_map10(spot_type, save_noext):
    word = spot_type.capitalize()
    return _save10(viz.create_national_population_map(
        spots=_hd10, centroids=_cent10, cities=sorted(_cp10.city), landscape_gdf=landscape_gdf,
        nigeria_gdf=_nig10, cluster25_colors=config.CLUSTER25_COLORS, spot_type_filter=spot_type,
        title=f"Nigeria Surface {word} Spot Summary — Area & Population within Spots",
        spot_label=f"{word} spot", pop_word="pop within", scale_title="Population\nwithin spots"), save_noext)


def plot_national_hd10(save_noext):
    return _save10(viz.create_population_halfdonut(
        _hd10, f"Nigeria — National Population within Surface Hot & Cold Spots\n{_RING10} zone"), save_noext)


def plot_cluster_hd10(cid, save_noext):
    return _save10(viz.create_population_halfdonut(
        _hd10[_hd10.cluster25 == cid], f"C{cid} — Population within Surface Hot & Cold Spots\n{_RING10}"), save_noext)


def plot_city_hd10(city, save_noext):
    return _save10(viz.create_population_halfdonut(
        _hd10[_hd10.city == city],
        f"City: {city}  |  C{_cl10[city]} — Population within Surface Hot & Cold Spots\n{_RING10}"), save_noext)


def s10_cl_path(tag, cid):
    return S102_DIR / f"Supp_Fig_{tag}_C{cid}_halfdonut"


def s10_ct_path(tag, city):
    return S103_DIR / f"Supp_Fig_{tag}_{city.replace(' ', '_')}_halfdonut"


# ── national (hot map and half-donut), one cluster, one city, shown here ──
_ex = plot_map10("hot", S10_DIR / "Supp_Fig_S10.1.1_national_hot_spot_map")[0]
show_file([_ex])
_ex = plot_national_hd10(S10_DIR / "Supp_Fig_S10.4_national_halfdonut")[0]
show_file([_ex])
_t, _cid = next((t, c) for t, c in S10_CL if c == S10_EXAMPLE["cluster"])
_ex = plot_cluster_hd10(_cid, s10_cl_path(_t, _cid))[0]
show_file([_ex])
_t, _c = next((t, c) for t, c in S10_CT if c == S10_EXAMPLE["city"])
_ex = plot_city_hd10(_c, s10_ct_path(_t, _c))[0]
show_file([_ex])

In [ ]:
begin("S10")
# ── S10 · the cold-spot map, every cluster and every city half-donut to the folder, none shown;
#    skipped when current ──
_stamp10 = code_stamp(
    plot_map10, plot_cluster_hd10, plot_city_hd10, _save10, viz.create_population_halfdonut,
    viz.create_national_population_map, inputs=[config.LANDSCAPE_DATA_PATH])
_files10 = ([S10_DIR / "Supp_Fig_S10.1.2_national_cold_spot_map.pdf"]
            + [Path(f"{s10_cl_path(t, c)}.pdf") for t, c in S10_CL] + [Path(f"{s10_ct_path(t, c)}.pdf") for t, c in S10_CT])
if set_is_current(S10_DIR, _stamp10, _files10):
    print("S10: the set is current (same drawing code and inputs), not redrawn; REBUILD = True redraws")
else:
    plot_map10("cold", S10_DIR / "Supp_Fig_S10.1.2_national_cold_spot_map")
    for _t, _cid in S10_CL:
        plot_cluster_hd10(_cid, s10_cl_path(_t, _cid))
    for _t, _c in S10_CT:
        plot_city_hd10(_c, s10_ct_path(_t, _c))
    mark_current(S10_DIR, _stamp10)
    print(f"S10: the cold-spot map, {len(S10_CL)} cluster and {len(S10_CT)} city half-donuts in "
          f"{S10_DIR.relative_to(OUT)}")
pd.DataFrame([{"supp_fig": t, "cluster": f"C{c}"} for t, c in S10_CL]
             + [{"supp_fig": t, "city": c, "cluster": f"C{_cl10[c]}"} for t, c in S10_CT]).to_csv(
    S10_DIR / "_index_S10_population_within_spots.csv", index=False)

<a id="nb07-s11"></a>

#### S11 · City-level stakeholder perception

I show the answers of the 14 cities with at least two participants and name the seven with one.

**Data:** package `stakeholder_perception`.  
**Output:** `supplementary_figures/S11_city_perception/`

In [ ]:
begin("S11")
import contextlib, io
_pp11 = pkg("stakeholder_perception")
_u11 = pd.DataFrame({"matched_city_y": _pp11.city,
                     "61_temp_increase_10yrs": _pp11.perceived_warming_10yr,
                     "60_diurnal_severity_heat": _pp11.perceived_diurnal_severity,
                     "58_temp_peak_city": _pp11.perceived_peak_period})
_n11 = _pp11.city.value_counts()
_left11 = sorted(_n11[_n11 < 2].index)
check("S11 - the perception cohort (105 participants, 21 cities); the 14 cities with at least 2 shown, "
      "the 7 with one named below the panels",
      len(_pp11) == 105 and _pp11.city.nunique() == 21 and int((_n11 >= 2).sum()) == 14 and len(_left11) == 7,
      f"{len(_pp11)} participants, {_pp11.city.nunique()} cities; left out: {', '.join(_left11)}")

S11_DIR = SUP_FIG / "S11_city_perception"
S11_DIR.mkdir(exist_ok=True)
with contextlib.redirect_stdout(io.StringIO()):          # its "charts created" message is not shown
    _f, _fd = viz.create_perception_chart_with_donuts(
        unique_resp=_u11,
        warming_total=_pp11.perceived_warming_10yr.value_counts(),
        diurnal_total=_pp11.perceived_diurnal_severity.value_counts(),
        peak_total=_pp11.perceived_peak_period.value_counts(),
        min_participants=2, figsize=(20, 5.5), legend_scale=1.1, panel_letters=("(a)", "(b)", "(c)"))
plt.close(_fd)
_f.text(0.01, 0.0, f"Cities with one participant, not shown: {', '.join(_left11)}. "
                   f"The Overall donuts count all {len(_pp11)} participants.",
        ha="left", va="top", fontsize=9, color="#333333")
_ex = savefig(_f, S11_DIR / "Supp_Fig_S11_city_perception", keep_size=True)[0]
plt.close(_f)
show_file([_ex])

<a id="nb07-supp-tables"></a>

## 3 · Supplementary tables

[↑ Contents](#nb07-contents)

<a id="nb07-supp-table-s1"></a>

#### Supp. Table S1 · The 24 physiographic landscape typologies

I list each typology's climate, geology and elevation class, its predominant land cover and its colour.

**Data:** the typology map; `cluster_labels.csv`.  
**Output:** `supplementary_tables/Supp_Table_S1_typologies` (.csv, .xlsx)

In [ ]:
begin("Supp. Table S1")
def _predom(col):
    a = (landscape_gdf.groupby(["Cluster25", col])["area_sqkm"].sum().reset_index()
         .sort_values(["Cluster25", "area_sqkm"], ascending=[True, False]))
    return a.groupby("Cluster25").first()[col]

_nat = pd.DataFrame({
    "Climate Class":   _predom("Climate_Class"),
    "Geology Class":   _predom("Geology_Class"),
    "Elevation Class": _predom("Elevation_Class"),
    "Predominant Land Cover": _predom("Landcover_Class"),
}).reset_index()
_nat["Cluster25"] = _nat["Cluster25"].astype(int)
_nat = _nat.sort_values("Cluster25").reset_index(drop=True)
_nat["Cluster"] = _nat["Cluster25"].map(lambda c: f"C{c}")
_nat["Colour"]  = _nat["Cluster25"].map(lambda c: config.CLUSTER25_COLORS.get(int(c), "#CCCCCC"))
_nat.insert(0, "S/No", range(1, len(_nat) + 1))

_ax1 = {"Climate Class": "climate", "Geology Class": "geology", "Elevation Class": "elevation"}
_same1 = [all(r[k] == CL.loc[r.Cluster25, v] for k, v in _ax1.items()) for _, r in _nat.iterrows()]
check("Supp. Table S1 - the 24 typologies; each one's area-dominant climate, geology and elevation "
      "equal the approved label file",
      len(_nat) == len(CL) == 24 and all(_same1)
      and all(int(c) in config.CLUSTER25_COLORS for c in _nat.Cluster25),
      f"{sum(_same1)} of {len(_nat)} agree; codes {_nat.Cluster.iloc[0]}, {_nat.Cluster.iloc[1]} … "
      f"{_nat.Cluster.iloc[-1]}")
# Reported, not adjusted: cluster_labels.csv's "landcover" is the land cover of each typology's first
# polygon in file order, not the predominant one.
_lc1 = landscape_gdf.groupby(["Cluster25", "Landcover_Class"]).area_sqkm.sum()
_lc1 = 100 * _lc1 / _lc1.groupby(level=0).transform("sum")
_diff1 = [(r.Cluster, r["Predominant Land Cover"], _lc1[(r.Cluster25, r["Predominant Land Cover"])],
           CL.loc[r.Cluster25, "landcover"], _lc1.get((r.Cluster25, CL.loc[r.Cluster25, "landcover"]), 0.0))
          for _, r in _nat.iterrows() if r["Predominant Land Cover"] != CL.loc[r.Cluster25, "landcover"]]
print(f"Note: the predominant land cover differs from cluster_labels.csv's in {len(_diff1)} of {len(_nat)} "
      "typologies (share of the typology's area):" + "".join(
          f"\n  {c} {x} {px:.0f} % (label file: {y} {py:.0f} %)" for c, x, px, y, py in _diff1))
FOOT_S1 = ("Each class is the one covering the largest area of the typology. A typology is defined by "
           "its climate, geology and elevation; the predominant land cover describes it and is not one "
           "of its axes.")

# CSV carries the hex under "Colour"; the inline preview and Excel render it as a swatch.
_cols1 = ["S/No", "Cluster", "Climate Class", "Geology Class", "Elevation Class", "Predominant Land Cover"]
appx = _nat[_cols1 + ["Colour"]]

_styler = (_nat[["S/No", "Colour"] + _cols1[1:]]
           .style
           .set_caption("Supplementary Table S1 — Physiographic landscape typologies in Nigeria and "
                        "their internal makeup")
           .set_table_styles([
               {"selector": "th", "props": [("background-color", "#404040"),
                                            ("color", "white"), ("font-weight", "bold")]},
               {"selector": "caption", "props": [("font-weight", "bold"),
                                                 ("font-size", "13px"), ("text-align", "left")]},
           ])
           .set_properties(**{"font-size": "11px"})
           .apply(lambda s: [f"background-color:{v}; color:{v}; border:1px solid #999" for v in s],
                  subset=["Colour"], axis=0)
           .hide(axis="index"))
display(_styler)

# ── (a) CSV (Colour column carries the hex) ──
csv_p = SUP_TAB / "Supp_Table_S1_typologies.csv"
appx.to_csv(csv_p, index=False)

# ── (b) Excel — Colour rendered as a filled swatch (no text) ──
xlsx_p = SUP_TAB / "Supp_Table_S1_typologies.xlsx"
wb = xlsxwriter.Workbook(str(xlsx_p))
ws = wb.add_worksheet("Supp. Table S1")
hdr  = wb.add_format({"bold": True, "bg_color": "#404040", "font_color": "white",
                      "border": 1, "align": "center", "valign": "vcenter", "text_wrap": True})
cell = wb.add_format({"border": 1, "valign": "vcenter", "text_wrap": True})
foot = wb.add_format({"italic": True, "font_size": 8})
cols = ["S/No", "Colour"] + _cols1[1:]
for j, c in enumerate(cols):
    ws.write(0, j, c, hdr)
for i, (_, r) in enumerate(_nat.iterrows(), start=1):
    ws.write(i, 0, int(r["S/No"]), cell)
    ws.write(i, 1, "", wb.add_format({"bg_color": str(r["Colour"]), "border": 1}))
    for j, c in enumerate(_cols1[1:], start=2):
        ws.write(i, j, r[c], cell)
ws.write(len(_nat) + 2, 0, FOOT_S1, foot)
ws.set_column(0, 0, 6); ws.set_column(1, 1, 9); ws.set_column(2, 2, 9); ws.set_column(3, 6, 30)
ws.set_row(0, 26)
wb.close()
print(f"Saved: {csv_p.name}, {xlsx_p.name} in {SUP_TAB.relative_to(ROOT)}")
print(FOOT_S1)

<a id="nb07-supp-table-s2"></a>

#### Supp. Table S2 · Normality tests (Shapiro–Wilk)

I report the normality tests of SUHII and L50 area for each spot type and period.

**Data:** NB06 `normality_diagnostics.csv`, checked against package `spots`.  
**Output:** `supplementary_tables/Supp_Table_S2_normality_tests` (.csv, .xlsx)

In [ ]:
begin("Supp. Table S2")
_nd2 = pd.read_csv(read_source(RESULTS_06 / "normality_diagnostics.csv"))
_nc2 = pkg("spots").groupby(["spot_type", "period"]).city.nunique()
check("Supp. Table S2 - each test's N is the number of cities with a spot of that type and period",
      len(_nd2) == 8 and all(int(r.N) == _nc2[(r.Spot, r.Period)] for r in _nd2.itertuples()),
      "; ".join(f"{t} {p} {n}" for (t, p), n in _nc2.items()))
check("Supp. Table S2 - normal means Shapiro-Wilk p >= 0.05, as NB06 decides it",
      (_nd2.Normal_at_alpha == (_nd2.Shapiro_p >= 0.05)).all(),
      "normal: " + ", ".join(f"{o.split(' (')[0]} {t} {p}" for o, t, p, n in
                             zip(_nd2.Outcome, _nd2.Spot, _nd2.Period, _nd2.Normal_at_alpha) if n))
_p2 = lambda p: "< 0.001" if p < 0.001 else f"{p:.3f}"
supp2 = pd.DataFrame({
    "Spot": _nd2.Spot.str.capitalize(), "Period": _nd2.Period.str.capitalize(),
    "Outcome": _nd2.Outcome.map({"SUHII (°C)": "SUHII", "Total area L50 (km²)": "Area (L50)"}),
    "N (cities)": _nd2.N, "Shapiro–Wilk W": _nd2.Shapiro_W.map("{:.4f}".format),
    "Shapiro–Wilk p": _nd2.Shapiro_p.map(_p2), "D'Agostino–Pearson p": _nd2.DAgostino_p.map(_p2),
    "Skewness": _nd2.Skew.map("{:.3f}".format), "Excess kurtosis": _nd2.Excess_kurt.map("{:.3f}".format),
    "Normal": _nd2.Normal_at_alpha.map({True: "Yes", False: "No"})})
FOOT_S2 = ("N = cities with a spot of that type and period. Normal: Shapiro–Wilk p ≥ 0.05; the "
           "D'Agostino–Pearson test is given alongside.")
csvS2 = SUP_TAB / "Supp_Table_S2_normality_tests.csv"
supp2.to_csv(csvS2, index=False)
_pht._simple_xlsx(supp2, SUP_TAB / "Supp_Table_S2_normality_tests.xlsx",
                  title="Supplementary Table S2: Normality of the association inputs (Shapiro–Wilk)",
                  footnote=FOOT_S2)
print(f"Saved: {csvS2.name}, {csvS2.with_suffix('.xlsx').name} in {SUP_TAB.relative_to(ROOT)}")
display(supp2)
print(FOOT_S2)

<a id="nb07-supp-table-s3"></a>

#### Supp. Table S3 · Heat events per city

I list each city's persistent heat events, the days in them, the longest event and the mean duration, day and night, with cluster and national means.

**Data:** package `city_year_summary`, `heat_event_catalogue`.  
**Output:** `supplementary_tables/Supp_Table_S3_heat_events_per_city.csv`

In [ ]:
begin("Supp. Table S3")
_cy3 = pkg("city_year_summary").sort_values("city").set_index("city")
_hc3 = pkg("heat_event_catalogue")
_NM3 = {"Events": "n_persistent_events", "PersistDays": "total_persistent_days", "Longest": "longest_event_days"}
_c3 = pd.DataFrame(index=_cy3.index)
_same3 = []
for _p in ("day", "night"):
    _g = _hc3[_hc3.period == _p].groupby("city").duration_days
    _ev = pd.DataFrame({"Events": _g.size(), "PersistDays": _g.sum(), "Longest": _g.max()}).reindex(_c3.index)
    for _k, _col in _NM3.items():
        _c3[f"{_p.title()}_{_k}"] = _cy3[f"era5_{_p}_{_col}"].astype(float)
        _same3.append((_c3[f"{_p.title()}_{_k}"] == _ev[_k].fillna(0)).all())
    _c3[f"{_p.title()}_MeanDur"] = _g.mean().reindex(_c3.index)     # no events: no mean
check("Supp. Table S3 - every city's events, persistent days and longest event equal the event catalogue",
      len(_c3) == 56 and all(_same3),
      f"day {int(_c3.Day_Events.sum())} events, night {int(_c3.Night_Events.sum())} "
      f"(catalogue {(_hc3.period == 'day').sum()} / {(_hc3.period == 'night').sum()})")

_cl3 = _cy3.cluster25.astype(int)
supp3e = pd.concat(
    [_c3.assign(Level="City", Entity=_c3.index, Cluster=_cl3.map("C{}".format))]
    + [_c3[_cl3 == c].mean().to_frame().T.assign(Level="Cluster", Entity=f"C{c}", Cluster=f"C{c}")
       for c in sorted(_cl3.unique())]
    + [_c3.mean().to_frame().T.assign(Level="National", Entity="National", Cluster="")],
    ignore_index=True)
supp3e = supp3e[["Level", "Entity", "Cluster"] + list(_c3.columns)].round(1)
FOOT_S3 = (f"ERA5-Land persistent heat events, {'/'.join(map(str, sorted(_cy3.study_year.unique())))}: "
           "events, days in them, the longest and the mean duration (days). Cluster and national rows "
           "are means of their cities.")
csvS3 = SUP_TAB / "Supp_Table_S3_heat_events_per_city.csv"
supp3e.to_csv(csvS3, index=False)
print(f"Saved: {csvS3.name} in {SUP_TAB.relative_to(ROOT)}: {len(supp3e)} rows "
      f"({(supp3e.Level == 'City').sum()} cities, {(supp3e.Level == 'Cluster').sum()} clusters, national)")
display(supp3e[supp3e.Level != "City"])
print(FOOT_S3)

<a id="nb07-supp-table-s4"></a>

#### Supp. Table S4 · Selected events, event tier (day / night) and sensor agreement

I list each city's selected events, its event tier (day / night) and the MODIS–ERA5-Land and SEVIRI–ERA5-Land agreement.

**Data:** package `city_year_summary`, checked against `heat_event_catalogue`.  
**Output:** `supplementary_tables/Supp_Table_S4_selected_events_sensor_agreement.csv`

In [ ]:
begin("Supp. Table S4")
_cy4 = pkg("city_year_summary").sort_values("city").reset_index(drop=True)
_sel4 = pkg("heat_event_catalogue").query("is_selected").set_index(["city", "period"])
_SEL4 = {"Start": "selected_event_start", "End": "selected_event_end",
         "Days": "selected_event_duration_days", "Tier": "selected_event_tier"}
_MOD4 = {"n": "modis_era5land_n", "r": "modis_era5land_r", "r2": "modis_era5land_r2",
         "Bias": "era5land_minus_modis_bias_c", "RMSE": "modis_era5land_rmse_c", "MAE": "modis_era5land_mae_c"}
_SEV4 = {"SEVIRI_n": "seviri_era5land_n", "SEVIRI_Bias": "seviri_minus_era5land_bias_c",
         "SEVIRI_RMSE": "seviri_era5land_rmse_c", "SEVIRI_MAE": "seviri_era5land_mae_c"}
_DEG4 = {"Bias", "RMSE", "MAE", "SEVIRI_Bias", "SEVIRI_RMSE", "SEVIRI_MAE"}     # °C, shown to 0.001
supp4 = pd.DataFrame({"City": _cy4.city, "Cluster": _cy4.cluster25.astype(int).map("C{}".format),
                      "Event tier (day / night)": _cy4.validation_tier})
for _p in ("day", "night"):
    for _k, _c in {**_SEL4, **_MOD4, **_SEV4}.items():
        _v = _cy4[f"{_p}_{_c}"]
        supp4[f"{_p.title()}_{_k}"] = _v.round(3) if _k in _DEG4 else _v

_ev4 = all(str(_sel4.loc[(c, p), "start_date"])[:10] == str(r[f"{p}_selected_event_start"])[:10]
           and str(_sel4.loc[(c, p), "end_date"])[:10] == str(r[f"{p}_selected_event_end"])[:10]
           and _sel4.loc[(c, p), "duration_days"] == r[f"{p}_selected_event_duration_days"]
           and _sel4.loc[(c, p), "tier"] == r[f"{p}_selected_event_tier"]
           for _, r in _cy4.iterrows() for c in [r.city] for p in ("day", "night"))
check("Supp. Table S4 - each city's selected day and night event (dates, days, tier) is the event "
      "catalogue's selected event; the event tier (day / night) is the pair of them",
      len(supp4) == 56 and _ev4
      and (supp4["Event tier (day / night)"] == supp4.Day_Tier + "/" + supp4.Night_Tier).all(),
      f"{len(_sel4)} selected events; " + ", ".join(f"{k} {v}" for k, v in
                                                     supp4["Event tier (day / night)"].value_counts().items()))
check("Supp. Table S4 - SEVIRI-ERA5-Land days lie within the selected event",
      all((supp4[f"{p}_SEVIRI_n"] <= supp4[f"{p}_Days"]).all() for p in ("Day", "Night")),
      "; ".join(f"{p.lower()}: all event days in {int((supp4[f'{p}_SEVIRI_n'] == supp4[f'{p}_Days']).sum())}"
                f" of 56 cities" for p in ("Day", "Night")))
FOOT_S4 = ("Selected event per period with its tier. MODIS–ERA5-Land agreement over the city's daily "
           "pairs (n, r, r², bias = ERA5-Land − MODIS, RMSE, MAE; °C); SEVIRI–ERA5-Land agreement over "
           "the selected event's days (n days, bias = SEVIRI − ERA5-Land, RMSE, MAE; °C).")
csvS4 = SUP_TAB / "Supp_Table_S4_selected_events_sensor_agreement.csv"
supp4.to_csv(csvS4, index=False)
print(f"Saved: {csvS4.name} in {SUP_TAB.relative_to(ROOT)}: {len(supp4)} cities × {supp4.shape[1]} columns")
display(supp4.head(4))
print(FOOT_S4)

<a id="nb07-supp-table-s5"></a>

#### Supp. Table S5 · Spots and contours per city

I list each city's spots, their L50 / L75 / L90 areas and the residents within them, under the cluster totals.

**Data:** package `spots`, `city_profile`, checked against `city_exposure_by_contour`.  
**Output:** `supplementary_tables/Supp_Table_S5_spots_per_city` (.csv, .xlsx)

In [ ]:
begin("Supp. Table S5")
_sp5 = pkg("spots")
_ex5 = pkg("city_exposure_by_contour").set_index(["city", "spot_type", "period"])
_cp5 = pkg("city_profile")
s6 = _pht.build_supp6(None, spots=_sp5, city_cluster=dict(zip(_cp5.city, _cp5.cluster25.astype(int))),
                      cluster_fmt="C{}")
_subs5 = _pht._S6_SUB + _pht._S6_POP
_city5 = s6[~s6._group].set_index("Cluster/Entity")
_LV5 = {"L50": ("area_l50_km2", "pop_l50_persons"), "L75": ("area_l75_km2", "pop_l75_persons"),
        "L90": ("area_l90_km2", "pop_l90_persons")}


_area5 = lambda v: "" if pd.isna(v) or not v else round(float(v), 2)      # as the table writes it
_bad5 = [(c, t, p, lv) for (c, t, p), r in _ex5.iterrows() for lv, (a, pop) in _LV5.items()
         if _city5.loc[c, f"{t.title()} · {p.title()}|{lv}"] != _area5(r[a])
         or (_area5(r[a]) != "" and _city5.loc[c, f"{t.title()} · {p.title()}|Pop {lv}"]
             != int(round(r[pop])))]
check("Supp. Table S5 - every city's L50 / L75 / L90 area and population equal city_exposure_by_contour",
      len(_city5) == 56 and not _bad5,
      f"{len(_ex5)} city-period-type rows × 3 levels; {len(_bad5)} differ")
_n5 = {g: s6.loc[~s6._group, f"{g}|n"].replace("", 0).sum() for _, _, g in _pht._S6_COMBOS}
_tot5 = {lv: _sp5[a].sum() for lv, (a, _) in _LV5.items()}
check("Supp. Table S5 - the city rows hold all 185 spots; cluster rows sum their cities",
      sum(_n5.values()) == len(_sp5) == 185
      and all(s6.loc[s6._group, f"{g}|n"].replace("", 0).sum() == n for g, n in _n5.items()),
      "; ".join(f"{g} {int(n)}" for g, n in _n5.items())
      + "; L50 / L75 / L90 " + " / ".join(f"{v:,.1f}" for v in _tot5.values()) + " km²")
FOOT_S5 = ("n = spots of the city in that type and period; L50 / L75 / L90 = contour-zone areas (km²), "
           "nested (L90 ⊂ L75 ⊂ L50) and never added across levels; Pop = residents within the zone "
           "(persons); BH = the city's spots of that type and period survive BH-FDR; blank = no spot or "
           "no zone at that level. Green rows = clusters, the sum of their cities (n = cities).")
s6.drop(columns=["_group"]).to_csv(SUP_TAB / "Supp_Table_S5_spots_per_city.csv", index=False)
_pht._grouped_xlsx(s6, SUP_TAB / "Supp_Table_S5_spots_per_city.xlsx",
                   id_cols=["Cluster/Entity"], groups=[g for _, _, g in _pht._S6_COMBOS], subs=_subs5,
                   title="Supplementary Table S5: City-level significant-spot zone footprints (L50/L75/L90, "
                         "km²) and the population within them",
                   footnote=FOOT_S5, col_width=9)
print(f"Saved: Supp_Table_S5_spots_per_city (.csv, .xlsx) in {SUP_TAB.relative_to(ROOT)}: {len(s6)} rows "
      f"({(~s6._group).sum()} cities, {s6._group.sum()} clusters)")
_show5 = s6[s6._group].drop(columns="_group").set_index("Cluster/Entity")
_show5.columns = pd.MultiIndex.from_tuples([tuple(c.split("|")) for c in _show5.columns])
display(_show5)
print(FOOT_S5)

<a id="nb07-supp-table-s6"></a>

#### Supp. Table S6 · Stakeholder accounts per city

I extend Table 3 to all 56 cities and 16 clusters; run the Table 3 cell first.

**Data:** as Table 3; package `city_profile`.  
**Output:** `supplementary_tables/Supp_Table_S6_stakeholder_per_city.csv`

In [ ]:
begin("Supp. Table S6")
assert "tab3" in globals(), "Supp. Table S6 extends Table 3: run the Table 3 cell first"
_cp6 = pkg("city_profile")
_cl6 = dict(zip(_cp6.city, _cp6.cluster25.astype(int)))
_t3 = tab3.to_dict("records")
_t3city = {r["Unit"].replace(" †", ""): r for r in _t3 if r["Level"] == "City"}
_t3cl = {r["Unit"]: r for r in _t3 if r["Level"] == "Cluster"}


def _none6(level, unit, cl, day, night):
    """A city or cluster without stakeholders: N = 0 and its urban-core trends only."""
    return {"Level": level, "Unit": unit, "Cluster": cl, "N": 0, "LST_Day_Trend": day, "LST_Night_Trend": night}


_lbl6 = lambda c, p: pf.trend_label(_ts3, c, p)
_rows6 = [_t3city.get(c) or _none6("City", c, f"C{_cl6[c]}", _lbl6(c, "day"), _lbl6(c, "night"))
          for c in sorted(_cl6)]
for _k in sorted(set(_cl6.values())):
    _cc = sorted(c for c, k in _cl6.items() if k == _k)
    _rows6.append(_t3cl.get(f"C{_k}") or _none6("Cluster", f"C{_k}", f"C{_k}", pf.trend_count(_ts3, _cc, "day"),
                                                 pf.trend_count(_ts3, _cc, "night")))
_rows6.append(tab3[tab3.Level == "National"].iloc[0].to_dict())
supp6 = pd.DataFrame(_rows6, columns=tab3.columns)

_rec6 = lambda d: sorted(d.astype(str).apply(tuple, axis=1))
_rep6 = supp6[supp6.N > 0]
check("Supp. Table S6 - the 21 cities, 10 clusters and national row with stakeholders are Table 3's rows",
      _rec6(_rep6) == _rec6(tab3),
      f"{(_rep6.Level == 'City').sum()} cities, {(_rep6.Level == 'Cluster').sum()} clusters; "
      f"{int(supp6[supp6.Level == 'City'].N.sum())} participants")
_c6 = supp6[supp6.Level == "City"]
check("Supp. Table S6 - all 56 cities and 16 clusters; every city's trends are its own urban-core trends",
      len(_c6) == 56 and (supp6.Level == "Cluster").sum() == 16
      and all(r.LST_Day_Trend == _lbl6(r.Unit.replace(" †", ""), "day")
              and r.LST_Night_Trend == _lbl6(r.Unit.replace(" †", ""), "night") for r in _c6.itertuples()),
      f"{(_c6.N == 0).sum()} cities and {((supp6.Level == 'Cluster') & (supp6.N == 0)).sum()} clusters "
      f"without stakeholders; significant increase in {(_c6.LST_Day_Trend == 'Significant increase').sum()} "
      f"(day) and {(_c6.LST_Night_Trend == 'Significant increase').sum()} (night) of 56 cities")
FOOT_S6 = (FOOT_T3 + " Cities and clusters without stakeholders (N = 0) carry only their urban-core "
           "trends; a cluster without stakeholders counts all its cities.")
csvS6 = SUP_TAB / "Supp_Table_S6_stakeholder_per_city.csv"
supp6.to_csv(csvS6, index=False)
print(f"Saved: {csvS6.name} in {SUP_TAB.relative_to(ROOT)}: {len(supp6)} rows ({(supp6.N == 0).sum()} without "
      f"stakeholders)")
display(supp6[supp6.Level != "City"])
print(FOOT_S6)

<a id="nb07-supp-table-s7"></a>

#### Supp. Table S7 · Hit rate by city and buffer

I list the hit rate of each city and cluster at every buffer from 0.5 to 10 km.

**Data:** package `hitrate_sensitivity`, `stakeholder_spots_gps`.  
**Output:** `supplementary_tables/Supp_Table_S7_hit_rate_by_city` (.csv, .xlsx)

In [ ]:
begin("Supp. Table S7")
_hs7 = pkg("hitrate_sensitivity")
_gp7 = pkg("stakeholder_spots_gps")
s7, _bufcols7 = _pht.build_supp7(None, hs=_hs7)
_ct7, _gr7 = s7[~s7._group], s7[s7._group]
_np7 = _gp7.groupby(["city", "spot_type"]).size()
_tp7 = _ct7.groupby(["Spot", "Period"])["No. of points"].sum()
check("Supp. Table S7 - the cohort points: each city's points are its marked points, 55 hot and 39 cold "
      "in each period, from 17 cities in 8 clusters",
      all(r["No. of points"] == _np7[(r["Cluster/Entity"], r.Spot.lower())] for _, r in _ct7.iterrows())
      and all(_tp7[(t, p)] == (_gp7.spot_type == t.lower()).sum() for t, p in _tp7.index)
      and sorted(_tp7.unique()) == [39, 55]
      and _ct7["Cluster/Entity"].nunique() == _gp7.city.nunique() == 17 and _gr7["Cluster/Entity"].nunique() == 8,
      "; ".join(f"{t.lower()} {p.lower()} {n}" for (t, p), n in _tp7.items()) + f"; {len(s7)} rows")
_cc7 = dict(zip(_hs7.city, _hs7.cluster25.astype(int)))
_sum7 = (_ct7.assign(_k=_ct7["Cluster/Entity"].map(lambda c: f"C{_cc7[c]}"))
         .groupby(["_k", "Spot", "Period"])["No. of points"].sum())
_h7 = _hs7[_hs7.buffer_m == _hs7.buffer_m.min()].groupby(["spot_type", "period"]).n_hit.sum()
check("Supp. Table S7 - each cluster row holds the points of all its cities; at 0.5 km the city rows give "
      "the layer's hits (as Fig. 14)",
      all(r["No. of points"] == _sum7[(r["Cluster/Entity"], r.Spot, r.Period)] for _, r in _gr7.iterrows())
      and all(round((g[_bufcols7[0]] * g["No. of points"] / 100).sum()) == _h7[(t.lower(), p.lower())]
              for (t, p), g in _ct7.groupby(["Spot", "Period"])),
      "; ".join(f"{t} {p} {int(n)}/{_tp7[(t.title(), p.title())]}" for (t, p), n in _h7.items()))
FOOT_S7 = (f"Values = hit rate (%) at each buffer radius, on {len(_gp7)} marked points "
           f"({(_gp7.spot_type == 'hot').sum()} hot, {(_gp7.spot_type == 'cold').sum()} cold) from "
           f"{_gp7.city.nunique()} cities and {_gp7.cluster25.nunique()} clusters. Hit rates are row-based: a "
           "point lying within several spot footprints counts once per footprint, so the denominator can "
           "grow with the buffer radius. Green rows = clusters, their cities pooled.")
s7.drop(columns=["_group"]).to_csv(SUP_TAB / "Supp_Table_S7_hit_rate_by_city.csv", index=False)
_pht._grouped_xlsx(s7, SUP_TAB / "Supp_Table_S7_hit_rate_by_city.xlsx",
                   id_cols=["Cluster/Entity", "Spot", "Period", "No. of points"],
                   groups=["Hit-rate Buffer Sensitivity (%)"], subs=_bufcols7,
                   title="Supplementary Table S7: Stakeholder–satellite spot corroboration — buffer sensitivity",
                   footnote=FOOT_S7)
print(f"Saved: Supp_Table_S7_hit_rate_by_city (.csv, .xlsx) in {SUP_TAB.relative_to(ROOT)}: {len(s7)} rows "
      f"({len(_ct7)} city, {len(_gr7)} cluster)")
display(_gr7.drop(columns=["_group"]))
print(FOOT_S7)

<a id="nb07-supp-table-s8"></a>

#### Supp. Table S8 · Best-element attribution, complete

I list the typology against its best single element for all 12 outcomes.

**Data:** NB06 `master_synthesis.csv`, checked against `partA_synthesis_cluster25.csv`.  
**Output:** `supplementary_tables/Supp_Table_S8_best_element_attribution` (.csv, .xlsx)

In [ ]:
begin("Supp. Table S8")
supp8, _, _ms8 = _pht.build_supp8_and_table7(read_source(RESULTS_06T))
_pa8 = pd.read_csv(read_source(RESULTS_06T / "partA_synthesis_cluster25.csv"))
_PA8 = {(r["Spot Type"], r["Period"], o): float(r[c]) for r in _pa8.to_dict("records")
        for o, c in (("Occurrence", "V (occ)"), ("SUHII", "η² (suhii)"), ("Area (L50)", "η² (area)"))}
_occ8 = _ms8.Outcome.str.lower().str.startswith("occ")
_A8 = np.where(_occ8, _ms8["A: V (Cramér)"], _ms8["A: η²"])
_B8 = np.where(_occ8, _ms8["B: V (Cramér)"], _ms8["B: η²"])
_dA8 = max(abs(a - _PA8[(s, p, o)]) for a, s, p, o in zip(_A8, _ms8["Spot Type"], _ms8.Period, _ms8.Outcome))
_dD8 = float(np.nanmax(np.abs(_A8 - _B8 - _ms8["Δ effect size"])))
check("Supp. Table S8 - all 12 outcomes; each composite effect is Table 4's (NB06 part A); every Δ is "
      "composite minus element",
      len(supp8) == 12 and _dA8 < 1e-9 and _dD8 < 1e-9,
      f"worst composite difference {_dA8:.1e}, worst Δ difference {_dD8:.1e}; best elements: "
      + ", ".join(f"{k} {v}" for k, v in _ms8["B: Best element"].value_counts().items()))
FOOT_S8 = ("Composite PLT vs best contributing element across all 12 outcomes. (***/**/* /ns) = element "
           "significance. Δ = composite − element effect size (η² for SUHII/Area; V for Occurrence).")
csvS8 = SUP_TAB / "Supp_Table_S8_best_element_attribution.csv"
supp8.to_csv(csvS8, index=False)
_pht._simple_xlsx(supp8, SUP_TAB / "Supp_Table_S8_best_element_attribution.xlsx",
                  title="Supplementary Table S8: Full best PLT single-element attribution (all outcomes)",
                  footnote=FOOT_S8)
print(f"Saved: {csvS8.name}, {csvS8.with_suffix('.xlsx').name} in {SUP_TAB.relative_to(ROOT)}")
display(supp8)
print(FOOT_S8)

<a id="nb07-assembly"></a>

## 4 · Assembly

I write `captions.md` for the main figures and tables, the main and supplementary contact sheets and
`figure_print_sizes.csv`. I then check every figure file's format, summarise the data sources, scan
this notebook's code for forbidden data paths and print the result of every check.

[↑ Contents](#nb07-contents)

In [ ]:
begin("Assembly")
import fitz as _fitzC

_profC = pkg("city_profile")
_pcC = pkg("stakeholder_perception")
_exC = pkg("city_exposure_by_contour")
_spC = pkg("spots")
_perCityC = _pcC.groupby("city").size()
_singlesC = sorted(_perCityC[_perCityC == 1].index)
_hotNightC = _exC[(_exC.spot_type == "hot") & (_exC.period == "night")].pop_l50_persons.sum()
_uc13C = _profC[_profC.cluster25 == 13].uc_area_km2.sum()
_share13C = 100.0 * _uc13C / _profC.uc_area_km2.sum()

PERCEPTION_BASE = ("105 participants in 21 cities and 10 physiographic landscape "
                   "typology (PLT) clusters")
POINT_BASE = "94 marked points (55 hot, 39 cold) from 17 cities and 8 PLT clusters"
ROW_RULE = ("Hit rates are row-based: a point lying within several spot footprints "
            "counts once per footprint, so the denominator grows with the buffer radius.")
NB06_NOTE = ("Source: the NB06 association caches, not the published GeoPackage. The "
             "association analysis is deliberately excluded from the dataset (README "
             "section 7), so this is the one item whose values cannot be traced to it.")

_ITEMS = [
 ("Fig. 1", "Fig_01_study_area",
  f"The 67 Nigerian cities with at least 300,000 inhabitants in 2024 (UN World "
  f"Urbanization Prospects), on the physiographic landscape typology of Nigeria.",
  f"Of these, {len(_profC)} were retained as study cities; 11 lie within the urban core "
  f"of a larger neighbouring city and were merged into it (Section 2.3). The background "
  f"is the national physiographic landscape typology (PLT), all {len(CL)} classes in "
  f"their published colours; each class is named by its three axes - climate, geology "
  f"and elevation. Symbol size gives 2024 city population, the smallest class beginning "
  f"at the 300,000 selection threshold.",
  "`city_profile` (city, cluster25, population_wup_k, absorbed_cities); the WUP table "
  "(positions of the 67 cities and the populations of the 11 absorbed ones; the 56 "
  "study-city populations asserted against `city_profile.population_wup_k`); the "
  "typology background from `landscape_ch_polygons_with_class_metrics.geojson`; the "
  "national boundary."),

 ("Fig. 3", "Fig_03_delineation_excerpt",
  "Three-zone delineation for five cities: Kano, Abuja, Jos, Lagos and Port Harcourt.",
  "Urban core, background reference area (BRA) and city region for each city.",
  "NB01 zone GeoJSONs."),

 ("Fig. 4", "Fig_04_stakeholder_rings",
  f"Stakeholder participants by physiographic cluster and city: {len(_pcC)} participants "
  f"from {_pcC.city.nunique()} cities in {_pcC.cluster25.nunique()} clusters.",
  "",
  "`stakeholder_perception` (participant_code, city, cluster25), with each participant's "
  "cluster asserted against `city_profile`."),

 ("Fig. 5", "Fig_05_annual_trends",
  "Annual land surface temperature (LST) by physiographic landscape typology, 2006-2025.",
  "For each of six selected PLT clusters, the annual series. The urban core and "
  "background reference area are read on the left axis and the city-region range is "
  "shaded; the surface urban heat island intensity (SUHII, urban core minus BRA) is read "
  "on its own right-hand axis, because it is a difference of a few degrees against "
  "absolutes of 15 to 45. Day and night are distinguished by colour within each zone, "
  "and dashed lines give the Sen slope. The table below the panels gives the urban-core, "
  "BRA and SUHII trends for both periods from the published cluster composites, with "
  "Mann-Kendall direction and significance.",
  "`composite_trend_statistics` (the table); the NB03 cluster series behind the panels, "
  "asserted equal to the cluster means of `annual_series`, and the NB03 panel statistics "
  "asserted to match the published composites on all 36 slopes."),

 ("Fig. 6", "Fig_06_monthly_trends",
  "Monthly land surface temperature (LST) climatology by physiographic landscape "
  "typology, 2006-2025.",
  "For the same six clusters, the monthly climatology, drawn as in Fig. 5. The table "
  "below the panels gives the Mann-Kendall direction and significance and the Sen slope "
  "of each cluster's 240-month series for the urban core, BRA and SUHII, by day and night.",
  "the NB03 monthly cluster series and monthly statistics (the package holds annual "
  "composites only); the series asserted equal to the cluster means of `monthly_series`."),

 ("Fig. 7", "Fig_07_heat_event_bars",
  "Persistent heat events of 2024 by physiographic landscape typology.",
  "(a) Mean number of ERA5-Land persistent heat events per city in each cluster, by day "
  "and night. (b) Mean longest event per city. In both, the bar is the cluster mean "
  "across its cities, the error bar spans the lowest and highest city, and the bar border "
  "gives the event tier of that period most common among the cluster's cities. Event "
  "tiers are reported per city in Supplementary Table S4.",
  "`city_year_summary` (ERA5-Land event counts and longest events; "
  "day_selected_event_tier, night_selected_event_tier)."),

 ("Fig. 8", "Fig_08_seviri_suhii_cities",
  "Event-scale surface urban heat island intensity (SUHII) by city.",
  "SUHII (urban core minus BRA) for every city from the MSG-SEVIRI composites over its "
  "selected event window, by day (left) and night (right), sorted by daytime value and "
  "filled by cluster colour; the box gives the national mean and range. Event tiers are "
  "reported per city in Supplementary Table S4.",
  "`city_year_summary` (day_seviri_suhii_c, night_seviri_suhii_c, cluster25)."),

 ("Fig. 9", "Fig_09_spot_maps_day",
  f"Surface hot and cold spots in all {len(_profC)} cities, by day.",
  "Kernel density contours at the 50, 75 and 90 per cent levels (L50, L75, L90) with "
  "tier-2 buffers, over the three zones of each city.",
  "the NB05 A4 sheet and map legend; the spot and contour counts per city asserted "
  "against `spots` and `spot_contours`."),

 ("Fig. 10", "Fig_10_spot_maps_night",
  f"Surface hot and cold spots in all {len(_profC)} cities, at night.",
  "As Fig. 9, for the night-time composites.",
  "as Fig. 9."),

 ("Fig. 11", "Fig_11_spot_area_by_cluster",
  "Spot footprint area by physiographic landscape typology.",
  f"Total spot area per cluster for (a) hot spots by day, (b) hot spots at night, (c) cold "
  f"spots by day and (d) cold spots at night. The three contour levels are NESTED and drawn "
  f"as segments of one bar: L90 at the base, then L75 minus L90, then L50 minus L75, so the "
  f"full bar height is the L50 total. The four panels cover all {len(_spC)} spots.",
  "`spots` (cluster25, spot_type, period, area_l50/l75/l90_km2), the level totals asserted "
  "equal to `spot_contours`."),

 ("Fig. 12", "Fig_12_population_within_spots",
  "Resident population within surface hot and cold spots.",
  f"(a), (b) National maps for hot and cold spots: one circle per city, sized by the "
  f"residents inside its day and its night L50 footprints added together, as the key states, "
  f"and split between day and night by spot area. "
  f"(c) Half-donuts for the nation and four clusters: the rings run from spot type to day "
  f"and night to contour level (L90, L75, L50), each segment labelled with its residents, "
  f"and the centre gives the total of the rings. Population is summed from GHS-POP on its "
  f"native 100 m grid, per polygon, with fractional overlap. Nationally, "
  f"{int(round(_hotNightC)):,} residents fall within night-time hot spot footprints at L50.",
  "`city_exposure_by_contour` (population and area per contour level); `city_profile` "
  "(clusters) and `city_zones` (urban-core centroids); the national boundary."),

 ("Fig. 13", "Fig_13_stakeholder_perception",
  "Stakeholder perception of urban heat by physiographic landscape typology.",
  f"Responses by cluster, as counts of participants with the national total as a donut "
  f"inset: (a) perceived temperature increase over the last ten years, (b) the period in "
  f"which heat is felt most severely, and (c) the season perceived as the temperature peak. "
  f"Based on {PERCEPTION_BASE}. Answers left blank are shown as stored, \"I'd rather not "
  f"say\".",
  "`stakeholder_perception` (the three perception items)."),

 ("Fig. 14", "Fig_14_hitrate_buffer",
  "Agreement between stakeholder-marked locations and detected spots across buffer radii.",
  f"Hot spots left and cold spots right, for buffer radii from 0.5 to 10 km, with the total "
  f"hit count shaded on the right-hand axis. Based on {POINT_BASE}. {ROW_RULE} Coordinates "
  f"are the published ones, rounded to three decimal degrees, so the curves can be "
  f"reproduced from the dataset alone.",
  "`hitrate_sensitivity` (buffer_m, n_hit, n_total); the 500 m values are asserted at "
  "4/55, 3/55, 4/39, 0/39."),

 ("Fig. 15", "Fig_15_association_effect_sizes",
  "Association of the physiographic landscape typology with the thermal anomalies, "
  "against its strongest single element.",
  f"Forest plot of effect sizes for each combination of spot type, period and outcome, "
  f"with the full typology against its strongest single element - climate zone, "
  f"geological substrate or elevation belt. Occurrence is tested with a Monte Carlo "
  f"permutation chi-square and reported as Cramer V; SUHII and area are tested with "
  f"Kruskal-Wallis and reported as eta squared. {NB06_NOTE}",
  "None from the package. `Results/06_hotspot_modelling/tables/`."),
]

_TABLES_C = [
 ("Table 1", "Table_01_typology_composition",
  f"Physiographic cluster composition of the {len(_profC)} study cities.",
  f"One row per PLT cluster, sorted by the number of cities, with five "
  f"single-city clusters (C6, C8, C15, C21, C23) grouped into a final row. Columns give "
  f"the member cities, the city count, the physiographic description (climate · geology "
  f"· elevation), the summed 2024 WUP population after absorption, and the summed "
  f"city-region and urban-core area, computed in ESRI:102022. C13 alone holds "
  f"{_share13C:.1f} per cent of the urban-core area of all {len(_profC)} study cities.",
  "`city_profile` (cluster25, city, population_wup_merged_k, cr_area_km2, uc_area_km2)."),

 ("Table 2", "Table_02_significant_spots_by_cluster",
  "Significant spots by physiographic landscape typology.",
  "Spot counts, contributing cities and total L50 footprint area for each cluster, by "
  "spot type and period, with a national total row. Benjamini-Hochberg FDR survival is "
  "a property of the city-period-type group, repeated on every spot of that group, not "
  "a per-spot test.",
  "`spots` (cluster25, city, period, spot_type, area_l50_km2, bh_significant), through "
  "`build_table5` in `src/pub_hotspot_tables.py`."),

 ("Table 3", "Table_03_stakeholder_vs_satellite",
  "Stakeholder accounts against the satellite record, by city, physiographic landscape "
  "typology and nation.",
  f"One row per city, per cluster and for the nation, based on {PERCEPTION_BASE}. City rows "
  f"give each city's own urban-core Mann-Kendall trend; cluster and national rows count the "
  f"member cities whose trend is a significant increase (p < 0.05), the same per-city basis "
  f"as the agreement percentages: nationally 12 of 21 cities by day and 14 of 21 by night. "
  f"Seven cities are represented by a single stakeholder ({', '.join(_singlesC)}) and are "
  f"flagged, because one account cannot express agreement within a city. Nationally, "
  f"{int(_d3.q58.sum())} of 105 participants named a season containing the observed hottest "
  f"month of their city.",
  "`stakeholder_perception`, `trend_statistics` (zone = uc), `monthly_climatology`, and "
  "`monthly_series` for the cluster composites; NB03's agreement and consensus rules applied "
  "in the notebook."),

 ("Table 4", "Table_04_typology_association",
  "Association of the physiographic landscape typology with the thermal anomalies.",
  f"Spot type by period by outcome, giving the test applied, the typology effect size and its "
  f"p-value. Five of the twelve typology tests reach p < 0.05. {NB06_NOTE}",
  "None from the package. `Results/06_hotspot_modelling/tables/partA_synthesis_cluster25.csv`."),

 ("Table 5", "Table_05_best_element_attribution_excerpt",
  "Best single-element attribution (excerpt).",
  f"For the six notable outcomes, the composite typology's effect size against its strongest "
  f"single element - climate zone, geological substrate or elevation belt - with the element's "
  f"significance and the difference between them. All twelve outcomes are in Supplementary "
  f"Table S8. {NB06_NOTE}",
  "None from the package. `Results/06_hotspot_modelling/tables/master_synthesis.csv`."),
]

_nl = chr(10)
_out = ["# Captions - SCS main figures and tables", "",
        "Generated by the Assembly section of `Notebook/07_publication_outputs.ipynb`, so the "
        "counts stay tied to the package.", "",
        "Figures carry no titles of their own; these captions carry them. Panel letters "
        "stay on the figures. Cluster names follow "
        "`Results/manuscript_outputs/cluster_labels.csv`, and typology colours are the "
        "published ones from `config.CLUSTER25_COLORS`.", "",
        "**Two stakeholder denominators, never mixed.** Perception items rest on "
        + PERCEPTION_BASE + ". Point-based items rest on " + POINT_BASE + ", because "
        "four cohort cities answered the perception items but marked no locations.", "",
        "**Event tier.** `validation_tier` and the selected-event tiers are a day/night "
        "pair, written as event tier (day / night), for example EXCELLENT/GOOD. A single "
        "event carries one grade, written as event tier with the period of that event.",
        "", "---", ""]
for _num, _stem, _title, _body, _src in _ITEMS:
    _out += [f"## {_num} - {_title}", "",
             f"`main_figures/{_stem}.(pdf|svg)`", "",
             f"**{_num}.** {_title} {_body}", "", f"*Reads:* {_src}", "", "---", ""]
for _num, _stem, _title, _body, _src in _TABLES_C:
    _out += [f"## {_num} - {_title}", "",
             f"`main_tables/{_stem}.(csv|xlsx)`", "",
             f"**{_num}.** {_title} {_body}", "", f"*Reads:* {_src}", "", "---", ""]
(OUT / "captions.md").write_text(_nl.join(_out), encoding="utf-8")
print("captions.md:", len(_ITEMS), "figures,", len(_TABLES_C), "tables")

# ── contact sheet, through the checked savefig ──
_pdfs = sorted(p for p in MAIN_FIG.glob("Fig_*.pdf"))
if _pdfs:
    _cols = 3; _rows = int(np.ceil(len(_pdfs) / _cols))
    figC, _axC = plt.subplots(_rows, _cols, figsize=(8.27, 11.69))
    _axC = np.atleast_1d(_axC).ravel()
    for _a in _axC:
        _a.set_axis_off()
    for _a, _p in zip(_axC, _pdfs):
        _doc = _fitzC.open(_p); _pm = _doc[0].get_pixmap(dpi=96)
        _img = np.frombuffer(_pm.samples, dtype=np.uint8).reshape(_pm.height, _pm.width, _pm.n)
        _a.imshow(_img[:, :, :3]); _a.set_title(_p.stem.replace("_", " "), fontsize=7)
        _doc.close()
    figC.suptitle("SCS main figures — contact sheet (%d items, %s)"
                  % (len(_pdfs), pd.Timestamp.today().date().isoformat()),
                  fontsize=10, fontweight="bold")
    figC.tight_layout(rect=(0, 0, 1, 0.97))
    _pC = savefig(figC, MAIN_FIG / "_thumbnails_main_figures")
    plt.show()
    print("Contact sheet:", len(_pdfs), "figures ->", [p.name for p in _pC])

# ── print size and smallest type, measured on the written PDFs ──────────
_SZ = fig_size_report()
print("" + chr(10) + "Print size and smallest type (target: width <= 190 mm, "
      "height <= 240 mm, text >= %.1f pt)" % MIN_PT)
print(_SZ.to_string(index=False))
_bad_sz = _SZ[~_SZ.ok]
print("Reported only:",
      f"{len(_SZ)} figures; widest {_SZ.width_mm.max():.0f} mm, tallest "
      f"{_SZ.height_mm.max():.0f} mm, smallest type {_SZ.min_pt.min():.1f} pt"
      + ("" if len(_bad_sz) == 0 else "; failing: "
         + ", ".join(f"{r.figure} ({r.width_mm:.0f}x{r.height_mm:.0f} mm, "
                     f"{r.min_pt:.1f} pt)" for r in _bad_sz.itertuples())))
_SZ.to_csv(OUT / "figure_print_sizes.csv", index=False)

# ── every published file must really be the format its extension claims ──
_figsC = sorted(q for d in (MAIN_FIG, SUP_FIG) for q in d.rglob("*") if q.suffix in (".pdf", ".svg"))
_badC = [q.name for q in _figsC if q.open("rb").read(5) != (b"%PDF-" if q.suffix == ".pdf" else b"<?xml")]
check("every figure file, main and supplementary, matches its extension", not _badC,
      f"{len(_figsC)} files checked" + (f"; bad: {_badC}" if _badC else ""))

In [ ]:
begin("Assembly")
# ── supplementary contact sheet: the first page of every set ─────────────────
_sets = []
for _d in sorted(p for p in SUP_FIG.iterdir() if p.is_dir()):
    _f = sorted(list(_d.rglob("*.pdf")) + list(_d.rglob("*.svg")))
    if _f:
        _sets.append((_d.name, _f[0]))
_cols = 4
figS, _axS = plt.subplots(int(np.ceil(len(_sets) / _cols)), _cols, figsize=(8.27, 11.69))
for _a in np.atleast_1d(_axS).ravel():
    _a.set_axis_off()
for _a, (_nm, _p) in zip(np.atleast_1d(_axS).ravel(), _sets):
    _doc = _as_page(_p)
    _pm = _doc[0].get_pixmap(dpi=60, alpha=False)
    _a.imshow(np.frombuffer(_pm.samples, dtype=np.uint8).reshape(_pm.height, _pm.width, _pm.n))
    _a.set_title(f"{_nm}\n{_p.name}", fontsize=6.5)
    _doc.close()
figS.tight_layout()
savefig(figS, SUP_FIG / "_thumbnails_supplementary", fit=None)
plt.close(figS)
print(f"Supplementary contact sheet: {len(_sets)} sets")

# ── provenance summary, from _data_sources.csv ───────────────────────────────
_src = pd.DataFrame(SOURCE_ROWS)
_oos = _src[_src.permitted == "OOS"].groupby("item")["category"].apply(
    lambda s: ", ".join(sorted(set(s))))
print(f"\n{len(_src)} reads logged in {SOURCES_CSV.name}: "
      + ", ".join(f"{k} {v}" for k, v in _src.category.value_counts().items()))
print("Reads outside the item's permitted sources:")
print(_oos.to_string() if len(_oos) else "  none")
check("every read passed the source guard (no forbidden data source)", True,
      f"{len(_src)} reads, {_src.path.nunique()} distinct files or folders")

# ── forbidden-path scan of this notebook's own code ──────────────────────────
_nbfile = ROOT / "Notebook" / "07_publication_outputs.ipynb"
_hits = [f"cell {i}: {ln.strip()[:80]}"
         for i, c in enumerate(json.loads(_nbfile.read_text(encoding="utf-8"))["cells"])
         if c["cell_type"] == "code"
         for ln in "".join(c["source"]).splitlines()
         if "forbidden-list" not in ln and any(f in ln for f in FORBIDDEN_PARTS)]
check("the notebook's code names no forbidden data path", not _hits, "; ".join(_hits[:5]))

# ── assertion summary ────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print("Assertions: %d of %d passed" % (sum(1 for a in ASSERTIONS if a[1]), len(ASSERTIONS)))
for _lbl, _ok, _d, _it in ASSERTIONS:
    print(("  [OK]   " if _ok else "  [FAIL] ") + f"[{_it}] " + _lbl + (f" - {_d}" if _d else ""))
print("=" * 72)